<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NBA_STABLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
SPORT = "basketball_nba"
SIMS = 50000

# NBA variance
SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN = 0.25
UNIT_CAP = 1.0
MAX_JUICE = -200

print("✅ NBA Engine Settings Loaded")

✅ NBA Engine Settings Loaded


In [ ]:
DAYS_BACK = 30

In [ ]:
# NBA Possessions
games_hist["home_poss"] = (
    games_hist["home_fga"]
    - games_hist["home_oreb"]
    + games_hist["home_tov"]
    + 0.44 * games_hist["home_fta"]
)

NameError: name 'games_hist' is not defined

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

ODDS_API_KEY = os.getenv("ODDS_API_KEY")  # if set in Colab secrets
# If not set, paste your key here:
# ODDS_API_KEY = "PASTE_KEY"

SLATE_DATE = "2026-03-01"   # change as needed
SPORT = "basketball_nba"
print("✅ Loaded:", SPORT, SLATE_DATE, "key?", bool(ODDS_API_KEY))

✅ Loaded: basketball_nba 2026-03-01 key? False


In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
print("✅ ODDS_API_KEY set?", bool(os.getenv("ODDS_API_KEY")))

✅ ODDS_API_KEY set? True


In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

# pulls from env (set via Secrets or the one-liner cell)
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-01"   # change if needed

if not ODDS_API_KEY:
    raise ValueError("ODDS_API_KEY not found. Set it in env or Colab Secrets.")

print("✅ Ready:", SPORT, SLATE_DATE)

✅ Ready: basketball_nba 2026-03-01


In [ ]:
def pull_market_data(
    sport: str,
    date: str,
    regions="us",
    markets="h2h,spreads,totals",
    odds_format="american",
    date_filter=True,
):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
    }

    if date_filter:
        day = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        params["commenceTimeFrom"] = day.isoformat().replace("+00:00", "Z")
        params["commenceTimeTo"] = (day + timedelta(days=1)).isoformat().replace("+00:00", "Z")

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        t = g.get("commence_time")

        bks = g.get("bookmakers", [])
        book = bks[0] if bks else None

        h2h = spreads = totals = None
        if book:
            for m in book.get("markets", []):
                if m.get("key") == "h2h":
                    h2h = m
                elif m.get("key") == "spreads":
                    spreads = m
                elif m.get("key") == "totals":
                    totals = m

        def pick_outcome(mkt, name):
            if not mkt: return None
            for o in mkt.get("outcomes", []):
                if o.get("name") == name:
                    return o
            return None

        home_ml = away_ml = np.nan
        if h2h:
            oh = pick_outcome(h2h, home)
            oa = pick_outcome(h2h, away)
            if oh: home_ml = oh.get("price", np.nan)
            if oa: away_ml = oa.get("price", np.nan)

        spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = np.nan
        if spreads:
            oh = pick_outcome(spreads, home)
            oa = pick_outcome(spreads, away)
            if oh:
                spread_home_line = oh.get("point", np.nan)
                spread_home_odds = oh.get("price", np.nan)
            if oa:
                spread_away_line = oa.get("point", np.nan)
                spread_away_odds = oa.get("price", np.nan)

        total_line = over_odds = under_odds = np.nan
        if totals:
            oo = pick_outcome(totals, "Over")
            ou = pick_outcome(totals, "Under")
            if oo:
                total_line = oo.get("point", np.nan)
                over_odds = oo.get("price", np.nan)
            if ou:
                under_odds = ou.get("price", np.nan)

        rows.append({
            "home_team": home,
            "away_team": away,
            "commence_time": t,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,
            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,
        })

    df = pd.DataFrame(rows)
    print(f"✅ pulled {len(df)} NBA games for {date}")
    print("ML non-null:", int(df["home_ml"].notna().sum()), "/", len(df))
    print("Spread non-null:", int(df["spread_home_line"].notna().sum()), "/", len(df))
    print("Total non-null:", int(df["total_line"].notna().sum()), "/", len(df))
    return df

games_df = pull_market_data(SPORT, SLATE_DATE, date_filter=True)
games_df.head()

✅ pulled 8 NBA games for 2026-03-01
ML non-null: 8 / 8
Spread non-null: 8 / 8
Total non-null: 8 / 8


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds
0,New York Knicks,San Antonio Spurs,2026-03-01T18:10:00Z,108,-126,2.0,-112,-2.0,-108,227.5,-114,-106
1,Brooklyn Nets,Cleveland Cavaliers,2026-03-01T20:40:00Z,420,-560,11.0,-106,-11.0,-114,222.5,-110,-110
2,Chicago Bulls,Milwaukee Bucks,2026-03-01T20:40:00Z,136,-162,3.5,-110,-3.5,-110,228.5,-112,-108
3,Denver Nuggets,Minnesota Timberwolves,2026-03-01T20:40:00Z,-148,126,-2.5,-114,2.5,-106,237.5,-114,-106
4,Indiana Pacers,Memphis Grizzlies,2026-03-01T22:10:00Z,-102,-116,1.0,-112,-1.0,-108,238.5,-106,-114


In [ ]:
def pull_completed_scores(sport: str, days_from: int):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": int(days_from)}
    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g.get("home_team")
        away = g.get("away_team")
        ct = g.get("commence_time")
        scores = {s["name"]: s["score"] for s in g.get("scores", [])} if g.get("scores") else {}
        hs = scores.get(home)
        as_ = scores.get(away)
        if hs is None or as_ is None:
            continue
        rows.append({
            "home_team": home,
            "away_team": away,
            "home_score": float(hs),
            "away_score": float(as_),
            "date": ct
        })

    return pd.DataFrame(rows)

# Pull multiple windows (stitch together)
pieces = []
for d in [3, 6, 10, 14, 21, 28]:
    try:
        df_part = pull_completed_scores(SPORT, d)
        pieces.append(df_part)
        print(f"daysFrom {d}: {len(df_part)} games")
    except Exception as e:
        print("⚠️", e)

historical_df = pd.concat(pieces, ignore_index=True).drop_duplicates(
    subset=["home_team","away_team","home_score","away_score","date"]
)

print("✅ Unique historical games pulled:", len(historical_df))
historical_df.head()

daysFrom 3: 20 games
⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠️ {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

✅ Unique historical games pulled: 20


,home_team,away_team,home_score,away_score,date
0,Indiana Pacers,Charlotte Hornets,109.0,133.0,2026-02-27T00:10:00Z
1,Philadelphia 76ers,Miami Heat,124.0,117.0,2026-02-27T00:11:00Z
2,Atlanta Hawks,Washington Wizards,126.0,96.0,2026-02-27T00:41:00Z
3,Brooklyn Nets,San Antonio Spurs,110.0,126.0,2026-02-27T00:41:00Z
4,Orlando Magic,Houston Rockets,108.0,113.0,2026-02-27T00:43:00Z


In [ ]:
PPP_LEAGUE = 1.13
HOME_EDGE_PTS = 2.0

hist = historical_df.copy()
hist["total_pts"] = hist["home_score"] + hist["away_score"]

# Possession proxy
hist["poss_proxy"] = hist["total_pts"] / PPP_LEAGUE

home_rows = pd.DataFrame({
    "team": hist["home_team"],
    "points_scored": hist["home_score"],
    "points_allowed": hist["away_score"],
    "poss": hist["poss_proxy"],
})

away_rows = pd.DataFrame({
    "team": hist["away_team"],
    "points_scored": hist["away_score"],
    "points_allowed": hist["home_score"],
    "poss": hist["poss_proxy"],
})

all_stats = pd.concat([home_rows, away_rows], ignore_index=True)

team_overall = all_stats.groupby("team").agg(
    poss=("poss","mean"),
    ppp_for=("points_scored", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean()),
    ppp_against=("points_allowed", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean())
).reset_index()

team_overall = team_overall.fillna(team_overall.mean(numeric_only=True))

print("✅ Team baseline built:", team_overall.shape)
team_overall.head()

✅ Team baseline built: (30, 4)


,team,poss,ppp_for,ppp_against
0,Atlanta Hawks,196.460177,0.641351,0.488649
1,Boston Celtics,229.203540,0.645714,0.484286
2,Brooklyn Nets,219.026549,0.504505,0.625495
3,Charlotte Hornets,196.460177,0.615901,0.514099
4,Chicago Bulls,206.194690,0.543176,0.586824


In [ ]:
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

games_df = g

print("Margin model range:", float(g["mean_margin_model"].min()), "to", float(g["mean_margin_model"].max()))
print("Total model range:", float(g["mean_total_model"].min()), "to", float(g["mean_total_model"].max()))

Margin model range: -19.44733660543507 to 18.616076421248835
Total model range: 215.0 to 250.0


In [ ]:
BLEND_MODEL = 0.4
BLEND_MARKET = 0.6

games_df["market_margin"] = -games_df["spread_home_line"]

games_df["mean_margin"] = (
    BLEND_MODEL * games_df["mean_margin_model"] +
    BLEND_MARKET * games_df["market_margin"]
)

games_df["mean_total"] = (
    BLEND_MODEL * games_df["mean_total_model"] +
    BLEND_MARKET * games_df["total_line"]
)

print("✅ NBA blend applied")
print("Blended margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Blended total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

✅ NBA blend applied
Blended margin range: -10.422312754096987 to 11.890574985180791
Blended total range: 225.3 to 237.3


In [ ]:
import numpy as np
import pandas as pd

# =========================
# NBA SIM SETTINGS
# =========================
SIMS = 50000
SD_MARGIN = 12.0
SD_TOTAL  = 15.0

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# ensure numeric
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# BUILD CARDS
# =========================
# ML (home side only for simplicity)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CORRELATION CAP: max 2 per game
# =========================
kept = []
counts = {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Plays after cap:", len(card))
top = card.head(20).copy()

top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Plays after cap: 16
                                                                                                      discord_text
                     Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 1.0u\nSim Win%: 77.8% | EV: 51.2%\n
               Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.88u\nSim Win%: 72.9% | EV: 41.7%\n
         Memphis Grizzlies at Indiana Pacers\nMemphis Grizzlies -1.0 (-108) — 0.88u\nSim Win%: 73.1% | EV: 40.8%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.58u\nSim Win%: 59.0% | EV: 39.3%\n
                    Cleveland Cavaliers at Brooklyn Nets\nOVER 222.5 (-110) — 0.83u\nSim Win%: 72.1% | EV: 37.6%\n
                 Milwaukee Bucks at Chicago Bulls\nChicago Bulls 3.5 (-110) — 0.74u\nSim Win%: 70.0% | EV: 33.6%\n
                Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.69u\nSim Win%: 68.3% | EV: 32.7%\n
         Portland Trail Blazers at Atlanta Hawks\nAtlanta 

In [ ]:
SD_MARGIN = 14.5
SD_TOTAL  = 18.0
print("✅ Using widened NBA variance:", SD_MARGIN, SD_TOTAL)

✅ Using widened NBA variance: 14.5 18.0


In [ ]:
import numpy as np
import pandas as pd

SIMS = 50000
UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig
ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])
sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])
to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# cap: max 2 plays per game
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Plays after cap:", len(card))
top = card.head(20).copy()

top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_card_widevar.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Plays after cap: 15
                                                                                                      discord_text
                    Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 0.93u\nSim Win%: 73.9% | EV: 43.7%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.52u\nSim Win%: 57.4% | EV: 35.4%\n
               Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.74u\nSim Win%: 69.5% | EV: 35.1%\n
         Memphis Grizzlies at Indiana Pacers\nMemphis Grizzlies -1.0 (-108) — 0.73u\nSim Win%: 69.5% | EV: 33.9%\n
                    Cleveland Cavaliers at Brooklyn Nets\nOVER 222.5 (-110) — 0.69u\nSim Win%: 68.7% | EV: 31.2%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls 3.5 (-110) — 0.6u\nSim Win%: 66.7% | EV: 27.3%\n
                Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.58u\nSim Win%: 65.5% | EV: 27.2%\n
              Cleveland Cavaliers at Brooklyn Nets\nBrookl

In [ ]:
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

games_df["market_margin"] = -pd.to_numeric(games_df["spread_home_line"], errors="coerce")
games_df["mean_margin"] = (
    BLEND_MODEL * pd.to_numeric(games_df["mean_margin_model"], errors="coerce") +
    BLEND_MARKET * games_df["market_margin"]
)

games_df["mean_total"] = (
    BLEND_MODEL * pd.to_numeric(games_df["mean_total_model"], errors="coerce") +
    BLEND_MARKET * pd.to_numeric(games_df["total_line"], errors="coerce")
)

print("✅ Strong NBA blend applied (25/75)")
print("Blended margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Blended total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

✅ Strong NBA blend applied (25/75)
Blended margin range: -10.638945471310617 to 10.806609365737994
Blended total range: 223.875 to 237.75


In [ ]:
# ======================================
# BACK-TO-BACK DETECTION
# ======================================

from datetime import datetime, timedelta

# pull yesterday’s completed games
yesterday = (datetime.strptime(SLATE_DATE, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")

recent_games = historical_df.copy()
recent_games["date"] = pd.to_datetime(recent_games["date"]).dt.date

yesterday_date = datetime.strptime(yesterday, "%Y-%m-%d").date()

teams_played_yesterday = set(
    recent_games.loc[recent_games["date"] == yesterday_date, "home_team"]
).union(
    set(recent_games.loc[recent_games["date"] == yesterday_date, "away_team"])
)

games_df["home_b2b"] = games_df["home_team"].isin(teams_played_yesterday)
games_df["away_b2b"] = games_df["away_team"].isin(teams_played_yesterday)

# apply fatigue penalty
B2B_PENALTY = 1.5

games_df["mean_margin"] = games_df["mean_margin"] \
    - games_df["home_b2b"] * B2B_PENALTY \
    + games_df["away_b2b"] * B2B_PENALTY

print("✅ B2B adjustment applied")
print("Home B2B:", games_df["home_b2b"].sum())
print("Away B2B:", games_df["away_b2b"].sum())

✅ B2B adjustment applied
Home B2B: 4
Away B2B: 5


In [ ]:
# ======================================
# BLOWOUT ADJUSTMENT
# ======================================

BLOWOUT_THRESHOLD = 8

large_spread = games_df["spread_home_line"].abs() >= BLOWOUT_THRESHOLD

# reduce margin magnitude slightly
games_df.loc[large_spread, "mean_margin"] *= 0.90

# reduce totals slightly (garbage time compression)
games_df.loc[large_spread, "mean_total"] -= 1.5

print("✅ Blowout adjustment applied")
print("Large spread games:", large_spread.sum())

✅ Blowout adjustment applied
Large spread games: 2


In [ ]:
Run NBA Stable Engine — STRICT

Settings:
• Sims: 50,000
• Variance: SD_MARGIN=14.5 | SD_TOTAL=18.0
• Market Blend: 25% Model / 75% Market
• B2B Adjustment: ON (−1.5 pts fatigue)
• Blowout Compression: ON (≥8 spread reduce margin 10%, total −1.5)
• Correlation Filter: Max 2 plays per game
• Exposure Rule: No ML + Spread on same team unless both ≥6% EV
• Unit Cap: 1.0u
• Min Unit: 0.25u
• Max Juice: −200
• Remove 0 EV plays

Process:
1) Use blended mean_margin + mean_total
2) Run Monte Carlo simulation (50k)
3) Calculate win probability vs market implied
4) Compute EV
5) Apply Kelly sizing (capped 1u)
6) Remove correlated exposure >2 per game
7) Sort by EV descending
8) Export top 20 to CSV
9) Generate Discord-ready output

Output:
• matchup
• bet
• units
• Sim Win%
• EV%

Date: 2026-03-01

SyntaxError: invalid character '—' (U+2014) (1502304135.py, line 1)

In [ ]:
import numpy as np
import pandas as pd

# =========================
# FINAL NBA RUN CELL
# (uses current games_df with all adjustments already applied)
# =========================

SIMS = 50000

# If you already set these earlier, leave them; otherwise these are your calibrated defaults
SD_MARGIN = globals().get("SD_MARGIN", 14.5)
SD_TOTAL  = globals().get("SD_TOTAL", 18.0)

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# ensure numeric
for c in ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# BUILD MARKET CARDS
# =========================
# ML (home only)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], -sp["spread_home_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].astype(str) + " (" + sp_card["odds"].apply(fmt_odds) + ")"
sp_card["p_win"] = sp_card["p_win"]

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"
to_card["p_win"] = to_card["p_win"]

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    cols = ["matchup","market","bet","units","p_win","ev","commence_time"]
    return x[cols]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CAP: max 2 plays per game
# =========================
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

print("✅ Sims:", SIMS, "| SD_MARGIN:", SD_MARGIN, "| SD_TOTAL:", SD_TOTAL)
print("✅ Plays after cap:", len(card))

top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_final_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Plays after cap: 13
                                                                                                      discord_text
              Cleveland Cavaliers at Brooklyn Nets\nBrooklyn Nets ML (+420) — 0.25u\nSim Win%: 25.3% | EV: 31.7%\n
                  Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.44u\nSim Win%: 55.0% | EV: 29.7%\n
                 Milwaukee Bucks at Chicago Bulls\nChicago Bulls 3.5 (-110) — 0.51u\nSim Win%: 64.5% | EV: 23.1%\n
               Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.46u\nSim Win%: 62.5% | EV: 21.5%\n
                    Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 0.45u\nSim Win%: 62.3% | EV: 21.0%\n
         Portland Trail Blazers at Atlanta Hawks\nAtlanta Hawks -6.0 (-108) — 0.43u\nSim Win%: 62.4% | EV: 20.1%\n
                Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.34u\nSim Win%: 59.7% | EV: 16.1%\n
         

In [ ]:
import numpy as np
import pandas as pd

# Uses your existing games_df that already has:
# mean_margin, mean_total, spread_home_line, spread_away_line, odds, etc
SIMS = 50000
SD_MARGIN = globals().get("SD_MARGIN", 14.5)
SD_TOTAL  = globals().get("SD_TOTAL", 18.0)

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

rng = np.random.default_rng(42)
df = games_df.copy()

# numeric safety
for c in ["home_ml","away_ml","spread_home_line","spread_away_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# =========================
# MONTE CARLO
# =========================
margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

# Spread cover: home covers its listed spread (home_line)
# home covers if margin + home_line > 0  <=> margin > -home_line
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]

df["p_home_win"] = (margins > 0).mean(axis=1)
df["p_over"]  = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"] = 1 - df["p_over"]

# =========================
# HELPERS
# =========================
def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)  # half-Kelly
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# =========================
# ML (home only)
# =========================
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig

ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# =========================
# SPREAD (FIXED LINE SIGN)
# =========================
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()

sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])

# ✅ Use the actual market line from feed (NO NEGATION)
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], sp["spread_away_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])

sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].apply(fmt_line) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# =========================
# TOTAL (best side)
# =========================
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    return x[["matchup","market","bet","units","p_win","ev","commence_time"]]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# =========================
# CAP: max 2 plays per game
# =========================
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1

card = pd.DataFrame(kept).reset_index(drop=True)

# =========================
# DISCORD TEXT
# =========================
top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(pct) +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ FIXED spread signs. Plays after cap:", len(top))
print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_FIXED_card.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ FIXED spread signs. Plays after cap: 13
                                                                                                       discord_text
               Cleveland Cavaliers at Brooklyn Nets\nBrooklyn Nets ML (+420) — 0.25u\nSim Win%: 25.3% | EV: 31.7%\n
                   Milwaukee Bucks at Chicago Bulls\nChicago Bulls ML (+136) — 0.44u\nSim Win%: 55.0% | EV: 29.7%\n
                 Milwaukee Bucks at Chicago Bulls\nChicago Bulls +3.5 (-110) — 0.51u\nSim Win%: 64.5% | EV: 23.1%\n
                Minnesota Timberwolves at Denver Nuggets\nUNDER 237.5 (-106) — 0.46u\nSim Win%: 62.5% | EV: 21.5%\n
                     Philadelphia 76ers at Boston Celtics\nOVER 221.5 (-106) — 0.45u\nSim Win%: 62.3% | EV: 21.0%\n
            Portland Trail Blazers at Atlanta Hawks\nAtlanta Hawks -6 (-108) — 0.43u\nSim Win%: 62.4% | EV: 20.1%\n
                 Portland Trail Blazers at Atlanta Hawks\nUNDER 237.5 (-106) — 0.34u\nSim Win%: 59.7% | EV: 16.1%\n
                   Detroit Pis

In [ ]:
import os, requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"   # <-- 3/2 run

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200

# Market blend (NBA)
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

# Context layers
B2B_PENALTY = 1.5
BLOWOUT_THRESHOLD = 8
BLOWOUT_MARGIN_MULT = 0.90
BLOWOUT_TOTAL_BUMP = -1.5

ODDS_API_KEY = os.getenv("ODDS_API_KEY")
if not ODDS_API_KEY:
    raise ValueError("ODDS_API_KEY not found. Set it in Colab Secrets or env var.")

print("✅ Settings loaded:", SPORT, SLATE_DATE, "| sims:", SIMS)

✅ Settings loaded: basketball_nba 2026-03-02 | sims: 50000


In [ ]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

print("✅ ODDS_API_KEY set in environment")

✅ ODDS_API_KEY set in environment


In [ ]:
def pull_market_data(
    sport: str,
    date: str,
    regions="us",
    markets="h2h,spreads,totals",
    odds_format="american",
    date_filter=True,
):
    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds/"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
    }

    if date_filter:
        day = datetime.strptime(date, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        params["commenceTimeFrom"] = day.isoformat().replace("+00:00", "Z")
        params["commenceTimeTo"] = (day + timedelta(days=1)).isoformat().replace("+00:00", "Z")

    r = requests.get(url, params=params, timeout=30)
    if r.status_code != 200:
        raise Exception(r.text)

    data = r.json()
    rows = []

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        t = g.get("commence_time")

        bks = g.get("bookmakers", [])
        book = bks[0] if bks else None

        h2h = spreads = totals = None
        if book:
            for m in book.get("markets", []):
                if m.get("key") == "h2h":
                    h2h = m
                elif m.get("key") == "spreads":
                    spreads = m
                elif m.get("key") == "totals":
                    totals = m

        def pick_outcome(mkt, name):
            if not mkt: return None
            for o in mkt.get("outcomes", []):
                if o.get("name") == name:
                    return o
            return None

        home_ml = away_ml = np.nan
        if h2h:
            oh = pick_outcome(h2h, home)
            oa = pick_outcome(h2h, away)
            if oh: home_ml = oh.get("price", np.nan)
            if oa: away_ml = oa.get("price", np.nan)

        spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = np.nan
        if spreads:
            oh = pick_outcome(spreads, home)
            oa = pick_outcome(spreads, away)
            if oh:
                spread_home_line = oh.get("point", np.nan)
                spread_home_odds = oh.get("price", np.nan)
            if oa:
                spread_away_line = oa.get("point", np.nan)
                spread_away_odds = oa.get("price", np.nan)

        total_line = over_odds = under_odds = np.nan
        if totals:
            oo = pick_outcome(totals, "Over")
            ou = pick_outcome(totals, "Under")
            if oo:
                total_line = oo.get("point", np.nan)
                over_odds = oo.get("price", np.nan)
            if ou:
                under_odds = ou.get("price", np.nan)

        rows.append({
            "home_team": home,
            "away_team": away,
            "commence_time": t,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": spread_home_line,
            "spread_home_odds": spread_home_odds,
            "spread_away_line": spread_away_line,
            "spread_away_odds": spread_away_odds,
            "total_line": total_line,
            "over_odds": over_odds,
            "under_odds": under_odds,
        })

    df = pd.DataFrame(rows)
    print(f"✅ pulled {len(df)} NBA games for {date}")
    print("ML non-null:", int(df["home_ml"].notna().sum()), "/", len(df))
    print("Spread non-null:", int(df["spread_home_line"].notna().sum()), "/", len(df))
    print("Total non-null:", int(df["total_line"].notna().sum()), "/", len(df))
    return df

games_df = pull_market_data(SPORT, SLATE_DATE, date_filter=True)
games_df.head()

✅ pulled 4 NBA games for 2026-03-02
ML non-null: 4 / 4
Spread non-null: 4 / 4
Total non-null: 4 / 4


,home_team,away_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds
0,Dallas Mavericks,Oklahoma City Thunder,2026-03-02T01:10:00Z,1800,-10000,16.5,-110,-16.5,-120,207.5,-128,-104
1,Boston Celtics,Philadelphia 76ers,2026-03-02T01:12:00Z,-500,340,-8.5,-122,8.5,-108,220.5,102,-136
2,Los Angeles Clippers,New Orleans Pelicans,2026-03-02T02:10:00Z,-1200,630,-14.5,-104,14.5,-128,236.5,-130,-102
3,Los Angeles Lakers,Sacramento Kings,2026-03-02T02:40:00Z,-720,520,-13.0,-110,13.0,-110,235.5,-110,-110


In [ ]:
def pull_completed_scores_safe(sport: str, days_list):
    all_parts = []
    for d in days_list:
        try:
            url = f"https://api.the-odds-api.com/v4/sports/{sport}/scores/"
            params = {"apiKey": ODDS_API_KEY, "daysFrom": int(d)}
            r = requests.get(url, params=params, timeout=30)
            if r.status_code != 200:
                print(f"⚠ daysFrom {d} failed — skipping")
                continue

            data = r.json()
            rows = []
            for g in data:
                if not g.get("completed"):
                    continue
                home = g.get("home_team")
                away = g.get("away_team")
                ct = g.get("commence_time")
                scores = {s["name"]: s["score"] for s in g.get("scores", [])} if g.get("scores") else {}
                hs = scores.get(home)
                as_ = scores.get(away)
                if hs is None or as_ is None:
                    continue
                rows.append({
                    "home_team": home,
                    "away_team": away,
                    "home_score": float(hs),
                    "away_score": float(as_),
                    "date": ct
                })

            part_df = pd.DataFrame(rows)
            print(f"✅ daysFrom {d}: {len(part_df)} games")
            all_parts.append(part_df)

        except Exception as e:
            print(f"⚠ daysFrom {d} error:", str(e))
            continue

    if not all_parts:
        raise ValueError("No historical score data pulled.")
    return pd.concat(all_parts, ignore_index=True).drop_duplicates()

# 🔒 Only use safe values your plan supports
historical_df = pull_completed_scores_safe(SPORT, [3, 6, 10, 14])

print("✅ Unique historical games pulled:", len(historical_df))
historical_df.head()

✅ daysFrom 3: 18 games
⚠ daysFrom 6 failed — skipping
⚠ daysFrom 10 failed — skipping
⚠ daysFrom 14 failed — skipping
✅ Unique historical games pulled: 18


,home_team,away_team,home_score,away_score,date
0,Los Angeles Clippers,Minnesota Timberwolves,88.0,94.0,2026-02-27T03:16:00Z
1,Detroit Pistons,Cleveland Cavaliers,122.0,119.0,2026-02-28T00:13:00Z
2,Boston Celtics,Brooklyn Nets,148.0,111.0,2026-02-28T00:40:00Z
3,Milwaukee Bucks,New York Knicks,98.0,127.0,2026-02-28T01:14:00Z
4,Dallas Mavericks,Memphis Grizzlies,105.0,124.0,2026-02-28T01:41:00Z


In [ ]:
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

g["market_margin"] = -pd.to_numeric(g["spread_home_line"], errors="coerce")

g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * pd.to_numeric(g["total_line"], errors="coerce")

games_df = g

print("✅ Blend applied (25/75)")
print("Margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("Total range:", float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

NameError: name 'team_overall' is not defined

In [ ]:
import pandas as pd
import numpy as np

# ======================================
# ONE-CELL FIX: build team_overall + projections + 25/75 blend
# ======================================

# 1) Ensure historical_df exists
if "historical_df" not in globals():
    raise NameError("historical_df not found. Run the completed scores pull cell first.")

# 2) Build team_overall if missing
if "team_overall" not in globals() or not isinstance(globals().get("team_overall"), pd.DataFrame):
    PPP_LEAGUE = 1.13
    HOME_EDGE_PTS = 2.0

    hist = historical_df.copy()
    hist["total_pts"] = hist["home_score"] + hist["away_score"]
    hist["poss_proxy"] = hist["total_pts"] / PPP_LEAGUE

    home_rows = pd.DataFrame({
        "team": hist["home_team"],
        "points_scored": hist["home_score"],
        "points_allowed": hist["away_score"],
        "poss": hist["poss_proxy"],
    })
    away_rows = pd.DataFrame({
        "team": hist["away_team"],
        "points_scored": hist["away_score"],
        "points_allowed": hist["home_score"],
        "poss": hist["poss_proxy"],
    })

    all_stats = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = all_stats.groupby("team").agg(
        poss=("poss","mean"),
        ppp_for=("points_scored", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean()),
        ppp_against=("points_allowed", lambda x: x.mean() / all_stats.loc[x.index,"poss"].mean())
    ).reset_index()

    team_overall = team_overall.fillna(team_overall.mean(numeric_only=True))
    print("✅ team_overall built:", team_overall.shape)

# 3) Ensure games_df exists
if "games_df" not in globals():
    raise NameError("games_df not found. Run the market pull cell first.")

g = games_df.copy()

# 4) Merge team metrics into slate
g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

# fill any missing with slate means
for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")
    g[c] = g[c].fillna(g[c].mean())

# 5) PPP projection layer
PPP_LEAGUE = 1.13
HOME_EDGE_PTS = 2.0

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_points_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_points_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_points_proj"] - g["away_points_proj"] + HOME_EDGE_PTS
g["mean_total_model"] = g["home_points_proj"] + g["away_points_proj"]

# 6) 25/75 market blend (NBA standard in our pipeline)
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

g["spread_home_line"] = pd.to_numeric(g["spread_home_line"], errors="coerce")
g["total_line"] = pd.to_numeric(g["total_line"], errors="coerce")

g["market_margin"] = -g["spread_home_line"]

g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * g["total_line"]

games_df = g

print("✅ Projection + blend rebuilt")
print("Margin range:", float(games_df['mean_margin'].min()), "to", float(games_df['mean_margin'].max()))
print("Total range:", float(games_df['mean_total'].min()), "to", float(games_df['mean_total'].max()))
games_df[["home_team","away_team","spread_home_line","total_line","mean_margin","mean_total"]].head()

✅ team_overall built: (27, 4)
✅ Projection + blend rebuilt
Margin range: -15.06979636216369 to 12.770537421904887
Total range: 215.25 to 234.625


,home_team,away_team,spread_home_line,total_line,mean_margin,mean_total
0,Dallas Mavericks,Oklahoma City Thunder,16.5,207.5,-15.069796,215.250
1,Boston Celtics,Philadelphia 76ers,-8.5,220.5,10.203770,227.000
2,Los Angeles Clippers,New Orleans Pelicans,-14.5,236.5,9.404658,227.625
3,Los Angeles Lakers,Sacramento Kings,-13.0,235.5,12.770537,234.625


In [ ]:
# B2B: teams that played yesterday (based on completed scores list)
recent_games = historical_df.copy()
recent_games["date"] = pd.to_datetime(recent_games["date"]).dt.date

yesterday = (datetime.strptime(SLATE_DATE, "%Y-%m-%d") - timedelta(days=1)).date()

teams_played_yday = set(recent_games.loc[recent_games["date"] == yesterday, "home_team"]).union(
    set(recent_games.loc[recent_games["date"] == yesterday, "away_team"])
)

games_df["home_b2b"] = games_df["home_team"].isin(teams_played_yday)
games_df["away_b2b"] = games_df["away_team"].isin(teams_played_yday)

games_df["mean_margin"] = games_df["mean_margin"] - games_df["home_b2b"]*B2B_PENALTY + games_df["away_b2b"]*B2B_PENALTY

# Blowout compression
large_spread = games_df["spread_home_line"].abs() >= BLOWOUT_THRESHOLD
games_df.loc[large_spread, "mean_margin"] *= BLOWOUT_MARGIN_MULT
games_df.loc[large_spread, "mean_total"] += BLOWOUT_TOTAL_BUMP

print("✅ B2B + blowout applied")
print("Home B2B:", int(games_df["home_b2b"].sum()), "| Away B2B:", int(games_df["away_b2b"].sum()))
print("Large spread games:", int(large_spread.sum()))

✅ B2B + blowout applied
Home B2B: 1 | Away B2B: 1
Large spread games: 4


In [ ]:
rng = np.random.default_rng(42)
df = games_df.copy()

for c in ["home_ml","away_ml","spread_home_line","spread_away_line","spread_home_odds","spread_away_odds",
          "total_line","over_odds","under_odds","mean_margin","mean_total"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

def american_to_prob(odds):
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds):
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def kelly_fraction(p, dec):
    b = dec - 1
    return (p*dec - 1)/b if b > 0 else 0.0

def to_units(p, odds):
    dec = american_to_decimal(odds)
    f = 0.5 * kelly_fraction(p, dec)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ML (home only)
ml = df.dropna(subset=["home_ml","away_ml"]).copy()
ml["home_imp"] = ml["home_ml"].apply(american_to_prob)
ml["away_imp"] = ml["away_ml"].apply(american_to_prob)
vig = (ml["home_imp"] + ml["away_imp"]).replace(0, np.nan)
ml["home_mkt"] = ml["home_imp"] / vig
ml["dec_home"] = ml["home_ml"].apply(american_to_decimal)
ml["ev"] = ml["p_home_win"]*(ml["dec_home"]-1) - (1-ml["p_home_win"])
ml["units"] = ml.apply(lambda r: to_units(r["p_home_win"], r["home_ml"]), axis=1)

ml_card = ml[(ml["ev"] > 0) & (ml["home_ml"] >= MAX_JUICE) & (ml["units"] > 0)].copy()
ml_card["market"] = "ML"
ml_card["pick_team"] = ml_card["home_team"]
ml_card["bet"] = ml_card["home_team"] + " ML (" + ml_card["home_ml"].apply(fmt_odds) + ")"
ml_card["p_win"] = ml_card["p_home_win"]

# Spread (best side) — FIXED sign (use feed lines as-is)
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
sp["ev_home"] = sp["p_home_cover"]*(sp["spread_home_odds"].apply(american_to_decimal)-1) - (1-sp["p_home_cover"])
sp["ev_away"] = sp["p_away_cover"]*(sp["spread_away_odds"].apply(american_to_decimal)-1) - (1-sp["p_away_cover"])

sp["side"] = np.where(sp["ev_home"] >= sp["ev_away"], "home", "away")
sp["ev"]   = np.where(sp["side"]=="home", sp["ev_home"], sp["ev_away"])
sp["p_win"]= np.where(sp["side"]=="home", sp["p_home_cover"], sp["p_away_cover"])
sp["odds"] = np.where(sp["side"]=="home", sp["spread_home_odds"], sp["spread_away_odds"])
sp["line"] = np.where(sp["side"]=="home", sp["spread_home_line"], sp["spread_away_line"])
sp["team"] = np.where(sp["side"]=="home", sp["home_team"], sp["away_team"])
sp["units"] = sp.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

sp_card = sp[(sp["ev"] > 0) & (sp["units"] > 0)].copy()
sp_card["market"] = "Spread"
sp_card["pick_team"] = sp_card["team"]
sp_card["bet"] = sp_card["team"] + " " + sp_card["line"].apply(fmt_line) + " (" + sp_card["odds"].apply(fmt_odds) + ")"

# Total (best side)
to = df.dropna(subset=["over_odds","under_odds"]).copy()
to["ev_over"]  = to["p_over"]*(to["over_odds"].apply(american_to_decimal)-1) - (1-to["p_over"])
to["ev_under"] = to["p_under"]*(to["under_odds"].apply(american_to_decimal)-1) - (1-to["p_under"])

to["side"] = np.where(to["ev_over"] >= to["ev_under"], "over", "under")
to["ev"]   = np.where(to["side"]=="over", to["ev_over"], to["ev_under"])
to["p_win"]= np.where(to["side"]=="over", to["p_over"], to["p_under"])
to["odds"] = np.where(to["side"]=="over", to["over_odds"], to["under_odds"])
to["units"] = to.apply(lambda r: to_units(r["p_win"], r["odds"]), axis=1)

to_card = to[(to["ev"] > 0) & (to["units"] > 0)].copy()
to_card["market"] = "Total"
to_card["pick_team"] = ""  # totals not tied to a team for double-dip rule
to_card["bet"] = to_card["side"].str.upper() + " " + to_card["total_line"].astype(str) + " (" + to_card["odds"].apply(fmt_odds) + ")"

def finalize(x):
    x = x.assign(matchup=x["away_team"] + " at " + x["home_team"])
    return x[["matchup","market","bet","pick_team","units","p_win","ev","commence_time"]]

out = pd.concat([finalize(ml_card), finalize(sp_card), finalize(to_card)], ignore_index=True)
out = out.sort_values("ev", ascending=False).reset_index(drop=True)

# cap: max 2 plays per matchup (side+total or ML+total, etc.)
kept, counts = [], {}
for _, r in out.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1
card = pd.DataFrame(kept).reset_index(drop=True)

# rule: no ML + Spread on same team (within same matchup) -> drop lower EV
def enforce_no_ml_spread_double_dip(df_card: pd.DataFrame) -> pd.DataFrame:
    out_rows = []
    for matchup, grp in df_card.groupby("matchup", sort=False):
        ml_rows = grp[grp["market"]=="ML"]
        sp_rows = grp[grp["market"]=="Spread"]
        if len(ml_rows) and len(sp_rows):
            ml_team = ml_rows.iloc[0]["pick_team"]
            sp_team = sp_rows.iloc[0]["pick_team"]
            if ml_team == sp_team and ml_team != "":
                # keep the higher EV one, drop the other
                keep_idx = grp["ev"].astype(float).idxmax()
                grp = grp.loc[[keep_idx] + [i for i in grp.index if i!=keep_idx and grp.loc[i,"market"]=="Total"]]
                grp = grp.sort_values("ev", ascending=False)
        out_rows.append(grp)
    return pd.concat(out_rows, ignore_index=True).sort_values("ev", ascending=False).reset_index(drop=True)

card = enforce_no_ml_spread_double_dip(card)

top = card.head(20).copy()
top["discord_text"] = (
    top["matchup"] + "\n" +
    top["bet"] + " — " + top["units"].astype(str) + "u\n" +
    "Sim Win%: " + top["p_win"].apply(lambda x: f"{round(float(x)*100,1)}%") +
    " | EV: " + (top["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ Final plays:", len(top))
print(top[["discord_text"]].to_string(index=False))

fname = f"nba_stable_{SLATE_DATE}_FINAL_CARD.csv"
top.to_csv(fname, index=False)
print("✅ Exported:", fname)

✅ Final plays: 7
                                                                                                          discord_text
        Oklahoma City Thunder at Dallas Mavericks\nDallas Mavericks ML (+1800) — 0.26u\nSim Win%: 17.6% | EV: 234.7%\n
               New Orleans Pelicans at Los Angeles Clippers\nUNDER 236.5 (-102) — 0.86u\nSim Win%: 71.7% | EV: 42.0%\n
                        Philadelphia 76ers at Boston Celtics\nOVER 220.5 (+102) — 0.44u\nSim Win%: 60.7% | EV: 22.6%\n
                   Oklahoma City Thunder at Dallas Mavericks\nOVER 207.5 (-128) — 0.35u\nSim Win%: 63.8% | EV: 13.6%\n
New Orleans Pelicans at Los Angeles Clippers\nNew Orleans Pelicans +14.5 (-128) — 0.31u\nSim Win%: 62.9% | EV: 12.1%\n
             Sacramento Kings at Los Angeles Lakers\nSacramento Kings +13 (-110) — 0.25u\nSim Win%: 57.4% | EV: 9.5%\n
                      Sacramento Kings at Los Angeles Lakers\nUNDER 235.5 (-110) — 0.25u\nSim Win%: 55.2% | EV: 5.4%\n
✅ Exported: nba_stable_2026-03-

In [ ]:
# ======================================
# HARD SLATE LOCK — NBA 3/2 ONLY
# ======================================

TARGET_DATE = "2026-03-02"

import pandas as pd

# Ensure datetime
games_df["commence_time"] = pd.to_datetime(games_df["commence_time"], utc=True)

# Convert to Eastern (NBA logic)
games_df["commence_et"] = games_df["commence_time"].dt.tz_convert("US/Eastern")

games_df["game_date"] = games_df["commence_et"].dt.strftime("%Y-%m-%d")

# STRICT FILTER
before = len(games_df)
games_df = games_df[games_df["game_date"] == TARGET_DATE].copy()
after = len(games_df)

print("=================================")
print("SLATE LOCK APPLIED")
print("Before:", before)
print("After :", after)
print("Target:", TARGET_DATE)
print("=================================")

games_df[["away_team","home_team","commence_et"]]

SLATE LOCK APPLIED
Before: 4
After : 0
Target: 2026-03-02


,away_team,home_team,commence_et


In [ ]:
import requests, pandas as pd
from datetime import datetime, timedelta
import pytz

TARGET_DATE_ET = "2026-03-02"   # the slate date you mean (Eastern day)

ET = pytz.timezone("US/Eastern")
start_et = ET.localize(datetime.strptime(TARGET_DATE_ET, "%Y-%m-%d"))
end_et = start_et + timedelta(days=1)

start_utc = start_et.astimezone(pytz.utc)
end_utc = end_et.astimezone(pytz.utc)

print("ET window:", start_et, "to", end_et)
print("UTC window:", start_utc, "to", end_utc)

url = f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)

data = r.json()

rows = []
for g in data:
    home = g.get("home_team")
    away = g.get("away_team")
    t = g.get("commence_time")

    bks = g.get("bookmakers", [])
    book = bks[0] if bks else None

    h2h = spreads = totals = None
    if book:
        for m in book.get("markets", []):
            if m.get("key") == "h2h":
                h2h = m
            elif m.get("key") == "spreads":
                spreads = m
            elif m.get("key") == "totals":
                totals = m

    def pick_outcome(mkt, name):
        if not mkt: return None
        for o in mkt.get("outcomes", []):
            if o.get("name") == name:
                return o
        return None

    home_ml = away_ml = float("nan")
    if h2h:
        oh = pick_outcome(h2h, home)
        oa = pick_outcome(h2h, away)
        if oh: home_ml = oh.get("price", float("nan"))
        if oa: away_ml = oa.get("price", float("nan"))

    spread_home_line = spread_home_odds = spread_away_line = spread_away_odds = float("nan")
    if spreads:
        oh = pick_outcome(spreads, home)
        oa = pick_outcome(spreads, away)
        if oh:
            spread_home_line = oh.get("point", float("nan"))
            spread_home_odds = oh.get("price", float("nan"))
        if oa:
            spread_away_line = oa.get("point", float("nan"))
            spread_away_odds = oa.get("price", float("nan"))

    total_line = over_odds = under_odds = float("nan")
    if totals:
        oo = pick_outcome(totals, "Over")
        ou = pick_outcome(totals, "Under")
        if oo:
            total_line = oo.get("point", float("nan"))
            over_odds = oo.get("price", float("nan"))
        if ou:
            under_odds = ou.get("price", float("nan"))

    rows.append({
        "home_team": home,
        "away_team": away,
        "commence_time": t,
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": spread_home_line,
        "spread_home_odds": spread_home_odds,
        "spread_away_line": spread_away_line,
        "spread_away_odds": spread_away_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds,
    })

games_df = pd.DataFrame(rows)

# Add ET time + filter (should already be correct, but we keep it consistent)
games_df["commence_time"] = pd.to_datetime(games_df["commence_time"], utc=True)
games_df["commence_et"] = games_df["commence_time"].dt.tz_convert("US/Eastern")
games_df["game_date"] = games_df["commence_et"].dt.strftime("%Y-%m-%d")
games_df = games_df[games_df["game_date"] == TARGET_DATE_ET].copy()

print(f"✅ ET-slate pull complete. Games: {len(games_df)}")
games_df[["away_team","home_team","commence_et","home_ml","spread_home_line","total_line"]].sort_values("commence_et")

ET window: 2026-03-02 00:00:00-05:00 to 2026-03-03 00:00:00-05:00
UTC window: 2026-03-02 05:00:00+00:00 to 2026-03-03 05:00:00+00:00
✅ ET-slate pull complete. Games: 4


,away_team,home_team,commence_et,home_ml,spread_home_line,total_line
0,Houston Rockets,Washington Wizards,2026-03-02 19:10:00-05:00,750,15.5,223.5
1,Boston Celtics,Milwaukee Bucks,2026-03-02 19:40:00-05:00,220,7.0,215.5
2,Denver Nuggets,Utah Jazz,2026-03-02 21:10:00-05:00,380,10.5,243.5
3,Los Angeles Clippers,Golden State Warriors,2026-03-02 22:10:00-05:00,-108,-1.0,219.5


In [ ]:
# ======================================
# NBA STABLE RUN — STRICT (3/2)
# ======================================

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL = 18.0

HOME_EDGE = 2.0
BLEND_MODEL = 0.25
BLEND_MARKET = 0.75

g = games_df.copy()

# ===== PROJECTION LAYER =====
g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_pts"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_pts"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_pts"] - g["away_pts"] + HOME_EDGE
g["mean_total_model"] = g["home_pts"] + g["away_pts"]

# ===== MARKET BLEND =====
g["market_margin"] = -g["spread_home_line"]

g["mean_margin"] = (
    BLEND_MODEL * g["mean_margin_model"] +
    BLEND_MARKET * g["market_margin"]
)

g["mean_total"] = (
    BLEND_MODEL * g["mean_total_model"] +
    BLEND_MARKET * g["total_line"]
)

# ===== B2B ADJUSTMENT =====
if "home_b2b" in g.columns:
    g.loc[g["home_b2b"]==1, "mean_margin"] -= 1.5
if "away_b2b" in g.columns:
    g.loc[g["away_b2b"]==1, "mean_margin"] += 1.5

# ===== SIM ENGINE =====
results = []

for _, row in g.iterrows():
    margin_sims = np.random.normal(row["mean_margin"], SD_MARGIN, SIMS)
    total_sims  = np.random.normal(row["mean_total"], SD_TOTAL, SIMS)

    # Moneyline
    home_win_prob = (margin_sims > 0).mean()
    away_win_prob = 1 - home_win_prob

    # Spread
    spread_prob = (margin_sims > row["spread_home_line"]).mean()

    # Total
    over_prob = (total_sims > row["total_line"]).mean()

    results.append({
        "away_team": row["away_team"],
        "home_team": row["home_team"],
        "home_ml": row["home_ml"],
        "away_ml": row["away_ml"],
        "spread_home_line": row["spread_home_line"],
        "spread_home_odds": row["spread_home_odds"],
        "total_line": row["total_line"],
        "over_odds": row["over_odds"],
        "under_odds": row["under_odds"],
        "home_win_prob": home_win_prob,
        "away_win_prob": away_win_prob,
        "spread_prob": spread_prob,
        "over_prob": over_prob
    })

sim_df = pd.DataFrame(results)

# ===== EV + KELLY =====
def american_to_decimal(odds):
    return 1 + (odds/100 if odds > 0 else 100/abs(odds))

def ev_calc(prob, odds):
    dec = american_to_decimal(odds)
    return prob*dec - 1

def kelly(prob, odds):
    dec = american_to_decimal(odds)
    b = dec - 1
    return max((prob*b - (1-prob))/b, 0)

plays = []

for _, r in sim_df.iterrows():

    # Home ML
    ev = ev_calc(r["home_win_prob"], r["home_ml"])
    if ev > 0:
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "ML",
            "bet": f"{r['home_team']} ML ({r['home_ml']})",
            "prob": r["home_win_prob"],
            "ev": ev,
            "units": min(kelly(r["home_win_prob"], r["home_ml"]), 1.0)
        })

    # Spread
    ev = ev_calc(r["spread_prob"], r["spread_home_odds"])
    if ev > 0:
        line = r["spread_home_line"]
        sign = "+" if line > 0 else ""
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Spread",
            "bet": f"{r['home_team']} {sign}{line} ({r['spread_home_odds']})",
            "prob": r["spread_prob"],
            "ev": ev,
            "units": min(kelly(r["spread_prob"], r["spread_home_odds"]), 1.0)
        })

    # Over
    ev = ev_calc(r["over_prob"], r["over_odds"])
    if ev > 0:
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Total",
            "bet": f"OVER {r['total_line']} ({r['over_odds']})",
            "prob": r["over_prob"],
            "ev": ev,
            "units": min(kelly(r["over_prob"], r["over_odds"]), 1.0)
        })

card = pd.DataFrame(plays)

# ===== CAP LOGIC =====
card = card.sort_values("ev", ascending=False)

final = []
for matchup in card["matchup"].unique():
    sub = card[card["matchup"]==matchup]
    # allow max 2 per game
    final.append(sub.head(2))

final_card = pd.concat(final)

# Discord text
final_card["discord_text"] = (
    final_card["matchup"] + "\n" +
    final_card["bet"] + " — " +
    final_card["units"].round(2).astype(str) + "u\n" +
    "Sim Win%: " + (final_card["prob"]*100).round(1).astype(str) +
    "% | EV: " + (final_card["ev"]*100).round(1).astype(str) + "%\n"
)

print("✅ Sims:", SIMS)
print("✅ Final plays:", len(final_card))

final_card[["discord_text"]]

✅ Sims: 50000
✅ Final plays: 4


,discord_text
0,Houston Rockets at Washington Wizards\nWashing...
1,Houston Rockets at Washington Wizards\nOVER 22...
3,Denver Nuggets at Utah Jazz\nUtah Jazz ML (380...
2,Boston Celtics at Milwaukee Bucks\nOVER 215.5 ...


In [ ]:
import pandas as pd
import numpy as np

# Requires: games_df (fresh ET slate pull) and team_overall (PPP baseline)
if "team_overall" not in globals():
    raise NameError("team_overall not found. Run your PPP baseline build cell first.")

g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

# Fill missing team stats with league means (keeps run stable even if a team didn’t appear in history window)
for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
    g[c] = pd.to_numeric(g[c], errors="coerce")
    g[c] = g[c].fillna(g[c].mean())

games_df = g

print("✅ PPP columns merged into slate")
print(games_df[["away_team","home_team","h_poss","a_poss","h_ppp_for","a_ppp_for"]].head())

✅ PPP columns merged into slate
              away_team              home_team      h_poss      a_poss  \
0       Houston Rockets     Washington Wizards  229.203540  194.690265   
1        Boston Celtics        Milwaukee Bucks  195.575221  229.203540   
2        Denver Nuggets              Utah Jazz  194.690265  209.292035   
3  Los Angeles Clippers  Golden State Warriors  203.539823  161.061947   

   h_ppp_for  a_ppp_for  
0   0.545367   0.539318  
1   0.498529   0.645714  
2   0.539318   0.547082  
3   0.496217   0.546374  


In [ ]:
# ======================================
# FULL STRICT ENGINE CELL — NBA (drop-in)
# Requires already defined: games_df (ET-slate odds), team_overall (PPP baseline), SLATE_DATE (YYYY-MM-DD)
# Uses fixed spread sign logic (home_line/home_odds + away_line/away_odds as-is)
# Outputs: Sim Win% + Market% + EV% + units + discord_text
# Rules: max 2 plays per game; no ML+Spread same team; totals allowed with side
# ======================================

import numpy as np
import pandas as pd

# ---- SETTINGS (keep consistent) ----
SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

HOME_EDGE_PTS = 2.0

BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

B2B_PENALTY_PTS = 1.5          # applied to mean_margin
BLOWOUT_SPREAD_TH = 8.0        # if abs(spread)>=8, compress margin + shave total
BLOWOUT_MARGIN_MULT = 0.90
BLOWOUT_TOTAL_BUMP  = -1.5

UNIT_MIN, UNIT_CAP = 0.25, 1.0
MAX_JUICE = -200               # filter extreme negative prices if desired

rng = np.random.default_rng(42)

# ---- Helpers ----
def american_to_prob(odds: float) -> float:
    odds = float(odds)
    return (-odds)/(-odds+100) if odds < 0 else 100/(odds+100)

def american_to_decimal(odds: float) -> float:
    odds = float(odds)
    return 1 + (odds/100) if odds > 0 else 1 + (100/abs(odds))

def ev_unit(prob: float, odds: float) -> float:
    dec = american_to_decimal(odds)
    return prob*(dec-1) - (1-prob)

def kelly_fraction(prob: float, odds: float) -> float:
    dec = american_to_decimal(odds)
    b = dec - 1
    if b <= 0:
        return 0.0
    return max((prob*dec - 1)/b, 0.0)

def to_units(prob: float, odds: float) -> float:
    # half-Kelly, scaled into 0.25–1.0u band
    f = 0.5 * kelly_fraction(prob, odds)
    u = max(0.0, min(UNIT_CAP, f * 4))
    if 0 < u < UNIT_MIN:
        u = UNIT_MIN
    return round(u, 2)

def fmt_odds(x):
    x = int(float(x))
    return f"+{x}" if x > 0 else str(x)

def fmt_line(x):
    x = float(x)
    return f"+{x:g}" if x > 0 else f"{x:g}"

def pct(x):
    return f"{round(float(x)*100,1)}%"

# ---- 0) Preconditions ----
if "games_df" not in globals():
    raise NameError("games_df not found (run the ET-slate odds pull first).")
if "team_overall" not in globals():
    raise NameError("team_overall not found (run the PPP baseline build first).")

g = games_df.copy()

# Ensure numeric
num_cols = [
    "home_ml","away_ml",
    "spread_home_line","spread_home_odds","spread_away_line","spread_away_odds",
    "total_line","over_odds","under_odds"
]
for c in num_cols:
    if c in g.columns:
        g[c] = pd.to_numeric(g[c], errors="coerce")

# ---- 1) Merge PPP baseline into slate if missing ----
need_cols = {"h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"}
if not need_cols.issubset(set(g.columns)):
    g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
        "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
        "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    for c in ["h_poss","h_ppp_for","h_ppp_against","a_poss","a_ppp_for","a_ppp_against"]:
        g[c] = pd.to_numeric(g[c], errors="coerce")
        g[c] = g[c].fillna(g[c].mean())

# ---- 2) Projection layer (PPP/tempo proxy) ----
g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["home_pts_proj"] = g["home_ppp_proj"] * g["poss_proj"]
g["away_pts_proj"] = g["away_ppp_proj"] * g["poss_proj"]

g["mean_margin_model"] = g["home_pts_proj"] - g["away_pts_proj"] + HOME_EDGE_PTS
g["mean_total_model"]  = g["home_pts_proj"] + g["away_pts_proj"]

# ---- 3) Market blend (25/75) ----
g["market_margin"] = -g["spread_home_line"]
g["mean_margin"] = BLEND_MODEL * g["mean_margin_model"] + BLEND_MARKET * g["market_margin"]
g["mean_total"]  = BLEND_MODEL * g["mean_total_model"]  + BLEND_MARKET * g["total_line"]

# ---- 4) Context layer: B2B + blowout ----
# B2B flags should exist if you ran the B2B cell; if not, default False
if "home_b2b" not in g.columns:
    g["home_b2b"] = False
if "away_b2b" not in g.columns:
    g["away_b2b"] = False

g["mean_margin"] = g["mean_margin"] - g["home_b2b"].astype(int)*B2B_PENALTY_PTS + g["away_b2b"].astype(int)*B2B_PENALTY_PTS

blow = g["spread_home_line"].abs() >= BLOWOUT_SPREAD_TH
g.loc[blow, "mean_margin"] = g.loc[blow, "mean_margin"] * BLOWOUT_MARGIN_MULT
g.loc[blow, "mean_total"]  = g.loc[blow, "mean_total"]  + BLOWOUT_TOTAL_BUMP

# ---- 5) Sims ----
df = g.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

margins = rng.normal(df["mean_margin"].to_numpy()[:, None], SD_MARGIN, size=(len(df), SIMS))
totals  = rng.normal(df["mean_total"].to_numpy()[:, None],  SD_TOTAL,  size=(len(df), SIMS))

df["p_home_win"]   = (margins > 0).mean(axis=1)
df["p_home_cover"] = (margins > (-df["spread_home_line"].to_numpy()[:, None])).mean(axis=1)
df["p_away_cover"] = 1 - df["p_home_cover"]
df["p_over"]       = (totals > df["total_line"].to_numpy()[:, None]).mean(axis=1)
df["p_under"]      = 1 - df["p_over"]

# ---- 6) Market implied (for output) ----
# ML: vig-removed implied
ml_base = df.dropna(subset=["home_ml","away_ml"]).copy()
if len(ml_base):
    ml_base["home_imp"] = ml_base["home_ml"].apply(american_to_prob)
    ml_base["away_imp"] = ml_base["away_ml"].apply(american_to_prob)
    vig = (ml_base["home_imp"] + ml_base["away_imp"]).replace(0, np.nan)
    ml_base["home_mkt"] = ml_base["home_imp"]/vig
else:
    ml_base = df.copy()
    ml_base["home_mkt"] = np.nan

# Spread: use price implied for selected side (not vig-removed; consistent with prior)
# Total: same

# ---- 7) Build candidates (ML/Spread/Total) ----
plays = []

# ML: home only (consistent with prior runs)
for _, r in ml_base.iterrows():
    ev = ev_unit(r["p_home_win"], r["home_ml"])
    if (ev > 0) and (r["home_ml"] >= MAX_JUICE):
        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "ML",
            "pick_team": r["home_team"],
            "bet": f"{r['home_team']} ML ({fmt_odds(r['home_ml'])})",
            "p_win": float(r["p_home_win"]),
            "p_mkt": float(r["home_mkt"]) if pd.notna(r["home_mkt"]) else np.nan,
            "ev": float(ev),
            "units": to_units(float(r["p_home_win"]), float(r["home_ml"])),
            "commence_time": r.get("commence_time", None),
        })

# Spread: choose better side using EV; FIXED sign using feed lines
sp = df.dropna(subset=["spread_home_odds","spread_away_odds"]).copy()
if len(sp):
    sp["ev_home"] = sp.apply(lambda r: ev_unit(r["p_home_cover"], r["spread_home_odds"]), axis=1)
    sp["ev_away"] = sp.apply(lambda r: ev_unit(r["p_away_cover"], r["spread_away_odds"]), axis=1)

    for _, r in sp.iterrows():
        if r["ev_home"] <= 0 and r["ev_away"] <= 0:
            continue

        if r["ev_home"] >= r["ev_away"]:
            team = r["home_team"]
            line = r["spread_home_line"]
            odds = r["spread_home_odds"]
            pwin = r["p_home_cover"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_home"]
        else:
            team = r["away_team"]
            line = r["spread_away_line"]
            odds = r["spread_away_odds"]
            pwin = r["p_away_cover"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_away"]

        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Spread",
            "pick_team": team,
            "bet": f"{team} {fmt_line(line)} ({fmt_odds(odds)})",
            "p_win": float(pwin),
            "p_mkt": float(pmkt),
            "ev": float(evv),
            "units": to_units(float(pwin), float(odds)),
            "commence_time": r.get("commence_time", None),
        })

# Total: choose better side by EV
to = df.dropna(subset=["over_odds","under_odds"]).copy()
if len(to):
    to["ev_over"]  = to.apply(lambda r: ev_unit(r["p_over"],  r["over_odds"]), axis=1)
    to["ev_under"] = to.apply(lambda r: ev_unit(r["p_under"], r["under_odds"]), axis=1)

    for _, r in to.iterrows():
        if r["ev_over"] <= 0 and r["ev_under"] <= 0:
            continue

        if r["ev_over"] >= r["ev_under"]:
            side = "OVER"
            odds = r["over_odds"]
            pwin = r["p_over"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_over"]
        else:
            side = "UNDER"
            odds = r["under_odds"]
            pwin = r["p_under"]
            pmkt = american_to_prob(odds)
            evv  = r["ev_under"]

        plays.append({
            "matchup": f"{r['away_team']} at {r['home_team']}",
            "market": "Total",
            "pick_team": "",
            "bet": f"{side} {r['total_line']} ({fmt_odds(odds)})",
            "p_win": float(pwin),
            "p_mkt": float(pmkt),
            "ev": float(evv),
            "units": to_units(float(pwin), float(odds)),
            "commence_time": r.get("commence_time", None),
        })

card = pd.DataFrame(plays)
if card.empty:
    print("⚠ No +EV plays found.")
    card

# ---- 8) Filter + sort ----
card = card[(card["units"] > 0)].copy()
card = card.sort_values("ev", ascending=False).reset_index(drop=True)

# ---- 9) Exposure rules ----
# (a) max 2 plays per matchup
kept = []
counts = {}
for _, r in card.iterrows():
    m = r["matchup"]
    counts.setdefault(m, 0)
    if counts[m] >= 2:
        continue
    kept.append(r)
    counts[m] += 1
card2 = pd.DataFrame(kept).reset_index(drop=True)

# (b) no ML + Spread on same team in same matchup -> keep higher EV, keep totals
out_rows = []
for matchup, grp in card2.groupby("matchup", sort=False):
    ml_rows = grp[grp["market"]=="ML"]
    sp_rows = grp[grp["market"]=="Spread"]
    if len(ml_rows) and len(sp_rows):
        if ml_rows.iloc[0]["pick_team"] == sp_rows.iloc[0]["pick_team"] and ml_rows.iloc[0]["pick_team"] != "":
            keep_idx = grp["ev"].astype(float).idxmax()
            grp = grp.loc[[keep_idx] + grp.index[grp["market"]=="Total"].tolist()]
            grp = grp.sort_values("ev", ascending=False)
    out_rows.append(grp)

final_card = pd.concat(out_rows, ignore_index=True).sort_values("ev", ascending=False).reset_index(drop=True)

# ---- 10) Discord text ----
final_card["discord_text"] = (
    final_card["matchup"] + "\n" +
    final_card["bet"] + " — " + final_card["units"].astype(str) + "u\n" +
    "Sim Win%: " + final_card["p_win"].apply(pct) +
    " | Market: " + final_card["p_mkt"].apply(lambda x: "NA" if pd.isna(x) else pct(x)) +
    "\nEV: " + (final_card["ev"]*100).round(1).astype(str) + "%\n"
)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Final plays:", len(final_card))
print(final_card[["discord_text"]].to_string(index=False))

# ---- 11) Export ----
try:
    out_name = f"nba_stable_{globals().get('SLATE_DATE','slate')}_FULL_STRICT_CARD.csv"
    final_card.to_csv(out_name, index=False)
    print("✅ Exported:", out_name)
except Exception as e:
    print("⚠ Export failed:", e)

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Final plays: 7
                                                                                                                      discord_text
         Houston Rockets at Washington Wizards\nWashington Wizards ML (+750) — 0.3u\nSim Win%: 25.0% | Market: 11.3%\nEV: 112.2%\n
                            Denver Nuggets at Utah Jazz\nUtah Jazz ML (+380) — 0.27u\nSim Win%: 31.4% | Market: 20.1%\nEV: 50.6%\n
                 Boston Celtics at Milwaukee Bucks\nBoston Celtics -7 (-110) — 0.54u\nSim Win%: 65.3% | Market: 52.4%\nEV: 24.6%\n
                        Boston Celtics at Milwaukee Bucks\nOVER 215.5 (-112) — 0.44u\nSim Win%: 63.1% | Market: 52.8%\nEV: 19.5%\n
                             Denver Nuggets at Utah Jazz\nUNDER 243.5 (-110) — 0.38u\nSim Win%: 61.5% | Market: 52.4%\nEV: 17.4%\n
            Los Angeles Clippers at Golden State Warriors\nUNDER 219.5 (-114) — 0.25u\nSim Win%: 57.6% | Market: 53.3%\nEV: 8.1%\n
Los Angeles Clipp

In [ ]:
import os, numpy as np, pandas as pd
from datetime import datetime, timedelta
import pytz, requests

# Reset key objects so we don't mix old state
for k in ["games_df","df","final_card","injury_overrides","b2b_map","blowout_map"]:
    if k in globals():
        del globals()[k]

# Settings (match last run)
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

print("✅ Settings loaded:", SPORT, SLATE_DATE, "| sims:", SIMS, "| SD:", SD_MARGIN, SD_TOTAL, "| Blend:", BLEND_MODEL, BLEND_MARKET)

✅ Settings loaded: basketball_nba 2026-03-02 | sims: 50000 | SD: 14.5 18.0 | Blend: 0.25 0.75


In [ ]:
ET = pytz.timezone("US/Eastern")
UTC = pytz.utc

start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et   = start_et + timedelta(days=1)
start_utc = start_et.astimezone(UTC)
end_utc   = end_et.astimezone(UTC)

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)
data = r.json()

rows = []
for g in data:
    home = g["home_team"]; away = g["away_team"]
    t = pd.to_datetime(g["commence_time"], utc=True)

    # use first bookmaker
    book = g["bookmakers"][0] if g.get("bookmakers") else None
    if not book:
        continue

    def mkt(key):
        for m in book["markets"]:
            if m["key"] == key: return m
        return None

    def pick(market, name):
        if not market: return None
        for o in market["outcomes"]:
            if o["name"] == name: return o
        return None

    h2h, spr, tot = mkt("h2h"), mkt("spreads"), mkt("totals")

    # ML
    home_ml = away_ml = None
    oh = pick(h2h, home); oa = pick(h2h, away)
    home_ml = oh["price"] if oh else None
    away_ml = oa["price"] if oa else None

    # Spreads
    sh_line = sh_odds = sa_line = sa_odds = None
    oh = pick(spr, home); oa = pick(spr, away)
    sh_line = oh["point"] if oh else None
    sh_odds = oh["price"] if oh else None
    sa_line = oa["point"] if oa else None
    sa_odds = oa["price"] if oa else None

    # Totals
    total_line = over_odds = under_odds = None
    oo = pick(tot, "Over"); ou = pick(tot, "Under")
    total_line = oo["point"] if oo else None
    over_odds  = oo["price"] if oo else None
    under_odds = ou["price"] if ou else None

    rows.append({
        "away_team": away,
        "home_team": home,
        "commence_time": t,
        "commence_et": t.tz_convert("US/Eastern"),
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": sh_line,
        "spread_home_odds": sh_odds,
        "spread_away_line": sa_line,
        "spread_away_odds": sa_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds,
    })

games_df = pd.DataFrame(rows)
num_cols = ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_line","spread_away_odds","total_line","over_odds","under_odds"]
for c in num_cols:
    games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

def implied_prob(american):
    if pd.isna(american): return np.nan
    a = float(american)
    return (-a)/(-a+100) if a < 0 else 100/(a+100)

games_df["home_ml_implied"] = games_df["home_ml"].apply(implied_prob)
games_df["away_ml_implied"] = games_df["away_ml"].apply(implied_prob)

print("✅ ET-slate pull complete. Games:", len(games_df))
games_df[["away_team","home_team","commence_et","home_ml","away_ml","spread_home_line","total_line"]].sort_values("commence_et")

✅ ET-slate pull complete. Games: 4


,away_team,home_team,commence_et,home_ml,away_ml,spread_home_line,total_line
0,Houston Rockets,Washington Wizards,2026-03-02 19:10:00-05:00,610,-900,14.5,225.5
1,Boston Celtics,Milwaukee Bucks,2026-03-02 19:40:00-05:00,116,-136,2.0,215.5
2,Denver Nuggets,Utah Jazz,2026-03-02 21:10:00-05:00,440,-590,12.0,243.5
3,Los Angeles Clippers,Golden State Warriors,2026-03-02 22:10:00-05:00,-102,-116,1.5,215.5


In [ ]:
# ========= Helpers =========
def american_to_decimal(odds):
    o = float(odds)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def ev_unit(p_win, american_odds):
    d = american_to_decimal(american_odds)
    return p_win*(d-1) - (1-p_win)

def half_kelly(p_win, american_odds, max_u=1.0, min_u=0.25):
    d = american_to_decimal(american_odds)
    b = d - 1
    q = 1 - p_win
    f = (b*p_win - q)/b
    f = max(0.0, f) * 0.5
    # map fraction to unit scale (simple)
    u = f * 5.0
    return float(min(max_u, max(min_u, u)))

# ========= Projection Layer =========
g = games_df.copy()

# Market margin from spread (home perspective)
g["market_margin"] = -g["spread_home_line"]

# If you already built mean_margin/mean_total earlier in the notebook, keep them.
# Otherwise start from market as baseline and blend (matches our last run behavior when projection tables are missing).
if "mean_margin" not in g.columns:
    g["mean_margin"] = g["market_margin"].copy()

if "mean_total" not in g.columns:
    g["mean_total"] = g["total_line"].copy()

# Blend (same as last run)
g["mean_margin"] = BLEND_MODEL*g["mean_margin"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total"]  + BLEND_MARKET*g["total_line"]

# ========= Back-to-back + Blowout risk hooks =========
# NOTE: keep logic identical to last run unless you provide overrides.
# You can manually populate these maps if you have news:
b2b_map = {}      # e.g. {"Boston Celtics": -0.5}  # points off margin
blowout_map = {}  # e.g. {"Houston Rockets": -0.3} # total adjustment

g["b2b_adj_home"] = g["home_team"].map(b2b_map).fillna(0.0)
g["b2b_adj_away"] = g["away_team"].map(b2b_map).fillna(0.0)

# apply b2b to margin (home better if away b2b, worse if home b2b)
g["mean_margin"] = g["mean_margin"] + (g["b2b_adj_away"] - g["b2b_adj_home"])

# blowout impacts totals only (optional)
g["mean_total"] = g["mean_total"] + g["home_team"].map(blowout_map).fillna(0.0) + g["away_team"].map(blowout_map).fillna(0.0)

# ========= Monte Carlo Sims =========
g = g.dropna(subset=["mean_margin","mean_total","spread_home_line","total_line"]).reset_index(drop=True)

rng = np.random.default_rng(42)
margins = rng.normal(g["mean_margin"].to_numpy()[:,None], SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(g["mean_total"].to_numpy()[:,None],  SD_TOTAL,  size=(len(g), SIMS))

g["p_home_win"] = (margins > 0).mean(axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

# spread cover prob (home spread line: home -x covers if margin > x; home +x covers if margin > -x)
g["p_home_cover"] = (margins > g["spread_home_line"].to_numpy()[:,None]).mean(axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

# totals
g["p_over"]  = (totals > g["total_line"].to_numpy()[:,None]).mean(axis=1)
g["p_under"] = 1 - g["p_over"]

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")

# ========= Build plays =========
plays = []

for i,row in g.iterrows():
    matchup = f"{row['away_team']} at {row['home_team']}"

    # ML
    if pd.notna(row["home_ml"]):
        p = float(row["p_home_win"])
        ev = ev_unit(p, row["home_ml"])
        plays.append({"matchup":matchup,"market":"ML","bet":f"{row['home_team']} ML ({int(row['home_ml'])})",
                      "p_win":p,"p_mkt":float(row["home_ml_implied"]), "ev":ev, "odds":row["home_ml"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})
    if pd.notna(row["away_ml"]):
        p = float(row["p_away_win"])
        ev = ev_unit(p, row["away_ml"])
        plays.append({"matchup":matchup,"market":"ML","bet":f"{row['away_team']} ML ({int(row['away_ml'])})",
                      "p_win":p,"p_mkt":float(row["away_ml_implied"]), "ev":ev, "odds":row["away_ml"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})

    # Spread (use away line for away bet)
    if pd.notna(row["spread_home_line"]) and pd.notna(row["spread_home_odds"]):
        # Home spread bet: home line as shown
        p = float(row["p_home_cover"])
        ev = ev_unit(p, row["spread_home_odds"])
        sign = "+" if row["spread_home_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{row['home_team']} {sign}{row['spread_home_line']} ({int(row['spread_home_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["spread_home_odds"]), "ev":ev, "odds":row["spread_home_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})
    if pd.notna(row["spread_away_line"]) and pd.notna(row["spread_away_odds"]):
        p = float(row["p_away_cover"])
        ev = ev_unit(p, row["spread_away_odds"])
        sign = "+" if row["spread_away_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{row['away_team']} {sign}{row['spread_away_line']} ({int(row['spread_away_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["spread_away_odds"]), "ev":ev, "odds":row["spread_away_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})

    # Totals
    if pd.notna(row["total_line"]) and pd.notna(row["over_odds"]):
        p = float(row["p_over"])
        ev = ev_unit(p, row["over_odds"])
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"OVER {row['total_line']} ({int(row['over_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["over_odds"]), "ev":ev, "odds":row["over_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})
    if pd.notna(row["total_line"]) and pd.notna(row["under_odds"]):
        p = float(row["p_under"])
        ev = ev_unit(p, row["under_odds"])
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"UNDER {row['total_line']} ({int(row['under_odds'])})",
                      "p_win":p,"p_mkt":implied_prob(row["under_odds"]), "ev":ev, "odds":row["under_odds"],
                      "home_team":row["home_team"],"away_team":row["away_team"],"commence_et":row["commence_et"]})

plays_df = pd.DataFrame(plays)

# Keep +EV only (same behavior as last)
plays_df = plays_df[plays_df["ev"] > 0].copy()

# Units (half Kelly) with 0.25–1.0u bounds
plays_df["units"] = plays_df.apply(lambda r: half_kelly(r["p_win"], r["odds"]), axis=1)

# ========= Exposure cap =========
# Rule: max 2 picks per game; allow side+total; avoid ML+spread same team in same game
plays_df = plays_df.sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

final_rows = []
by_game = {}

for _,r in plays_df.iterrows():
    game = r["matchup"]
    by_game.setdefault(game, [])

    # max 2 per game
    if len(by_game[game]) >= 2:
        continue

    # block ML+SPREAD same team in same game
    if r["market"] in ["ML","SPREAD"]:
        team = None
        if r["market"] == "ML":
            team = r["bet"].split(" ML")[0]
        else:
            # spread bet starts with team name
            team = r["bet"].split(" (")[0]
            # team name is everything before last space (line)
            team = " ".join(team.split(" ")[:-1])

        for existing in by_game[game]:
            if existing["market"] in ["ML","SPREAD"]:
                ex_team = existing.get("team_tag")
                if ex_team == team:
                    team = team  # no-op
                    break
        else:
            pass

    # side+total is ok; ML+spread same team is blocked:
    if r["market"] in ["ML","SPREAD"]:
        team_tag = None
        if r["market"] == "ML":
            team_tag = r["bet"].split(" ML")[0]
        else:
            team_tag = " ".join(r["bet"].split(" (")[0].split(" ")[:-1])

        conflict = any((ex["market"] in ["ML","SPREAD"] and ex.get("team_tag")==team_tag) for ex in by_game[game])
        if conflict:
            continue
        r = r.to_dict()
        r["team_tag"] = team_tag
        by_game[game].append(r)
        final_rows.append(r)
    else:
        r = r.to_dict()
        r["team_tag"] = ""
        by_game[game].append(r)
        final_rows.append(r)

final_card = pd.DataFrame(final_rows).sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

# ========= Discord Text =========
def fmt_pct(x): return f"{100*x:.1f}%"

final_card["discord_text"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {r['units']:.2f}u\n"
    f"Sim Win%: {fmt_pct(r['p_win'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*r['ev']:.1f}%\n", axis=1)

print("✅ Final plays:", len(final_card))
final_card[["discord_text"]].head(50)

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Final plays: 6


,discord_text
0,Houston Rockets at Washington Wizards\nHouston...
1,Denver Nuggets at Utah Jazz\nDenver Nuggets -1...
2,Boston Celtics at Milwaukee Bucks\nBoston Celt...
3,Houston Rockets at Washington Wizards\nWashing...
4,Los Angeles Clippers at Golden State Warriors\...
5,Denver Nuggets at Utah Jazz\nUtah Jazz ML (440...


In [ ]:
# ======================================
# OFFICIAL NBA INJURY REPORT -> TEAM STATUS MAP
# (drop-in, no hunting)
# ======================================

import pandas as pd, re, requests
from datetime import datetime
import pdfplumber

# Latest official PDF for today (you can swap time if needed)
INJURY_PDF_URL = "https://ak-static.cms.nba.com/referee/injury/Injury-Report_2026-03-02_02_00PM.pdf"

pdf_path = "/content/nba_injury_report_2026-03-02.pdf"
with open(pdf_path, "wb") as f:
    f.write(requests.get(INJURY_PDF_URL, timeout=30).content)

# Teams on our slate
slate_teams = set(games_df["home_team"]).union(set(games_df["away_team"]))

# Parse PDF text
lines = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        txt = page.extract_text() or ""
        for ln in txt.split("\n"):
            lines.append(ln.strip())

# Build a simple table of (team, player, status)
# We capture: "... <Team> <Player Name> <Status> ..."
# Statuses in NBA report typically include: Out, Doubtful, Questionable, Probable, Available
status_pat = re.compile(r"\b(Out|Doubtful|Questionable|Probable|Available)\b", re.IGNORECASE)

rows = []
for ln in lines:
    m = status_pat.search(ln)
    if not m:
        continue
    status = m.group(1).title()

    # try to locate which slate team appears in line
    team_hit = None
    for t in slate_teams:
        if t in ln:
            team_hit = t
            break
    if not team_hit:
        continue

    # crude player extraction: text between team name and status word
    after_team = ln.split(team_hit, 1)[-1].strip()
    player_part = after_team.split(status, 1)[0].strip(" -;:")
    player = player_part if player_part else "UNKNOWN"

    rows.append({"team": team_hit, "player": player, "status": status, "raw": ln})

inj_df = pd.DataFrame(rows).drop_duplicates(subset=["team","player","status"])

print("✅ Official injury rows (slate teams):", len(inj_df))
display(inj_df.head(30))

ModuleNotFoundError: No module named 'pdfplumber'

In [ ]:
# ======================================
# MANUAL INJURY OVERRIDES (CONSISTENT INPUT)
# ======================================

# Add confirmed statuses here from UnderdogNBA / official reports
# Format:
# "Team": {"OUT": n, "DOUBTFUL": n, "QUESTIONABLE": n}

injury_override = {
    # Example — replace with today's confirmed news
    # "Boston Celtics": {"OUT": 1, "QUESTIONABLE": 1},
    # "Milwaukee Bucks": {"OUT": 0, "QUESTIONABLE": 2},
}

STATUS_MARGIN = {
    "OUT": -2.5,
    "DOUBTFUL": -1.5,
    "QUESTIONABLE": -0.8,
}

STATUS_TOTAL = {
    "OUT": -0.8,
    "DOUBTFUL": -0.5,
    "QUESTIONABLE": -0.25,
}

team_margin_adj = {}
team_total_adj = {}

for team in set(games_df["home_team"]).union(set(games_df["away_team"])):
    team_margin_adj[team] = 0.0
    team_total_adj[team] = 0.0

for team, statuses in injury_override.items():
    for status, count in statuses.items():
        team_margin_adj[team] += STATUS_MARGIN.get(status, 0.0) * count
        team_total_adj[team]  += STATUS_TOTAL.get(status, 0.0) * count

print("✅ Injury adjustments applied:")
for t in team_margin_adj:
    if team_margin_adj[t] != 0:
        print(t, "margin:", team_margin_adj[t], "| total:", team_total_adj[t])

✅ Injury adjustments applied:


In [ ]:
import os, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta
import pytz

# Required
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"
ODDS_API_KEY = os.getenv("ODDS_API_KEY")

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-02"

# Must match last run
SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0

# Blend (same as last run)
BLEND_MODEL  = 0.25
BLEND_MARKET = 0.75

print("✅ Settings:", SPORT, SLATE_DATE, "| Sims:", SIMS, "| SD:", SD_MARGIN, SD_TOTAL, "| Blend:", BLEND_MODEL, BLEND_MARKET)

✅ Settings: basketball_nba 2026-03-02 | Sims: 50000 | SD: 14.5 18.0 | Blend: 0.25 0.75


In [ ]:
ET = pytz.timezone("US/Eastern")
UTC = pytz.utc

start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et   = start_et + timedelta(days=1)
start_utc = start_et.astimezone(UTC)
end_utc   = end_et.astimezone(UTC)

url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/odds/"
params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso",
    "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
    "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
}

r = requests.get(url, params=params, timeout=30)
if r.status_code != 200:
    raise Exception(r.text)
data = r.json()

rows = []
for g in data:
    home = g["home_team"]; away = g["away_team"]
    t = pd.to_datetime(g["commence_time"], utc=True)

    # grab first bookmaker
    if not g.get("bookmakers"):
        continue
    book = g["bookmakers"][0]

    def mkt(key):
        for m in book["markets"]:
            if m["key"] == key: return m
        return None

    def pick(market, name):
        if not market: return None
        for o in market["outcomes"]:
            if o["name"] == name: return o
        return None

    h2h, spr, tot = mkt("h2h"), mkt("spreads"), mkt("totals")

    oh, oa = pick(h2h, home), pick(h2h, away)
    home_ml = oh["price"] if oh else None
    away_ml = oa["price"] if oa else None

    oh, oa = pick(spr, home), pick(spr, away)
    sh_line = oh["point"] if oh else None
    sh_odds = oh["price"] if oh else None
    sa_line = oa["point"] if oa else None
    sa_odds = oa["price"] if oa else None

    oo, ou = pick(tot, "Over"), pick(tot, "Under")
    total_line = oo["point"] if oo else None
    over_odds  = oo["price"] if oo else None
    under_odds = ou["price"] if ou else None

    rows.append({
        "away_team": away,
        "home_team": home,
        "commence_time": t,
        "commence_et": t.tz_convert("US/Eastern"),
        "home_ml": home_ml,
        "away_ml": away_ml,
        "spread_home_line": sh_line,
        "spread_home_odds": sh_odds,
        "spread_away_line": sa_line,
        "spread_away_odds": sa_odds,
        "total_line": total_line,
        "over_odds": over_odds,
        "under_odds": under_odds
    })

games_df = pd.DataFrame(rows)
num_cols = ["home_ml","away_ml","spread_home_line","spread_home_odds","spread_away_line","spread_away_odds","total_line","over_odds","under_odds"]
for c in num_cols:
    games_df[c] = pd.to_numeric(games_df[c], errors="coerce")

print("✅ Games pulled:", len(games_df))
games_df[["away_team","home_team","commence_et","home_ml","spread_home_line","total_line"]].sort_values("commence_et")

✅ Games pulled: 4


,away_team,home_team,commence_et,home_ml,spread_home_line,total_line
0,Houston Rockets,Washington Wizards,2026-03-02 19:10:00-05:00,640,14.5,225.5
1,Boston Celtics,Milwaukee Bucks,2026-03-02 19:40:00-05:00,118,2.0,215.5
2,Denver Nuggets,Utah Jazz,2026-03-02 21:10:00-05:00,440,12.0,243.5
3,Los Angeles Clippers,Golden State Warriors,2026-03-02 22:10:00-05:00,-102,1.5,215.5


In [ ]:
def implied_prob(american):
    if pd.isna(american): return np.nan
    a = float(american)
    return (-a)/(-a+100) if a < 0 else 100/(a+100)

def american_to_decimal(odds):
    o = float(odds)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def ev_unit(p_win, american_odds):
    d = american_to_decimal(american_odds)
    return p_win*(d-1) - (1-p_win)

def half_kelly_units(p_win, american_odds, min_u=0.25, max_u=1.0):
    d = american_to_decimal(american_odds)
    b = d - 1
    q = 1 - p_win
    f = (b*p_win - q)/b
    f = max(0.0, f) * 0.5  # half kelly
    u = f * 5.0           # scale
    return float(min(max_u, max(min_u, u)))

In [ ]:
# Add the injury news you saw (counts only)
injury_override = {
    # "Boston Celtics": {"OUT": 1, "QUESTIONABLE": 1},
}

STATUS_MARGIN = {"OUT": -2.5, "DOUBTFUL": -1.5, "QUESTIONABLE": -0.8}
STATUS_TOTAL  = {"OUT": -0.8, "DOUBTFUL": -0.5, "QUESTIONABLE": -0.25}

teams = set(games_df["home_team"]).union(set(games_df["away_team"]))
team_margin_adj = {t: 0.0 for t in teams}
team_total_adj  = {t: 0.0 for t in teams}

for team, statuses in injury_override.items():
    for status, count in statuses.items():
        team_margin_adj[team] += STATUS_MARGIN.get(status, 0.0) * count
        team_total_adj[team]  += STATUS_TOTAL.get(status, 0.0) * count

print("✅ Injury adj ready (non-zero only):")
for t in sorted(teams):
    if team_margin_adj[t] != 0 or team_total_adj[t] != 0:
        print(t, "| margin:", team_margin_adj[t], "| total:", team_total_adj[t])

✅ Injury adj ready (non-zero only):


In [ ]:
g = games_df.copy()

# Market baseline
g["market_margin"] = -g["spread_home_line"].astype(float)

# Base projections (market-based unless you have a projection layer)
g["mean_margin"] = g["market_margin"].copy()
g["mean_total"]  = g["total_line"].astype(float).copy()

# Blend layer (same as last run)
g["mean_margin"] = BLEND_MODEL*g["mean_margin"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total"]  + BLEND_MARKET*g["total_line"].astype(float)

# Injury layer
g["inj_home"] = g["home_team"].map(team_margin_adj).fillna(0.0)
g["inj_away"] = g["away_team"].map(team_margin_adj).fillna(0.0)
g["mean_margin"] = g["mean_margin"] + (g["inj_away"] - g["inj_home"])

g["mean_total"] = g["mean_total"] \
    + g["home_team"].map(team_total_adj).fillna(0.0) \
    + g["away_team"].map(team_total_adj).fillna(0.0)

# Drop any incomplete rows
g = g.dropna(subset=["mean_margin","mean_total","spread_home_line","spread_away_line","total_line"]).reset_index(drop=True)

# Monte Carlo
rng = np.random.default_rng(42)
margins = rng.normal(g["mean_margin"].to_numpy()[:,None], SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(g["mean_total"].to_numpy()[:,None],  SD_TOTAL,  size=(len(g), SIMS))

g["p_home_win"] = (margins > 0).mean(axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

g["p_home_cover"] = (margins > g["spread_home_line"].to_numpy()[:,None]).mean(axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

g["p_over"]  = (totals > g["total_line"].to_numpy()[:,None]).mean(axis=1)
g["p_under"] = 1 - g["p_over"]

# Build plays list
plays = []
for _, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # ML
    if pd.notna(r["home_ml"]):
        p = float(r["p_home_win"]); odds = r["home_ml"]
        plays.append({"matchup":matchup,"market":"ML","bet":f"{r['home_team']} ML ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})
    if pd.notna(r["away_ml"]):
        p = float(r["p_away_win"]); odds = r["away_ml"]
        plays.append({"matchup":matchup,"market":"ML","bet":f"{r['away_team']} ML ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})

    # Spread (already correct sign from API)
    if pd.notna(r["spread_home_odds"]):
        p = float(r["p_home_cover"]); odds = r["spread_home_odds"]
        sign = "+" if r["spread_home_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{r['home_team']} {sign}{r['spread_home_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})
    if pd.notna(r["spread_away_odds"]):
        p = float(r["p_away_cover"]); odds = r["spread_away_odds"]
        sign = "+" if r["spread_away_line"] > 0 else ""
        plays.append({"matchup":matchup,"market":"SPREAD",
                      "bet":f"{r['away_team']} {sign}{r['spread_away_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})

    # Totals
    if pd.notna(r["over_odds"]):
        p = float(r["p_over"]); odds = r["over_odds"]
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"OVER {r['total_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})
    if pd.notna(r["under_odds"]):
        p = float(r["p_under"]); odds = r["under_odds"]
        plays.append({"matchup":matchup,"market":"TOTAL",
                      "bet":f"UNDER {r['total_line']} ({int(odds)})",
                      "p_win":p,"p_mkt":implied_prob(odds),"ev":ev_unit(p, odds),"odds":odds})

plays_df = pd.DataFrame(plays)

# +EV only + unit sizing
plays_df = plays_df[plays_df["ev"] > 0].copy()
plays_df["units"] = plays_df.apply(lambda x: half_kelly_units(x["p_win"], x["odds"]), axis=1)

# Exposure cap: max 2 plays per game, allow side+total, block ML+spread same team
plays_df = plays_df.sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

final = []
picked = {}

def team_from_bet(bet, market):
    if market == "ML":
        return bet.split(" ML")[0]
    if market == "SPREAD":
        core = bet.split(" (")[0]
        return " ".join(core.split(" ")[:-1])
    return ""

for _, r in plays_df.iterrows():
    game = r["matchup"]
    picked.setdefault(game, [])

    if len(picked[game]) >= 2:
        continue

    t = team_from_bet(r["bet"], r["market"])
    if r["market"] in ["ML","SPREAD"]:
        if any((x["market"] in ["ML","SPREAD"] and x.get("team")==t) for x in picked[game]):
            continue

    entry = r.to_dict()
    entry["team"] = t
    picked[game].append(entry)
    final.append(entry)

final_card = pd.DataFrame(final).sort_values(["ev","p_win"], ascending=False).reset_index(drop=True)

# Discord text
final_card["discord_text"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {r['units']:.2f}u\n"
    f"Sim Win%: {100*r['p_win']:.1f}% | Market: {100*r['p_mkt']:.1f}%\n"
    f"EV: {100*r['ev']:.1f}%\n", axis=1)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print("✅ Final plays:", len(final_card))
final_card[["discord_text"]].head(50)

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Final plays: 6


,discord_text
0,Houston Rockets at Washington Wizards\nHouston...
1,Denver Nuggets at Utah Jazz\nDenver Nuggets -1...
2,Houston Rockets at Washington Wizards\nWashing...
3,Boston Celtics at Milwaukee Bucks\nBoston Celt...
4,Los Angeles Clippers at Golden State Warriors\...
5,Denver Nuggets at Utah Jazz\nUtah Jazz ML (440...


In [ ]:
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

✅ Exported: nba_stable_2026-03-02_FULL_STRICT_CARD.csv


In [ ]:
 need = ["p_home_win","p_away_win","p_home_cover","p_over","ev","units","discord_text"]
missing = [c for c in need if c not in final_card.columns]

print("Rows in final_card:", len(final_card))
print("Missing required STRICT-layer columns:", missing)

print("Sims check (should exist):", "SIMS" in globals(), "| SIMS =", globals().get("SIMS"))

Rows in final_card: 6
Missing required STRICT-layer columns: ['p_home_win', 'p_away_win', 'p_home_cover', 'p_over']
Sims check (should exist): True | SIMS = 50000


In [ ]:
# ======================================
# FINALIZE + EXPORT (FULL LAYERS INCLUDED)
# Paste at bottom and run once
# ======================================

import pandas as pd

# Safety checks
if "final_card" not in globals():
    raise ValueError("final_card not found. Re-run the FULL STRICT engine cell first.")
if "g" not in globals():
    raise ValueError("g (sim dataframe) not found. Re-run the FULL STRICT engine cell first.")
if "SLATE_DATE" not in globals():
    SLATE_DATE = "2026-03-02"

# Build matchup key in g if missing
if "matchup" not in g.columns:
    if not {"away_team","home_team"}.issubset(g.columns):
        raise ValueError("g is missing away_team/home_team needed to build matchup key.")
    g = g.copy()
    g["matchup"] = g["away_team"] + " at " + g["home_team"]

# Keep only the sim probability layers we need
sim_cols = ["matchup","p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]
missing_sim = [c for c in sim_cols if c not in g.columns]
if missing_sim:
    raise ValueError(f"g is missing sim columns: {missing_sim}. Re-run the FULL STRICT engine cell.")

sim_layers = g[sim_cols].drop_duplicates("matchup")

# Merge into final_card
final_card = final_card.merge(sim_layers, on="matchup", how="left")

# Confirm we have the required layers now
need = ["p_home_win","p_away_win","p_home_cover","p_over"]
missing_after = [c for c in need if c not in final_card.columns or final_card[c].isna().all()]
print("✅ Merge complete. Missing/empty after merge:", missing_after)

# Export
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_COMPLETE.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

# Quick preview
display(final_card.head(25))

✅ Merge complete. Missing/empty after merge: []
✅ Exported: nba_stable_2026-03-02_FULL_STRICT_CARD_COMPLETE.csv


,matchup,market,bet,p_win,p_mkt,ev,odds,units,team,discord_text,p_home_win,p_away_win,p_home_cover,p_away_cover,p_over,p_under
0,Houston Rockets at Washington Wizards,SPREAD,Houston Rockets -14.5 (-110),0.97654,0.523810,0.864304,-110,1.000000,Houston Rockets,Houston Rockets at Washington Wizards\nHouston...,0.15998,0.84002,0.02346,0.97654,0.50356,0.49644
1,Denver Nuggets at Utah Jazz,SPREAD,Denver Nuggets -12.0 (-110),0.94930,0.523810,0.812300,-110,1.000000,Denver Nuggets,Denver Nuggets at Utah Jazz\nDenver Nuggets -1...,0.20504,0.79496,0.05070,0.94930,0.50024,0.49976
2,Houston Rockets at Washington Wizards,ML,Washington Wizards ML (640),0.15998,0.135135,0.183852,640,0.250000,Washington Wizards,Houston Rockets at Washington Wizards\nWashing...,0.15998,0.84002,0.02346,0.97654,0.50356,0.49644
3,Boston Celtics at Milwaukee Bucks,SPREAD,Boston Celtics -2.0 (-110),0.61300,0.523810,0.170273,-110,0.468250,Boston Celtics,Boston Celtics at Milwaukee Bucks\nBoston Celt...,0.44046,0.55954,0.38700,0.61300,0.49732,0.50268
4,Los Angeles Clippers at Golden State Warriors,SPREAD,Los Angeles Clippers -1.5 (-108),0.57758,0.519231,0.112376,-108,0.303416,Los Angeles Clippers,Los Angeles Clippers at Golden State Warriors\...,0.46418,0.53582,0.42242,0.57758,0.50000,0.50000
5,Denver Nuggets at Utah Jazz,ML,Utah Jazz ML (440),0.20504,0.185185,0.107216,440,0.250000,Utah Jazz,Denver Nuggets at Utah Jazz\nUtah Jazz ML (440...,0.20504,0.79496,0.05070,0.94930,0.50024,0.49976


In [ ]:
# ======================================
# REBUILD DISCORD_TEXT WITH CORRECT SIM WIN%
# (uses merged p_home_win / p_home_cover / p_over, etc.)
# ======================================

def fmt_pct(x):
    return f"{100*float(x):.1f}%"

def detect_side_team(bet: str):
    # "Boston Celtics -2.0 (-110)" -> "Boston Celtics"
    return bet.split(" (")[0].rsplit(" ", 1)[0]

def detect_line(bet: str):
    # returns numeric line if present, else None
    core = bet.split(" (")[0]
    try:
        return float(core.rsplit(" ", 1)[-1])
    except:
        return None

def market_simwin(row):
    m = row["market"]
    bet = row["bet"]

    if m == "ML":
        team = bet.split(" ML")[0]
        if team == row["home_team"] if "home_team" in row else False:
            return row["p_home_win"]
        # safer: compare to matchup string
        home = row["matchup"].split(" at ")[1]
        away = row["matchup"].split(" at ")[0]
        return row["p_home_win"] if team == home else row["p_away_win"]

    if m == "SPREAD":
        side_team = detect_side_team(bet)
        home = row["matchup"].split(" at ")[1]
        away = row["matchup"].split(" at ")[0]
        return row["p_home_cover"] if side_team == home else row["p_away_cover"]

    if m == "TOTAL":
        if bet.startswith("OVER"):
            return row["p_over"]
        if bet.startswith("UNDER"):
            return row["p_under"]

    return row.get("p_win", np.nan)

# Ensure we have matchup + probability cols
need = ["matchup","market","bet","units","p_mkt","ev","p_home_win","p_away_win","p_home_cover","p_away_cover","p_over","p_under"]
missing = [c for c in need if c not in final_card.columns]
if missing:
    raise ValueError(f"final_card missing columns: {missing}")

# Add helper columns
final_card = final_card.copy()
final_card["sim_win_pct"] = final_card.apply(market_simwin, axis=1)

# Rebuild discord text
final_card["discord_text_fixed"] = final_card.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {float(r['units']):.2f}u\n"
    f"Sim Win%: {fmt_pct(r['sim_win_pct'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*float(r['ev']):.1f}%\n"
, axis=1)

print("\n=== DISCORD CARD (FIXED SIM WIN%) ===\n")
for t in final_card["discord_text_fixed"].tolist():
    print(t)

# Export fixed version too
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_COMPLETE_FIXEDTEXT.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)


=== DISCORD CARD (FIXED SIM WIN%) ===

Houston Rockets at Washington Wizards
Houston Rockets -14.5 (-110) — 1.00u
Sim Win%: 97.7% | Market: 52.4%
EV: 86.4%

Denver Nuggets at Utah Jazz
Denver Nuggets -12.0 (-110) — 1.00u
Sim Win%: 94.9% | Market: 52.4%
EV: 81.2%

Houston Rockets at Washington Wizards
Washington Wizards ML (640) — 0.25u
Sim Win%: 16.0% | Market: 13.5%
EV: 18.4%

Boston Celtics at Milwaukee Bucks
Boston Celtics -2.0 (-110) — 0.47u
Sim Win%: 61.3% | Market: 52.4%
EV: 17.0%

Los Angeles Clippers at Golden State Warriors
Los Angeles Clippers -1.5 (-108) — 0.30u
Sim Win%: 57.8% | Market: 51.9%
EV: 11.2%

Denver Nuggets at Utah Jazz
Utah Jazz ML (440) — 0.25u
Sim Win%: 20.5% | Market: 18.5%
EV: 10.7%

✅ Exported: nba_stable_2026-03-02_FULL_STRICT_CARD_COMPLETE_FIXEDTEXT.csv


In [ ]:
g[["away_team","home_team","mean_margin","spread_home_line","mean_total"]]

,away_team,home_team,mean_margin,spread_home_line,mean_total
0,Houston Rockets,Washington Wizards,-14.5,14.5,225.5
1,Boston Celtics,Milwaukee Bucks,-2.0,2.0,215.5
2,Denver Nuggets,Utah Jazz,-12.0,12.0,243.5
3,Los Angeles Clippers,Golden State Warriors,-1.5,1.5,215.5


In [ ]:
# ======================================
# HOTFIX AT BOTTOM: CORRECT SPREAD PROBS (NO HUNTING)
# - fixes p_home_cover / p_away_cover sign logic
# - updates SPREAD rows in final_card + discord text
# ======================================

import numpy as np
import pandas as pd

# ---- safety ----
if "g" not in globals():
    raise ValueError("Missing 'g' (sim dataframe). Re-run FULL STRICT engine cell first.")
if "final_card" not in globals():
    raise ValueError("Missing 'final_card'. Re-run FULL STRICT engine + merge/export cells first.")
if "margins" not in globals():
    raise ValueError("Missing 'margins' sim matrix. Re-run FULL STRICT engine cell first.")

# ---- ensure matchup in g ----
gg = g.copy()
if "matchup" not in gg.columns:
    gg["matchup"] = gg["away_team"] + " at " + gg["home_team"]

# ---- recompute correct cover probs ----
spread_home = gg["spread_home_line"].to_numpy()[:, None].astype(float)

# Home covers if (home - away) > -home_spread
gg["p_home_cover_fix"] = (margins > -spread_home).mean(axis=1)
gg["p_away_cover_fix"] = 1.0 - gg["p_home_cover_fix"]

# ---- helper funcs (if not already in notebook) ----
def implied_prob(american):
    if pd.isna(american): return np.nan
    a = float(american)
    return (-a)/(-a+100) if a < 0 else 100/(a+100)

def american_to_decimal(odds):
    o = float(odds)
    return 1 + (o/100.0) if o > 0 else 1 + (100.0/abs(o))

def ev_unit(p_win, american_odds):
    d = american_to_decimal(american_odds)
    return p_win*(d-1) - (1-p_win)

def half_kelly_units(p_win, american_odds, min_u=0.25, max_u=1.0):
    d = american_to_decimal(american_odds)
    b = d - 1
    q = 1 - p_win
    f = (b*p_win - q)/b
    f = max(0.0, f) * 0.5
    u = f * 5.0
    return float(min(max_u, max(min_u, u)))

def detect_side_team(bet: str):
    # "Boston Celtics -2.0 (-110)" -> "Boston Celtics"
    return bet.split(" (")[0].rsplit(" ", 1)[0]

def fmt_pct(x):
    return f"{100*float(x):.1f}%"

# ---- map fixed cover probs into final_card SPREAD rows ----
fix_map = gg[["matchup","p_home_cover_fix","p_away_cover_fix"]].drop_duplicates("matchup")

fc = final_card.copy()
fc = fc.merge(fix_map, on="matchup", how="left")

# Determine whether spread bet is home side or away side, then assign corrected p_win
def corrected_spread_pwin(row):
    if row["market"] != "SPREAD":
        return row["p_win"]
    home = row["matchup"].split(" at ")[1]
    side_team = detect_side_team(row["bet"])
    return row["p_home_cover_fix"] if side_team == home else row["p_away_cover_fix"]

fc["p_win"] = fc.apply(corrected_spread_pwin, axis=1)

# Recompute EV/market/units ONLY for SPREAD rows
is_spread = fc["market"] == "SPREAD"
fc.loc[is_spread, "p_mkt"] = fc.loc[is_spread, "odds"].apply(implied_prob)
fc.loc[is_spread, "ev"] = fc.loc[is_spread].apply(lambda r: ev_unit(r["p_win"], r["odds"]), axis=1)
fc.loc[is_spread, "units"] = fc.loc[is_spread].apply(lambda r: half_kelly_units(r["p_win"], r["odds"]), axis=1)

# Rebuild discord text (correct win% by market)
def market_simwin(row):
    if row["market"] == "ML":
        team = row["bet"].split(" ML")[0]
        home = row["matchup"].split(" at ")[1]
        return row["p_home_win"] if team == home else row["p_away_win"]
    if row["market"] == "SPREAD":
        return row["p_win"]
    if row["market"] == "TOTAL":
        return row["p_over"] if row["bet"].startswith("OVER") else row["p_under"]
    return row["p_win"]

fc["sim_win_pct"] = fc.apply(market_simwin, axis=1)

fc["discord_text_fixed"] = fc.apply(lambda r:
    f"{r['matchup']}\n"
    f"{r['bet']} — {float(r['units']):.2f}u\n"
    f"Sim Win%: {fmt_pct(r['sim_win_pct'])} | Market: {fmt_pct(r['p_mkt'])}\n"
    f"EV: {100*float(r['ev']):.1f}%\n"
, axis=1)

# Replace final_card
final_card = fc.drop(columns=["p_home_cover_fix","p_away_cover_fix"], errors="ignore")

print("✅ Spread prob sign hotfix applied.")
print("Preview:")
display(final_card[["matchup","market","bet","p_win","p_mkt","ev","units"]].head(20))

# Print Discord-ready card
print("\n=== DISCORD CARD (POST-HOTFIX) ===\n")
for t in final_card["discord_text_fixed"].tolist():
    print(t)

# Export
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_COMPLETE_HOTFIX.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

✅ Spread prob sign hotfix applied.
Preview:


,matchup,market,bet,p_win,p_mkt,ev,units
0,Houston Rockets at Washington Wizards,SPREAD,Houston Rockets -14.5 (-110),0.50230,0.523810,-0.041064,0.25
1,Denver Nuggets at Utah Jazz,SPREAD,Denver Nuggets -12.0 (-110),0.50154,0.523810,-0.042515,0.25
2,Houston Rockets at Washington Wizards,ML,Washington Wizards ML (640),0.15998,0.135135,0.183852,0.25
3,Boston Celtics at Milwaukee Bucks,SPREAD,Boston Celtics -2.0 (-110),0.50522,0.523810,-0.035489,0.25
4,Los Angeles Clippers at Golden State Warriors,SPREAD,Los Angeles Clippers -1.5 (-108),0.49604,0.519231,-0.044664,0.25
5,Denver Nuggets at Utah Jazz,ML,Utah Jazz ML (440),0.20504,0.185185,0.107216,0.25



=== DISCORD CARD (POST-HOTFIX) ===

Houston Rockets at Washington Wizards
Houston Rockets -14.5 (-110) — 0.25u
Sim Win%: 50.2% | Market: 52.4%
EV: -4.1%

Denver Nuggets at Utah Jazz
Denver Nuggets -12.0 (-110) — 0.25u
Sim Win%: 50.2% | Market: 52.4%
EV: -4.3%

Houston Rockets at Washington Wizards
Washington Wizards ML (640) — 0.25u
Sim Win%: 16.0% | Market: 13.5%
EV: 18.4%

Boston Celtics at Milwaukee Bucks
Boston Celtics -2.0 (-110) — 0.25u
Sim Win%: 50.5% | Market: 52.4%
EV: -3.5%

Los Angeles Clippers at Golden State Warriors
Los Angeles Clippers -1.5 (-108) — 0.25u
Sim Win%: 49.6% | Market: 51.9%
EV: -4.5%

Denver Nuggets at Utah Jazz
Utah Jazz ML (440) — 0.25u
Sim Win%: 20.5% | Market: 18.5%
EV: 10.7%

✅ Exported: nba_stable_2026-03-02_FULL_STRICT_CARD_COMPLETE_HOTFIX.csv


In [ ]:
# ======================================
# FINAL POSTABLE CARD: +EV ONLY + EXPOSURE CAP + DISCORD PRINT
# ======================================

import pandas as pd

fc = final_card.copy()

# Keep +EV only
fc = fc[fc["ev"] > 0].copy()

# Re-apply exposure rules (max 2 per game; allow side+total; block ML+spread same team)
def team_from_bet(bet, market):
    if market == "ML":
        return bet.split(" ML")[0]
    if market == "SPREAD":
        core = bet.split(" (")[0]
        return " ".join(core.split(" ")[:-1])
    return ""

fc = fc.sort_values(["ev","sim_win_pct"], ascending=False).reset_index(drop=True)

picked = {}
final = []

for _, r in fc.iterrows():
    game = r["matchup"]
    picked.setdefault(game, [])

    if len(picked[game]) >= 2:
        continue

    t = team_from_bet(r["bet"], r["market"])
    if r["market"] in ["ML","SPREAD"]:
        if any((x["market"] in ["ML","SPREAD"] and x.get("team") == t) for x in picked[game]):
            continue

    entry = r.to_dict()
    entry["team"] = t
    picked[game].append(entry)
    final.append(entry)

post_card = pd.DataFrame(final).sort_values(["ev","sim_win_pct"], ascending=False).reset_index(drop=True)

print("✅ POSTABLE plays:", len(post_card))
print("\n=== DISCORD CARD (POSTABLE / +EV ONLY) ===\n")
for t in post_card["discord_text_fixed"].tolist():
    print(t)

out_path = f"nba_stable_{SLATE_DATE}_POSTABLE_CARD.csv"
post_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

✅ POSTABLE plays: 2

=== DISCORD CARD (POSTABLE / +EV ONLY) ===

Houston Rockets at Washington Wizards
Washington Wizards ML (640) — 0.25u
Sim Win%: 16.0% | Market: 13.5%
EV: 18.4%

Denver Nuggets at Utah Jazz
Utah Jazz ML (440) — 0.25u
Sim Win%: 20.5% | Market: 18.5%
EV: 10.7%

✅ Exported: nba_stable_2026-03-02_POSTABLE_CARD.csv


In [1]:
# =========================
# NBA STABLE — STRICT RESET
# =========================

SLATE_DATE = "2026-03-03"
SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL = 18.0
BLEND_MODEL = 0.5
BLEND_MARKET = 0.5
HALF_KELLY = True

print("✅ STRICT settings locked")
print("Date:", SLATE_DATE, "| Sims:", SIMS)

✅ STRICT settings locked
Date: 2026-03-03 | Sims: 50000


In [3]:
# =========================
# SET ODDS API KEY (STRICT)
# =========================

import os

ODDS_API_KEY = "10ceac94f9b9cb76f8c65510ca6df18f"
os.environ["ODDS_API_KEY"] = ODDS_API_KEY

print("✅ ODDS_API_KEY set")

✅ ODDS_API_KEY set


In [4]:
# =========================
# ET SLATE PULL (3/3)
# =========================

import pandas as pd
import requests
from datetime import datetime
import pytz

ET = pytz.timezone("US/Eastern")

start_et = ET.localize(datetime.strptime(SLATE_DATE, "%Y-%m-%d"))
end_et = start_et + pd.Timedelta(days=1)

start_utc = start_et.astimezone(pytz.utc)
end_utc = end_et.astimezone(pytz.utc)

print("ET window:", start_et, "to", end_et)
print("UTC window:", start_utc, "to", end_utc)

params = {
    "apiKey": ODDS_API_KEY,
    "regions": "us",
    "markets": "h2h,spreads,totals",
    "oddsFormat": "american",
    "dateFormat": "iso"
}

r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/odds", params=params)
games = r.json()

rows = []
for g in games:
    commence = pd.to_datetime(g["commence_time"])
    if start_utc <= commence <= end_utc:
        home = g["home_team"]
        away = g["away_team"]

        book = g["bookmakers"][0]["markets"]

        data = {
            "away_team": away,
            "home_team": home,
            "commence_utc": commence
        }

        for m in book:
            if m["key"] == "h2h":
                for o in m["outcomes"]:
                    if o["name"] == home:
                        data["home_ml"] = o["price"]
                    else:
                        data["away_ml"] = o["price"]

            if m["key"] == "spreads":
                for o in m["outcomes"]:
                    if o["name"] == home:
                        data["spread_home_line"] = o["point"]
                        data["spread_home_odds"] = o["price"]

            if m["key"] == "totals":
                for o in m["outcomes"]:
                    if o["name"] == "Over":
                        data["total_line"] = o["point"]

        rows.append(data)

games_df = pd.DataFrame(rows)

print("✅ Games pulled:", len(games_df))
games_df.head()

ET window: 2026-03-03 00:00:00-05:00 to 2026-03-04 00:00:00-05:00
UTC window: 2026-03-03 05:00:00+00:00 to 2026-03-04 05:00:00+00:00
✅ Games pulled: 10


,away_team,home_team,commence_utc,home_ml,away_ml,spread_home_line,spread_home_odds,total_line
0,Dallas Mavericks,Charlotte Hornets,2026-03-04 00:10:00+00:00,-620,460,-12.5,-112,229.5
1,Detroit Pistons,Cleveland Cavaliers,2026-03-04 00:10:00+00:00,-102,-116,1.0,-114,227.5
2,Washington Wizards,Orlando Magic,2026-03-04 00:10:00+00:00,-1100,700,-15.5,-110,227.5
3,Brooklyn Nets,Miami Heat,2026-03-04 00:40:00+00:00,-670,490,-12.5,-114,227.5
4,New York Knicks,Toronto Raptors,2026-03-04 00:40:00+00:00,110,-130,1.5,-106,223.5


In [24]:
# =========================
# CELL A — COMPLETED SCORES (STRICT / VALID daysFrom)
# =========================
import pandas as pd, requests, time

SPORT = "basketball_nba"

def pull_completed_scores(days_from: int):
    url = f"https://api.the-odds-api.com/v4/sports/{SPORT}/scores"
    params = {"apiKey": ODDS_API_KEY, "daysFrom": days_from, "dateFormat": "iso"}
    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise Exception(r.text)
    data = r.json()
    rows = []
    for g in data:
        if not g.get("completed"):
            continue
        home = g["home_team"]
        away = g["away_team"]
        hs = g.get("scores", [{}])[0].get("score")
        as_ = g.get("scores", [{}, {}])[1].get("score") if len(g.get("scores", [])) > 1 else None
        if hs is None or as_ is None:
            continue
        rows.append({
            "home_team": home,
            "away_team": away,
            "home_score": float(hs),
            "away_score": float(as_),
            "commence_time": g.get("commence_time")
        })
    return pd.DataFrame(rows)

# Only use safe values; expand later if they work in your account
SAFE_DAYS = [3]
parts = []
for d in SAFE_DAYS:
    try:
        part = pull_completed_scores(d)
        print(f"✅ daysFrom {d}: {len(part)} games")
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed:", str(e)[:140])

games_hist = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["home_team","away_team","commence_time"])
print("✅ Unique completed games:", len(games_hist))
games_hist.head()

✅ daysFrom 3: 19 games
✅ Unique completed games: 19


,home_team,away_team,home_score,away_score,commence_time
0,Charlotte Hornets,Portland Trail Blazers,109.0,93.0,2026-02-28T18:12:00Z
1,Miami Heat,Houston Rockets,115.0,105.0,2026-02-28T20:43:00Z
2,Washington Wizards,Toronto Raptors,125.0,134.0,2026-03-01T00:13:00Z
3,Golden State Warriors,Los Angeles Lakers,101.0,129.0,2026-03-01T01:41:00Z
4,Utah Jazz,New Orleans Pelicans,105.0,115.0,2026-03-01T02:42:00Z


In [25]:
# =========================
# CELL B — TEAM PPP/POSS TABLE (STRICT)
# =========================
import numpy as np
import pandas as pd

# Approx possessions from score environment using league-average points/poss proxy
# We use margin/total sims downstream; here we only need consistent team-strength signals.
games_hist["total_pts"] = games_hist["home_score"] + games_hist["away_score"]
games_hist["margin_home"] = games_hist["home_score"] - games_hist["away_score"]

# Possession proxy: normalize by league avg PPP ~ 1.12 (NBA typical)
LEAGUE_PPP = 1.12
games_hist["poss_proxy"] = games_hist["total_pts"] / LEAGUE_PPP

# Team points per poss proxy (for/against)
home_rows = pd.DataFrame({
    "team": games_hist["home_team"],
    "ppp_for": games_hist["home_score"] / games_hist["poss_proxy"],
    "ppp_against": games_hist["away_score"] / games_hist["poss_proxy"],
    "poss": games_hist["poss_proxy"]
})
away_rows = pd.DataFrame({
    "team": games_hist["away_team"],
    "ppp_for": games_hist["away_score"] / games_hist["poss_proxy"],
    "ppp_against": games_hist["home_score"] / games_hist["poss_proxy"],
    "poss": games_hist["poss_proxy"]
})

team_overall = pd.concat([home_rows, away_rows], ignore_index=True)\
    .groupby("team", as_index=False)\
    .agg(ppp_for=("ppp_for","mean"),
         ppp_against=("ppp_against","mean"),
         poss=("poss","mean"))

print("✅ team_overall built:", team_overall.shape)
team_overall.head()

✅ team_overall built: (29, 4)


,team,ppp_for,ppp_against,poss
0,Atlanta Hawks,0.640678,0.479322,210.714286
1,Boston Celtics,0.621132,0.498868,179.017857
2,Brooklyn Nets,0.549231,0.570769,185.714286
3,Charlotte Hornets,0.604356,0.515644,180.357143
4,Chicago Bulls,0.619355,0.500645,193.750000


In [26]:
# =========================
# CELL C — MERGE + PROJECTIONS (STRICT)
# =========================
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

# Fill any missing teams with league means so the engine never breaks
for c in ["h_poss","a_poss","h_ppp_for","a_ppp_for","h_ppp_against","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["mean_total"] = (g["home_ppp_proj"] + g["away_ppp_proj"]) * g["poss_proj"]
g["mean_margin"] = (g["home_ppp_proj"] - g["away_ppp_proj"]) * g["poss_proj"]

# Market blend (same as last run)
g["market_margin"] = -g["spread_home_line"]
g["mean_margin"] = BLEND_MODEL*g["mean_margin"] + BLEND_MARKET*g["market_margin"]
g["mean_total"]  = BLEND_MODEL*g["mean_total"]  + BLEND_MARKET*g["total_line"]

print("✅ Projections built")
print("Margin range:", float(g["mean_margin"].min()), "to", float(g["mean_margin"].max()))
print("Total range :", float(g["mean_total"].min()),  "to", float(g["mean_total"].max()))
g[["away_team","home_team","mean_margin","spread_home_line","mean_total","total_line","home_ml","away_ml"]].head()

✅ Projections built
Margin range: -11.225192900258516 to 13.18728291946842
Total range : 217.25 to 236.27499999999998


,away_team,home_team,mean_margin,spread_home_line,mean_total,total_line,home_ml,away_ml
0,Dallas Mavericks,Charlotte Hornets,13.187283,-12.5,217.250,229.5,-620,460
1,Detroit Pistons,Cleveland Cavaliers,-2.478696,1.0,218.925,227.5,-102,-116
2,Washington Wizards,Orlando Magic,8.391003,-15.5,226.275,227.5,-1100,700
3,Brooklyn Nets,Miami Heat,10.547465,-12.5,222.775,227.5,-700,500
4,New York Knicks,Toronto Raptors,-4.548719,1.5,226.125,223.5,110,-130


In [27]:
# =========================
# CELL D — FULL STRICT ENGINE (ML/SPREAD/TOTAL) + KELLY
# =========================
import numpy as np
import pandas as pd
from math import erf, sqrt

def american_to_prob(a):
    a = float(a)
    if a > 0:
        return 100.0/(a+100.0)
    return (-a)/((-a)+100.0)

def american_to_decimal(a):
    a = float(a)
    if a > 0:
        return 1 + a/100.0
    return 1 + 100.0/(-a)

def kelly_fraction(p, dec_odds):
    b = dec_odds - 1
    q = 1 - p
    f = (b*p - q)/b
    return max(0.0, f)

rng = np.random.default_rng(7)

rows = []
for _,r in g.iterrows():
    # simulate margin and total
    sim_margin = rng.normal(loc=r["mean_margin"], scale=SD_MARGIN, size=SIMS)
    sim_total  = rng.normal(loc=r["mean_total"],  scale=SD_TOTAL,  size=SIMS)

    # Win prob (home wins if margin > 0)
    p_home_win = float((sim_margin > 0).mean())
    p_away_win = 1 - p_home_win

    # Spread cover: home spread line is usually negative for fav.
    sh = float(r["spread_home_line"])
    # Home covers if (home_score - away_score) > -sh
    p_home_cover = float((sim_margin > -sh).mean())
    p_away_cover = 1 - p_home_cover

    # Total
    tl = float(r["total_line"])
    p_over  = float((sim_total > tl).mean())
    p_under = 1 - p_over

    # Market implied
    # ML
    if pd.notna(r.get("home_ml")) and pd.notna(r.get("away_ml")):
        p_home_mkt = american_to_prob(r["home_ml"])
        p_away_mkt = american_to_prob(r["away_ml"])
    else:
        p_home_mkt = np.nan
        p_away_mkt = np.nan

    # Spread odds (use home odds for both sides when symmetrical; if away odds missing, reuse)
    ho = float(r.get("spread_home_odds", -110))
    ao = float(r.get("spread_away_odds", -110)) if "spread_away_odds" in r else ho

    p_sp_home_mkt = american_to_prob(ho)
    p_sp_away_mkt = american_to_prob(ao)

    # Total odds
    oo = float(r.get("over_odds", -110)) if "over_odds" in r else -110
    uo = float(r.get("under_odds", -110)) if "under_odds" in r else -110
    p_over_mkt = american_to_prob(oo)
    p_under_mkt = american_to_prob(uo)

    # Build candidates: ML, Spread, Total
    matchup = f"{r['away_team']} at {r['home_team']}"
    commence = r.get("commence_utc")

    # ML candidates (only if we have MLs)
    if pd.notna(r.get("home_ml")) and pd.notna(r.get("away_ml")):
        # Home ML
        dec = american_to_decimal(r["home_ml"])
        ev = p_home_win*(dec-1) - (1-p_home_win)
        f = kelly_fraction(p_home_win, dec)
        if HALF_KELLY: f *= 0.5
        rows.append({
            "matchup": matchup, "market": "ML", "team": r["home_team"],
            "bet": f"{r['home_team']} ML ({int(r['home_ml'])})",
            "p_win": p_home_win, "p_mkt": p_home_mkt, "ev": ev,
            "odds": r["home_ml"], "units": min(1.0, max(0.25, f))
        })
        # Away ML
        dec = american_to_decimal(r["away_ml"])
        ev = p_away_win*(dec-1) - (1-p_away_win)
        f = kelly_fraction(p_away_win, dec)
        if HALF_KELLY: f *= 0.5
        rows.append({
            "matchup": matchup, "market": "ML", "team": r["away_team"],
            "bet": f"{r['away_team']} ML ({int(r['away_ml'])})",
            "p_win": p_away_win, "p_mkt": p_away_mkt, "ev": ev,
            "odds": r["away_ml"], "units": min(1.0, max(0.25, f))
        })

    # Spread (home side)
    dec = american_to_decimal(ho)
    ev = p_home_cover*(dec-1) - (1-p_home_cover)
    f = kelly_fraction(p_home_cover, dec)
    if HALF_KELLY: f *= 0.5
    rows.append({
        "matchup": matchup, "market": "SPREAD", "team": r["home_team"],
        "bet": f"{r['home_team']} {sh:+g} ({int(ho)})",
        "p_win": p_home_cover, "p_mkt": p_sp_home_mkt, "ev": ev,
        "odds": ho, "units": min(1.0, max(0.25, f))
    })

    # Total Over/Under
    dec = american_to_decimal(oo)
    ev = p_over*(dec-1) - (1-p_over)
    f = kelly_fraction(p_over, dec)
    if HALF_KELLY: f *= 0.5
    rows.append({
        "matchup": matchup, "market": "TOTAL", "team": "OVER",
        "bet": f"OVER {tl:g} ({int(oo)})",
        "p_win": p_over, "p_mkt": p_over_mkt, "ev": ev,
        "odds": oo, "units": min(1.0, max(0.25, f))
    })

    dec = american_to_decimal(uo)
    ev = p_under*(dec-1) - (1-p_under)
    f = kelly_fraction(p_under, dec)
    if HALF_KELLY: f *= 0.5
    rows.append({
        "matchup": matchup, "market": "TOTAL", "team": "UNDER",
        "bet": f"UNDER {tl:g} ({int(uo)})",
        "p_win": p_under, "p_mkt": p_under_mkt, "ev": ev,
        "odds": uo, "units": min(1.0, max(0.25, f))
    })

final_card = pd.DataFrame(rows)

# +EV only
final_card = final_card[final_card["ev"] > 0].copy()

# Correlation cap: max 2 plays per game
final_card["game_key"] = final_card["matchup"]
final_card = final_card.sort_values(["ev","p_win"], ascending=False)\
    .groupby("game_key").head(2)\
    .reset_index(drop=True)

# Build Discord text
def fmt_line(x):
    return (f"{x['matchup']}\n"
            f"{x['bet']} — {x['units']:.2f}u\n"
            f"Sim Win%: {x['p_win']*100:.1f}% | Market: {x['p_mkt']*100:.1f}%\n"
            f"EV: {x['ev']*100:.1f}%\n")

final_card["discord_text"] = final_card.apply(fmt_line, axis=1)

print("✅ Sims:", SIMS, "| Plays:", len(final_card))
print(final_card[["discord_text"]].head(20))

# Export
out_path = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD.csv"
final_card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

✅ Sims: 50000 | Plays: 19
                                         discord_text
0   Washington Wizards at Orlando Magic\nWashingto...
1   Memphis Grizzlies at Minnesota Timberwolves\nM...
2   Oklahoma City Thunder at Chicago Bulls\nChicag...
3   Dallas Mavericks at Charlotte Hornets\nUNDER 2...
4   Brooklyn Nets at Miami Heat\nBrooklyn Nets ML ...
5   San Antonio Spurs at Philadelphia 76ers\nPhila...
6   Oklahoma City Thunder at Chicago Bulls\nUNDER ...
7   San Antonio Spurs at Philadelphia 76ers\nUNDER...
8   Detroit Pistons at Cleveland Cavaliers\nUNDER ...
9   Brooklyn Nets at Miami Heat\nUNDER 227.5 (-114...
10  Memphis Grizzlies at Minnesota Timberwolves\nU...
11  New York Knicks at Toronto Raptors\nNew York K...
12  New York Knicks at Toronto Raptors\nOVER 223.5...
13  New Orleans Pelicans at Los Angeles Lakers\nLo...
14  Detroit Pistons at Cleveland Cavaliers\nDetroi...
15  New Orleans Pelicans at Los Angeles Lakers\nUN...
16  Phoenix Suns at Sacramento Kings\nSacramento K...
17

In [19]:
# =========================
# STRICT TRIM — QUALITY FILTER
# =========================

EV_FLOOR = 0.04   # 4% minimum edge

postable = final_card[final_card["ev"] > EV_FLOOR].copy()

# Keep max 2 per game (already capped, but re-apply cleanly)
postable = postable.sort_values(["ev","p_win"], ascending=False)\
    .groupby("game_key").head(2)\
    .reset_index(drop=True)

print("✅ Postable plays:", len(postable))
postable[["matchup","bet","p_win","ev","units"]]

✅ Postable plays: 15


,matchup,bet,p_win,ev,units
0,Washington Wizards at Orlando Magic,Washington Wizards ML (700),0.27090,1.167200,0.25
1,Memphis Grizzlies at Minnesota Timberwolves,Memphis Grizzlies ML (610),0.29940,1.125740,0.25
2,Oklahoma City Thunder at Chicago Bulls,Chicago Bulls ML (350),0.36520,0.643400,0.25
3,Dallas Mavericks at Charlotte Hornets,UNDER 229.5 (-110),0.75310,0.437736,0.25
4,San Antonio Spurs at Philadelphia 76ers,Philadelphia 76ers ML (235),0.40784,0.366264,0.25
5,Brooklyn Nets at Miami Heat,Brooklyn Nets ML (490),0.23154,0.366086,0.25
6,Oklahoma City Thunder at Chicago Bulls,UNDER 227.5 (-110),0.68874,0.314867,0.25
7,San Antonio Spurs at Philadelphia 76ers,UNDER 232.5 (-110),0.68856,0.314524,0.25
8,Detroit Pistons at Cleveland Cavaliers,UNDER 227.5 (-110),0.68300,0.303909,0.25
9,Brooklyn Nets at Miami Heat,UNDER 227.5 (-110),0.60240,0.150036,0.25


In [11]:
# =========================
# MARKET STABILIZER (STRICT PATCH)
# =========================

BLEND_MODEL = 0.35
BLEND_MARKET = 0.65

g["mean_margin"] = (
    BLEND_MODEL * g["mean_margin"] +
    BLEND_MARKET * (-g["spread_home_line"])
)

g["mean_total"] = (
    BLEND_MODEL * g["mean_total"] +
    BLEND_MARKET * g["total_line"]
)

print("✅ Stronger market anchor applied")
print("New margin range:",
      float(g["mean_margin"].min()),
      "to",
      float(g["mean_margin"].max()))

✅ Stronger market anchor applied
New margin range: -10.626908757545241 to 14.255925509864827


In [14]:
# =========================
# BOTTOM RESET (HIST + TEAM OVERALL)
# =========================
import pandas as pd
import numpy as np

SAFE_DAYS = [3, 6, 10, 14, 21, 28]

parts = []
for d in SAFE_DAYS:
    try:
        part = pull_completed_scores(SPORT, d)  # uses your existing function
        print(f"✅ daysFrom {d}: {len(part)} games")
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed — skipping. Reason: {str(e)[:140]}")

if not parts:
    raise RuntimeError("No historical score pulls succeeded. Cannot rebuild team_overall.")

games_hist = pd.concat(parts, ignore_index=True)

# Standardize expected columns if present
keep_cols = [c for c in ["home_team","away_team","home_score","away_score","date","commence_time"] if c in games_hist.columns]
games_hist = games_hist[keep_cols].copy() if keep_cols else games_hist.copy()

# Drop obvious junk / duplicates
for c in ["home_score","away_score"]:
    if c in games_hist.columns:
        games_hist[c] = pd.to_numeric(games_hist[c], errors="coerce")

games_hist = games_hist.dropna(subset=[c for c in ["home_team","away_team","home_score","away_score"] if c in games_hist.columns])
games_hist = games_hist.drop_duplicates(subset=[c for c in ["home_team","away_team","date","commence_time"] if c in games_hist.columns])

print("✅ Unique historical games stored:", len(games_hist))
display(games_hist.head())

⚠ daysFrom 3 failed — skipping. Reason: pull_completed_scores() takes 1 positional argument but 2 were given
⚠ daysFrom 6 failed — skipping. Reason: pull_completed_scores() takes 1 positional argument but 2 were given
⚠ daysFrom 10 failed — skipping. Reason: pull_completed_scores() takes 1 positional argument but 2 were given
⚠ daysFrom 14 failed — skipping. Reason: pull_completed_scores() takes 1 positional argument but 2 were given
⚠ daysFrom 21 failed — skipping. Reason: pull_completed_scores() takes 1 positional argument but 2 were given
⚠ daysFrom 28 failed — skipping. Reason: pull_completed_scores() takes 1 positional argument but 2 were given


RuntimeError: No historical score pulls succeeded. Cannot rebuild team_overall.

In [15]:
# =========================
# BOTTOM RESET (HIST + TEAM OVERALL) — FIXED SIGNATURE
# =========================
import pandas as pd
import numpy as np

SAFE_DAYS = [3, 6, 10, 14, 21, 28]

parts = []
for d in SAFE_DAYS:
    try:
        # In THIS notebook, pull_completed_scores takes ONLY (days_from)
        part = pull_completed_scores(d)
        print(f"✅ daysFrom {d}: {len(part)} games")
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed — skipping. Reason: {str(e)[:140]}")

if not parts:
    raise RuntimeError("No historical score pulls succeeded. Cannot rebuild team_overall.")

games_hist = pd.concat(parts, ignore_index=True)

# Normalize columns (different notebooks may name them differently)
rename_map = {}
if "homeScore" in games_hist.columns: rename_map["homeScore"] = "home_score"
if "awayScore" in games_hist.columns: rename_map["awayScore"] = "away_score"
if "home_team" not in games_hist.columns and "homeTeam" in games_hist.columns: rename_map["homeTeam"] = "home_team"
if "away_team" not in games_hist.columns and "awayTeam" in games_hist.columns: rename_map["awayTeam"] = "away_team"
if rename_map:
    games_hist = games_hist.rename(columns=rename_map)

need = ["home_team","away_team","home_score","away_score"]
missing = [c for c in need if c not in games_hist.columns]
if missing:
    raise RuntimeError(f"games_hist missing required columns: {missing}. Columns are: {list(games_hist.columns)[:25]}")

for c in ["home_score","away_score"]:
    games_hist[c] = pd.to_numeric(games_hist[c], errors="coerce")

games_hist = games_hist.dropna(subset=["home_team","away_team","home_score","away_score"])
games_hist = games_hist.drop_duplicates(subset=["home_team","away_team","home_score","away_score"])

print("✅ Unique historical games stored:", len(games_hist))
display(games_hist.head())

# -------------------------
# TEAM_OVERALL BUILD (points-based, stable)
# -------------------------
POSS_PROXY = 99.0  # stable NBA proxy

home = (games_hist.groupby("home_team")
        .agg(ppg_for=("home_score","mean"),
             ppg_against=("away_score","mean"),
             games=("home_score","size"))
        .reset_index().rename(columns={"home_team":"team"}))

away = (games_hist.groupby("away_team")
        .agg(ppg_for=("away_score","mean"),
             ppg_against=("home_score","mean"),
             games=("away_score","size"))
        .reset_index().rename(columns={"away_team":"team"}))

team_overall = pd.concat([home, away], ignore_index=True)
team_overall = (team_overall.groupby("team", as_index=False)
                .apply(lambda x: pd.Series({
                    "ppg_for": np.average(x["ppg_for"], weights=x["games"]),
                    "ppg_against": np.average(x["ppg_against"], weights=x["games"]),
                    "games": x["games"].sum()
                }))
                .reset_index(drop=True))

team_overall["poss"] = POSS_PROXY
team_overall["ppp_for"] = team_overall["ppg_for"] / POSS_PROXY
team_overall["ppp_against"] = team_overall["ppg_against"] / POSS_PROXY

print("✅ team_overall built:", team_overall.shape)
display(team_overall.sort_values("games", ascending=False).head(12))

✅ daysFrom 3: 19 games
⚠ daysFrom 6 failed — skipping. Reason: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-co
⚠ daysFrom 10 failed — skipping. Reason: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-co
⚠ daysFrom 14 failed — skipping. Reason: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-co
⚠ daysFrom 21 failed — skipping. Reason: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-co
⚠ daysFrom 28 failed — skipping. Reason: {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-co
✅ Unique historical games stored: 19


,home_team,away_team,home_score,away_score,commence_time
0,Charlotte Hornets,Portland Trail Blazers,109.0,93.0,2026-02-28T18:12:00Z
1,Miami Heat,Houston Rockets,115.0,105.0,2026-02-28T20:43:00Z
2,Washington Wizards,Toronto Raptors,125.0,134.0,2026-03-01T00:13:00Z
3,Golden State Warriors,Los Angeles Lakers,101.0,129.0,2026-03-01T01:41:00Z
4,Utah Jazz,New Orleans Pelicans,105.0,115.0,2026-03-01T02:42:00Z


✅ team_overall built: (29, 7)


/tmp/ipykernel_510/1559257144.py:66: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,team,ppg_for,ppg_against,games,poss,ppp_for,ppp_against
1,Boston Celtics,111.0,89.5,2.0,99.0,1.121212,0.904040
10,Houston Rockets,114.0,116.5,2.0,99.0,1.151515,1.176768
7,Denver Nuggets,118.0,121.0,2.0,99.0,1.191919,1.222222
16,Milwaukee Bucks,89.0,114.0,2.0,99.0,0.898990,1.151515
13,Los Angeles Lakers,128.5,102.5,2.0,99.0,1.297980,1.035354
23,Portland Trail Blazers,97.0,122.0,2.0,99.0,0.979798,1.232323
18,New Orleans Pelicans,116.0,121.0,2.0,99.0,1.171717,1.222222
27,Utah Jazz,115.0,121.5,2.0,99.0,1.161616,1.227273
28,Washington Wizards,121.5,128.5,2.0,99.0,1.227273,1.297980
4,Chicago Bulls,120.0,97.0,1.0,99.0,1.212121,0.979798


In [16]:
# =========================
# MERGE PROJECTIONS INTO games_df + REBUILD MEANS
# =========================
g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team","ppg_for","ppg_against","games"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team","ppg_for","ppg_against","games"], errors="ignore")

for col in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
    g[col] = pd.to_numeric(g[col], errors="coerce")
    g[col] = g[col].fillna(g[col].mean())

g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

g["mean_total"]  = (g["home_ppp_proj"] + g["away_ppp_proj"]) * g["poss_proj"]
g["mean_margin"] = (g["home_ppp_proj"] - g["away_ppp_proj"]) * g["poss_proj"]  # + => home favored

games_df = g

print("✅ Projection layer rebuilt")
print("mean_margin range:", float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("mean_total  range:", float(games_df["mean_total"].min()),  "to", float(games_df["mean_total"].max()))
display(games_df[["away_team","home_team","mean_margin","spread_home_line","mean_total","total_line"]].head(10))

✅ Projection layer rebuilt
mean_margin range: -12.944444444444446 to 15.500000000000023
mean_total  range: 194.5 to 233.99999999999997


,away_team,home_team,mean_margin,spread_home_line,mean_total,total_line
0,Dallas Mavericks,Charlotte Hornets,14.500000,-12.5,194.500000,229.5
1,Detroit Pistons,Cleveland Cavaliers,-5.000000,1.0,203.000000,227.5
2,Washington Wizards,Orlando Magic,-3.500000,-15.5,224.000000,227.5
3,Brooklyn Nets,Miami Heat,7.000000,-12.5,214.000000,227.5
4,New York Knicks,Toronto Raptors,-8.000000,1.5,231.000000,223.5
5,Oklahoma City Thunder,Chicago Bulls,5.000000,10.5,202.000000,227.5
6,Memphis Grizzlies,Minnesota Timberwolves,-5.000000,-14.5,228.000000,237.5
7,San Antonio Spurs,Philadelphia 76ers,4.500000,7.5,207.500000,232.5
8,New Orleans Pelicans,Los Angeles Lakers,15.500000,-9.5,234.000000,237.5
9,Phoenix Suns,Sacramento Kings,-12.944444,10.5,221.777778,225.5


In [17]:
# =========================
# MARKET ANCHOR (CONSISTENT)
# =========================
BLEND_MODEL = 0.35
BLEND_MARKET = 0.65

games_df["mean_margin"] = BLEND_MODEL*games_df["mean_margin"] + BLEND_MARKET*(-games_df["spread_home_line"])
games_df["mean_total"]  = BLEND_MODEL*games_df["mean_total"]  + BLEND_MARKET*(games_df["total_line"])

print("✅ Anchor applied | margin range:",
      float(games_df["mean_margin"].min()), "to", float(games_df["mean_margin"].max()))
print("✅ Anchor applied | total range:",
      float(games_df["mean_total"].min()), "to", float(games_df["mean_total"].max()))

✅ Anchor applied | margin range: -11.355555555555556 to 13.199999999999998
✅ Anchor applied | total range: 217.25 to 236.27499999999998


In [20]:
# =========================
# STRICT IDENTICAL-RUN GATE (HARD FAIL IF NOT IDENTICAL)
# =========================

REQUIRED_FUNCS = [
    "pull_market_data",
    "pull_completed_scores",
]

REQUIRED_COLS_AFTER_PROJ = [
    "h_poss","a_poss",
    "h_ppp_for","h_ppp_against",
    "a_ppp_for","a_ppp_against",
    "poss_proj","home_ppp_proj","away_ppp_proj",
    "mean_margin","mean_total"
]

REQUIRED_COLS_AFTER_ENGINE = [
    "p_home_win","p_away_win","p_home_cover","p_over"
]

missing_funcs = [f for f in REQUIRED_FUNCS if f not in globals()]
if missing_funcs:
    raise RuntimeError(f"STOP — notebook missing required functions for IDENTICAL run: {missing_funcs}")

if "games_df" not in globals():
    raise RuntimeError("STOP — games_df not found. Run the market pull cell first.")

# check projection-layer columns
missing_proj = [c for c in REQUIRED_COLS_AFTER_PROJ if c not in games_df.columns]
if missing_proj:
    raise RuntimeError(
        "STOP — projection layer is NOT identical.\n"
        f"Missing columns: {missing_proj}\n"
        "This means we are NOT using the real possessions/PPP layer.\n"
        "Fix by restoring the original team metrics + projection cells from the last good notebook."
    )

# check strict-engine outputs if final_card exists
if "final_card" in globals() and isinstance(final_card, pd.DataFrame) and len(final_card):
    missing_eng = [c for c in REQUIRED_COLS_AFTER_ENGINE if c not in final_card.columns]
    if missing_eng:
        raise RuntimeError(
            "STOP — strict engine outputs missing.\n"
            f"Missing: {missing_eng}\n"
            "Re-run the FULL STRICT engine cell that computes win/cover/over probabilities."
        )

print("✅ IDENTICAL-RUN GATE PASSED — all required layers present.")

RuntimeError: STOP — notebook missing required functions for IDENTICAL run: ['pull_market_data']

In [21]:
# =========================
# RESTORE: pull_market_data (Odds API v4) + ET slate window
# =========================
import os, requests, pandas as pd
from datetime import datetime, timedelta
import pytz

def _american_to_implied(odds):
    try:
        o = float(odds)
    except:
        return None
    if o > 0:
        return 100.0 / (o + 100.0)
    else:
        return (-o) / ((-o) + 100.0)

def pull_market_data(
    sport: str,
    date: str,
    outside: bool = True,
    regions: str = "us",
    markets: str = "h2h,spreads,totals",
    odds_format: str = "american",
    preferred_book: str | None = None,
    tz_str: str = "America/Indiana/Indianapolis",
):
    """
    Pull Odds API markets for a single ET-slate date window [00:00 ET, 24:00 ET).
    Returns a games_df with columns used by our STRICT engine:
    away_team, home_team, commence_time, home_ml, away_ml,
    spread_home_line, spread_home_odds, spread_away_line, spread_away_odds,
    total_line, over_odds, under_odds,
    home_ml_implied, away_ml_implied
    """
    if not outside:
        raise RuntimeError("outside=False not supported for this restored cell. Set OUTSIDE_ON=True.")

    ODDS_API_KEY = os.getenv("ODDS_API_KEY")
    if not ODDS_API_KEY:
        raise ValueError("ODDS_API_KEY not found. Set it in env var first (os.environ['ODDS_API_KEY']=...).")

    # ET slate window → UTC window
    tz = pytz.timezone(tz_str)
    day_et = tz.localize(datetime.strptime(date, "%Y-%m-%d"))
    start_et = day_et
    end_et = day_et + timedelta(days=1)
    start_utc = start_et.astimezone(pytz.utc)
    end_utc = end_et.astimezone(pytz.utc)

    print(f"ET window: {start_et} to {end_et}")
    print(f"UTC window: {start_utc} to {end_utc}")

    url = f"https://api.the-odds-api.com/v4/sports/{sport}/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
        "dateFormat": "iso",
        "commenceTimeFrom": start_utc.isoformat().replace("+00:00","Z"),
        "commenceTimeTo": end_utc.isoformat().replace("+00:00","Z"),
    }

    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise RuntimeError(f"Odds API failed: {r.status_code} | {r.text[:300]}")

    data = r.json()
    rows = []

    def pick_book(bookmakers):
        if not bookmakers:
            return None
        if preferred_book:
            for b in bookmakers:
                if b.get("key") == preferred_book or b.get("title","").lower() == preferred_book.lower():
                    return b
        # default: first bookmaker
        return bookmakers[0]

    for g in data:
        home = g.get("home_team")
        away = g.get("away_team")
        commence = g.get("commence_time")

        bm = pick_book(g.get("bookmakers", []))
        if not bm:
            continue

        # initialize
        home_ml = away_ml = None
        sh_line = sh_odds = sa_line = sa_odds = None
        total = over = under = None

        for m in bm.get("markets", []):
            k = m.get("key")
            outs = m.get("outcomes", [])

            if k == "h2h":
                for o in outs:
                    if o.get("name") == home:
                        home_ml = o.get("price")
                    elif o.get("name") == away:
                        away_ml = o.get("price")

            elif k == "spreads":
                for o in outs:
                    if o.get("name") == home:
                        sh_line = o.get("point")
                        sh_odds = o.get("price")
                    elif o.get("name") == away:
                        sa_line = o.get("point")
                        sa_odds = o.get("price")

            elif k == "totals":
                for o in outs:
                    if o.get("name","").lower() == "over":
                        total = o.get("point")
                        over = o.get("price")
                    elif o.get("name","").lower() == "under":
                        # total point should match
                        under = o.get("price")

        rows.append({
            "away_team": away,
            "home_team": home,
            "commence_time": commence,
            "home_ml": home_ml,
            "away_ml": away_ml,
            "spread_home_line": sh_line,
            "spread_home_odds": sh_odds,
            "spread_away_line": sa_line,
            "spread_away_odds": sa_odds,
            "total_line": total,
            "over_odds": over,
            "under_odds": under,
            "home_ml_implied": _american_to_implied(home_ml) if home_ml is not None else None,
            "away_ml_implied": _american_to_implied(away_ml) if away_ml is not None else None,
        })

    games_df = pd.DataFrame(rows)
    print(f"✅ pull_market_data: pulled {len(games_df)} games for {sport} on {date} (ET window)")

    if len(games_df):
        print("ML non-null:", int(games_df["home_ml"].notna().sum() + games_df["away_ml"].notna().sum())//2, "/", len(games_df))
        print("Spread non-null:", int(games_df["spread_home_line"].notna().sum()), "/", len(games_df))
        print("Total non-null:", int(games_df["total_line"].notna().sum()), "/", len(games_df))

    return games_df

In [22]:
import os
os.environ["ODDS_API_KEY"] = "10ceac94f9b9cb76f8c65510ca6df18f"

SPORT = "basketball_nba"
SLATE_DATE = "2026-03-03"
OUTSIDE_ON = True

In [23]:
games_df = pull_market_data(
    sport=SPORT,
    date=SLATE_DATE,
    outside=OUTSIDE_ON
)

games_df.head()

ET window: 2026-03-03 00:00:00-05:00 to 2026-03-04 00:00:00-05:00
UTC window: 2026-03-03 05:00:00+00:00 to 2026-03-04 05:00:00+00:00
✅ pull_market_data: pulled 10 games for basketball_nba on 2026-03-03 (ET window)
ML non-null: 10 / 10
Spread non-null: 10 / 10
Total non-null: 10 / 10


,away_team,home_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied
0,Dallas Mavericks,Charlotte Hornets,2026-03-04T00:10:00Z,-620,460,-12.5,-112,12.5,-108,229.5,-110,-110,0.861111,0.178571
1,Detroit Pistons,Cleveland Cavaliers,2026-03-04T00:10:00Z,-102,-116,1.0,-114,-1.0,-106,227.5,-106,-114,0.504950,0.537037
2,Washington Wizards,Orlando Magic,2026-03-04T00:10:00Z,-1100,700,-15.5,-110,15.5,-110,227.5,-110,-110,0.916667,0.125000
3,Brooklyn Nets,Miami Heat,2026-03-04T00:40:00Z,-700,500,-12.5,-114,12.5,-106,227.5,-106,-114,0.875000,0.166667
4,New York Knicks,Toronto Raptors,2026-03-04T00:40:00Z,110,-130,1.5,-106,-1.5,-114,223.5,-110,-110,0.476190,0.565217


In [28]:
# =========================
# RESTORE: pull_completed_scores (Odds API v4 scores endpoint)
# =========================
import requests
import pandas as pd

def pull_completed_scores(days_from: int):
    """
    Pull completed NBA scores using Odds API v4.
    Returns a DataFrame with:
    home_team, away_team, home_score, away_score
    """

    ODDS_API_KEY = os.getenv("ODDS_API_KEY")
    if not ODDS_API_KEY:
        raise ValueError("ODDS_API_KEY not set.")

    url = "https://api.the-odds-api.com/v4/sports/basketball_nba/scores"

    params = {
        "apiKey": ODDS_API_KEY,
        "daysFrom": days_from
    }

    r = requests.get(url, params=params, timeout=45)
    if r.status_code != 200:
        raise RuntimeError(f"Scores API failed: {r.status_code} | {r.text[:200]}")

    data = r.json()

    rows = []
    for g in data:
        if not g.get("completed"):
            continue

        scores = g.get("scores", [])
        if len(scores) != 2:
            continue

        rows.append({
            "home_team": g.get("home_team"),
            "away_team": g.get("away_team"),
            "home_score": scores[0].get("score") if scores[0].get("name") == g.get("home_team") else scores[1].get("score"),
            "away_score": scores[1].get("score") if scores[1].get("name") == g.get("away_team") else scores[0].get("score"),
        })

    df = pd.DataFrame(rows)
    print(f"✅ pull_completed_scores({days_from}) → {len(df)} games")
    return df

In [29]:
# =========================
# REBUILD HISTORICAL DATA CLEAN
# =========================

parts = []
for d in [3, 6, 10, 14, 21, 28]:
    try:
        part = pull_completed_scores(d)
        parts.append(part)
    except Exception as e:
        print(f"⚠ daysFrom {d} failed:", e)

if not parts:
    raise RuntimeError("No historical score pulls succeeded.")

games_hist = pd.concat(parts, ignore_index=True).drop_duplicates()

print("✅ Historical games rebuilt:", len(games_hist))
games_hist.head()

✅ pull_completed_scores(3) → 19 games
⚠ daysFrom 6 failed: Scores API failed: 422 | {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠ daysFrom 10 failed: Scores API failed: 422 | {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠ daysFrom 14 failed: Scores API failed: 422 | {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠ daysFrom 21 failed: Scores API failed: 422 | {"message":"Invalid daysFrom","error_code":"INVALID_SCORES_DAYS_FROM","details_url":"https://the-odds-api.com/liveapi/guides/v4/api-error-codes.html#invalid-scores-days-from"}

⚠ daysFrom 28 failed: Scores API failed: 422 | {"message":"Invalid 

,home_team,away_team,home_score,away_score
0,Charlotte Hornets,Portland Trail Blazers,109,93
1,Miami Heat,Houston Rockets,115,105
2,Washington Wizards,Toronto Raptors,125,134
3,Golden State Warriors,Los Angeles Lakers,101,129
4,Utah Jazz,New Orleans Pelicans,105,115


In [30]:
# =========================
# PERMANENT FIX: historical rebuild (auto daysFrom cap) + team_overall + merge into games_df
# =========================
import numpy as np
import pandas as pd

def _compute_possessions(df, home_prefix="home_", away_prefix="away_"):
    """
    If we ever have boxscore components, use this.
    For OddsAPI scores-only, we won't. Placeholder kept for compatibility.
    """
    return None

def build_team_overall_from_scores(games_hist: pd.DataFrame):
    """
    Scores-only fallback: estimate team strength from scoring margins + totals.
    This matches what we did in the last NBA stable run when boxscore stats weren't available.
    Produces: team, poss, ppp_for, ppp_against
    """
    gh = games_hist.copy()

    # basic derived signals
    gh["home_margin"] = gh["home_score"] - gh["away_score"]
    gh["away_margin"] = -gh["home_margin"]
    gh["game_total"] = gh["home_score"] + gh["away_score"]

    # approximate possessions proxy (stable): use total points scaled
    # (we only need a consistent poss signal for the projection layer; sims handle variance)
    # clamp to reasonable NBA range
    gh["poss_proxy"] = (gh["game_total"] / 2.25).clip(90, 105)

    # home rows
    home_rows = pd.DataFrame({
        "team": gh["home_team"],
        "ppp_for": gh["home_score"] / gh["poss_proxy"],
        "ppp_against": gh["away_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })

    # away rows
    away_rows = pd.DataFrame({
        "team": gh["away_team"],
        "ppp_for": gh["away_score"] / gh["poss_proxy"],
        "ppp_against": gh["home_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })

    team_game = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = (
        team_game.groupby("team", as_index=False)
        .agg(poss=("poss","mean"), ppp_for=("ppp_for","mean"), ppp_against=("ppp_against","mean"))
    )

    print("✅ team_overall built:", team_overall.shape)
    return team_overall

def rebuild_hist_and_merge_into_games_df(days_try=(3,2,1)):
    # 1) Find max valid daysFrom
    best = None
    for d in days_try:
        try:
            df = pull_completed_scores(d)
            if len(df) >= 10:
                best = d
                break
        except Exception as e:
            print(f"⚠ daysFrom {d} failed:", e)

    if best is None:
        raise RuntimeError("No valid daysFrom found for scores endpoint.")

    print(f"✅ Using daysFrom={best} for historical rebuild (plan constraint safe).")

    # 2) Build historical sample
    games_hist = pull_completed_scores(best).drop_duplicates()
    if len(games_hist) < 10:
        raise RuntimeError(f"Not enough historical games ({len(games_hist)}) to build team_overall reliably.")

    # 3) Build team_overall (scores-only consistent layer)
    team_overall = build_team_overall_from_scores(games_hist)

    # 4) Merge into current slate games_df to produce STRICT layer columns
    if "games_df" not in globals():
        raise RuntimeError("games_df not found. Run the market pull cell first.")

    g = games_df.copy()

    # Merge home
    g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
        "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    # Merge away
    g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
        "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
    }).drop(columns=["team"], errors="ignore")

    # Fill any missing with league means (prevents NaNs breaking sims)
    for c in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
        if c in g.columns:
            g[c] = g[c].fillna(g[c].mean())

    # Write back
    globals()["games_hist"] = games_hist
    globals()["team_overall"] = team_overall
    globals()["games_df"] = g

    print("✅ Historical + team_overall merged into games_df.")
    print("Null check:",
          g[["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]].isna().sum().to_dict())
    return g

# Run it
games_df = rebuild_hist_and_merge_into_games_df()
games_df.head()

✅ pull_completed_scores(3) → 19 games
✅ Using daysFrom=3 for historical rebuild (plan constraint safe).
✅ pull_completed_scores(3) → 19 games


TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [31]:
# =========================
# HOTFIX: coerce OddsAPI scores to numeric + rebuild team_overall merge
# =========================
import pandas as pd
import numpy as np

def build_team_overall_from_scores(games_hist: pd.DataFrame):
    gh = games_hist.copy()

    # coerce scores to numeric (fix for 'str' subtraction error)
    gh["home_score"] = pd.to_numeric(gh["home_score"], errors="coerce")
    gh["away_score"] = pd.to_numeric(gh["away_score"], errors="coerce")

    gh = gh.dropna(subset=["home_score","away_score"]).copy()
    if gh.empty:
        raise RuntimeError("After coercion, no valid score rows remain.")

    gh["home_margin"] = gh["home_score"] - gh["away_score"]
    gh["away_margin"] = -gh["home_margin"]
    gh["game_total"] = gh["home_score"] + gh["away_score"]

    # possessions proxy (stable + bounded)
    gh["poss_proxy"] = (gh["game_total"] / 2.25).clip(90, 105)

    home_rows = pd.DataFrame({
        "team": gh["home_team"],
        "ppp_for": gh["home_score"] / gh["poss_proxy"],
        "ppp_against": gh["away_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })
    away_rows = pd.DataFrame({
        "team": gh["away_team"],
        "ppp_for": gh["away_score"] / gh["poss_proxy"],
        "ppp_against": gh["home_score"] / gh["poss_proxy"],
        "poss": gh["poss_proxy"]
    })

    team_game = pd.concat([home_rows, away_rows], ignore_index=True)

    team_overall = (
        team_game.groupby("team", as_index=False)
        .agg(poss=("poss","mean"), ppp_for=("ppp_for","mean"), ppp_against=("ppp_against","mean"))
    )
    print("✅ team_overall rebuilt (scores coerced):", team_overall.shape)
    return team_overall

# Re-run rebuild using your existing games_hist pulled already
if "games_hist" not in globals():
    raise RuntimeError("games_hist not found. Run the historical pull cell first.")

team_overall = build_team_overall_from_scores(games_hist)

g = games_df.copy()

g = g.merge(team_overall, left_on="home_team", right_on="team", how="left").rename(columns={
    "poss":"h_poss","ppp_for":"h_ppp_for","ppp_against":"h_ppp_against"
}).drop(columns=["team"], errors="ignore")

g = g.merge(team_overall, left_on="away_team", right_on="team", how="left").rename(columns={
    "poss":"a_poss","ppp_for":"a_ppp_for","ppp_against":"a_ppp_against"
}).drop(columns=["team"], errors="ignore")

for c in ["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]:
    g[c] = g[c].fillna(g[c].mean())

games_df = g

print("✅ Merged into games_df. Null check:",
      games_df[["h_poss","a_poss","h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against"]].isna().sum().to_dict())

games_df.head()

✅ team_overall rebuilt (scores coerced): (29, 4)
✅ Merged into games_df. Null check: {'h_poss': 0, 'a_poss': 0, 'h_ppp_for': 0, 'h_ppp_against': 0, 'a_ppp_for': 0, 'a_ppp_against': 0}


,away_team,home_team,commence_time,home_ml,away_ml,spread_home_line,spread_home_odds,spread_away_line,spread_away_odds,total_line,over_odds,under_odds,home_ml_implied,away_ml_implied,h_poss,h_ppp_for,h_ppp_against,a_poss,a_ppp_for,a_ppp_against
0,Dallas Mavericks,Charlotte Hornets,2026-03-04T00:10:00Z,-620,460,-12.5,-112,12.5,-108,229.5,-110,-110,0.861111,0.178571,90.000000,1.211111,1.033333,90.000000,0.966667,1.111111
1,Detroit Pistons,Cleveland Cavaliers,2026-03-04T00:10:00Z,-102,-116,1.0,-114,-1.0,-106,227.5,-106,-114,0.504950,0.537037,92.444444,1.146635,1.103365,90.000000,1.177778,1.022222
2,Washington Wizards,Orlando Magic,2026-03-04T00:10:00Z,-1100,700,-15.5,-110,15.5,-110,227.5,-110,-110,0.916667,0.125000,90.000000,1.022222,1.177778,105.000000,1.157143,1.223810
3,Brooklyn Nets,Miami Heat,2026-03-04T00:40:00Z,-700,500,-12.5,-114,12.5,-106,227.5,-106,-114,0.875000,0.166667,97.777778,1.176136,1.073864,92.444444,1.103365,1.146635
4,New York Knicks,Toronto Raptors,2026-03-04T00:40:00Z,110,-130,1.5,-106,-1.5,-114,223.5,-110,-110,0.476190,0.565217,105.000000,1.276190,1.190476,90.222222,1.263547,0.986453


In [32]:
# ==============================
# NBA FULL STRICT ENGINE (BOTTOM CELL)
# - No hunting required
# - Builds all STRICT probability columns
# - Runs Monte Carlo sims (ML/Spread/Total)
# - Caps correlated exposure (max 2 picks per game)
# ==============================
import numpy as np
import pandas as pd
from math import erf, sqrt
from datetime import datetime

# ---------- SETTINGS (edit if needed) ----------
SIMS = int(globals().get("SIMS", 50000))
SD_MARGIN = float(globals().get("SD_MARGIN", 14.5))   # spread outcome volatility
SD_TOTAL  = float(globals().get("SD_TOTAL", 18.0))    # total outcome volatility

MIN_U = 0.25
MAX_U = 1.00
HALF_KELLY = True
MAX_PICKS_PER_GAME = 2

SLATE_DATE = globals().get("SLATE_DATE", None) or globals().get("TARGET_DATE", None) or "slate"
EXPORT_NAME = f"nba_stable_{SLATE_DATE}_FULL_STRICT_CARD_BOTTOM.csv"

# ---------- HELPERS ----------
def american_to_decimal(odds):
    odds = float(odds)
    if odds > 0:
        return 1.0 + (odds / 100.0)
    else:
        return 1.0 + (100.0 / abs(odds))

def american_to_implied_prob(odds):
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return abs(odds) / (abs(odds) + 100.0)

def norm_cdf(x):
    # standard normal CDF using erf
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def kelly_fraction(p, dec_odds):
    # f* = (bp - q)/b where b = dec-1, q=1-p
    b = dec_odds - 1.0
    q = 1.0 - p
    f = (b * p - q) / b if b > 0 else 0.0
    return max(0.0, f)

def clip_units(u):
    if u <= 0:
        return 0.0
    return float(np.clip(u, MIN_U, MAX_U))

def _pick_sign_fix(row):
    """
    Ensure spread bet text + probability mapping is consistent with spread_home_line meaning.
    We treat spread_home_line as: home spread (e.g. -2 means home favored by 2).
    """
    return row

# ---------- INPUT CHECK ----------
if "games_df" not in globals():
    raise RuntimeError("games_df not found. Run the market pull / ET slate lock first.")

g = games_df.copy()

need_cols = ["home_team","away_team","spread_home_line","total_line"]
missing = [c for c in need_cols if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required columns: {missing}")

# if commence time exists, keep it
if "commence_et" in g.columns:
    g["commence_time"] = g["commence_et"].astype(str)
elif "commence_utc" in g.columns:
    g["commence_time"] = g["commence_utc"].astype(str)
else:
    g["commence_time"] = ""

# ---------- PROJECTION LAYER ----------
# If your notebook already has model projections (mean_margin / mean_total), use them.
# Otherwise default to market anchors (spread/total) as the mean.
if "mean_margin" not in g.columns:
    # mean_margin is HOME margin (home_score - away_score)
    g["mean_margin"] = -g["spread_home_line"].astype(float)

if "mean_total" not in g.columns:
    g["mean_total"] = g["total_line"].astype(float)

# ---------- MONTE CARLO SIMS ----------
rng = np.random.default_rng(7)

# simulate home margin and total per game
n = len(g)
margins = rng.normal(loc=g["mean_margin"].to_numpy(), scale=SD_MARGIN, size=(SIMS, n))
totals  = rng.normal(loc=g["mean_total"].to_numpy(),  scale=SD_TOTAL,  size=(SIMS, n))

# win probs
p_home_win = (margins > 0).mean(axis=0)
p_away_win = 1.0 - p_home_win

g["p_home_win"] = p_home_win
g["p_away_win"] = p_away_win

# spread cover probs:
# Home covers if (home_margin - spread_home_line) > 0
spread_home = g["spread_home_line"].to_numpy(dtype=float)
p_home_cover = ((margins - spread_home) > 0).mean(axis=0)
p_away_cover = 1.0 - p_home_cover
g["p_home_cover"] = p_home_cover
g["p_away_cover"] = p_away_cover

# total probs
total_line = g["total_line"].to_numpy(dtype=float)
p_over = (totals > total_line).mean(axis=0)
p_under = 1.0 - p_over
g["p_over"] = p_over
g["p_under"] = p_under

# ---------- BUILD MARKET ROWS (ML / SPREAD / TOTAL) ----------
rows = []

# ML rows (if ml exists; otherwise skip)
has_ml = ("home_ml" in g.columns) and ("away_ml" in g.columns)
if has_ml:
    for i, r in g.iterrows():
        # HOME ML
        if pd.notna(r.get("home_ml")):
            odds = float(r["home_ml"])
            rows.append({
                "matchup": f'{r["away_team"]} at {r["home_team"]}',
                "game_key": f'{r["away_team"]} @ {r["home_team"]}',
                "market": "ML",
                "team": r["home_team"],
                "bet": f'{r["home_team"]} ML ({int(odds) if odds.is_integer() else odds})',
                "odds": odds,
                "p_win": float(r["p_home_win"]),
                "p_mkt": float(american_to_implied_prob(odds)),
                "commence_time": r["commence_time"],
            })
        # AWAY ML
        if pd.notna(r.get("away_ml")):
            odds = float(r["away_ml"])
            rows.append({
                "matchup": f'{r["away_team"]} at {r["home_team"]}',
                "game_key": f'{r["away_team"]} @ {r["home_team"]}',
                "market": "ML",
                "team": r["away_team"],
                "bet": f'{r["away_team"]} ML ({int(odds) if odds.is_integer() else odds})',
                "odds": odds,
                "p_win": float(r["p_away_win"]),
                "p_mkt": float(american_to_implied_prob(odds)),
                "commence_time": r["commence_time"],
            })

# SPREAD rows (use spread_home_line + both prices if present, else assume -110)
for i, r in g.iterrows():
    sh = float(r["spread_home_line"])
    home_odds = float(r["spread_home_odds"]) if "spread_home_odds" in g.columns and pd.notna(r.get("spread_home_odds")) else -110.0
    away_odds = float(r["spread_away_odds"]) if "spread_away_odds" in g.columns and pd.notna(r.get("spread_away_odds")) else -110.0

    # Home spread bet (e.g. -2 means home -2)
    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "SPREAD",
        "team": r["home_team"],
        "bet": f'{r["home_team"]} {sh:+g} ({int(home_odds)})'.replace("+", ""),
        "odds": home_odds,
        "p_win": float(r["p_home_cover"]),
        "p_mkt": float(american_to_implied_prob(home_odds)),
        "commence_time": r["commence_time"],
    })

    # Away spread is the opposite sign
    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "SPREAD",
        "team": r["away_team"],
        "bet": f'{r["away_team"]} {(-sh):+g} ({int(away_odds)})'.replace("+", ""),
        "odds": away_odds,
        "p_win": float(r["p_away_cover"]),
        "p_mkt": float(american_to_implied_prob(away_odds)),
        "commence_time": r["commence_time"],
    })

# TOTAL rows (assume over_odds/under_odds else -110)
for i, r in g.iterrows():
    tl = float(r["total_line"])
    over_odds  = float(r["over_odds"])  if "over_odds"  in g.columns and pd.notna(r.get("over_odds"))  else -110.0
    under_odds = float(r["under_odds"]) if "under_odds" in g.columns and pd.notna(r.get("under_odds")) else -110.0

    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "TOTAL",
        "team": "OVER",
        "bet": f'OVER {tl:g} ({int(over_odds)})',
        "odds": over_odds,
        "p_win": float(r["p_over"]),
        "p_mkt": float(american_to_implied_prob(over_odds)),
        "commence_time": r["commence_time"],
    })

    rows.append({
        "matchup": f'{r["away_team"]} at {r["home_team"]}',
        "game_key": f'{r["away_team"]} @ {r["home_team"]}',
        "market": "TOTAL",
        "team": "UNDER",
        "bet": f'UNDER {tl:g} ({int(under_odds)})',
        "odds": under_odds,
        "p_win": float(r["p_under"]),
        "p_mkt": float(american_to_implied_prob(under_odds)),
        "commence_time": r["commence_time"],
    })

card = pd.DataFrame(rows)

# ---------- EV + KELLY UNITS ----------
card["dec_odds"] = card["odds"].astype(float).apply(american_to_decimal)
# Expected value as ROI: EV = p*(dec-1) - (1-p)
card["ev"] = card["p_win"] * (card["dec_odds"] - 1.0) - (1.0 - card["p_win"])

# Kelly sizing (half Kelly default)
card["kelly_f"] = card.apply(lambda x: kelly_fraction(x["p_win"], x["dec_odds"]), axis=1)
if HALF_KELLY:
    card["kelly_f"] = 0.5 * card["kelly_f"]

# Map Kelly fraction to "units" (1.0u = 5% bankroll baseline)
# This keeps sizing consistent across slates.
BANKROLL_UNIT = 0.05
card["units_raw"] = card["kelly_f"] / BANKROLL_UNIT
card["units"] = card["units_raw"].apply(clip_units)

# ---------- FILTER POSTABLE (+EV ONLY) ----------
postable = card[(card["ev"] > 0) & (card["units"] > 0)].copy()

# ---------- CORRELATED EXPOSURE CAP (MAX 2 PICKS PER GAME) ----------
# Rank by a blend: EV then win prob (so we keep strong +EV but not only longshots)
postable["rank_score"] = (postable["ev"] * 0.65) + (postable["p_win"] * 0.35)
postable = postable.sort_values("rank_score", ascending=False)

kept = []
counts = {}
for _, r in postable.iterrows():
    k = r["game_key"]
    counts.setdefault(k, 0)
    if counts[k] >= MAX_PICKS_PER_GAME:
        continue
    kept.append(r)
    counts[k] += 1

final_card = pd.DataFrame(kept).copy()
final_card = final_card.sort_values(["rank_score"], ascending=False)

# ---------- DISCORD TEXT ----------
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_ev(x):  return f"{100*x:.1f}%"

final_card["discord_text"] = final_card.apply(
    lambda r: (
        f'{r["matchup"]}\n'
        f'{r["bet"]} — {r["units"]:.2f}u\n'
        f'Sim Win%: {fmt_pct(r["p_win"])} | Market: {fmt_pct(r["p_mkt"])}\n'
        f'EV: {fmt_ev(r["ev"])}\n'
    ),
    axis=1
)

# ---------- EXPORT ----------
final_card.to_csv(EXPORT_NAME, index=False)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print(f"✅ Final postable plays: {len(final_card)} (cap {MAX_PICKS_PER_GAME} per game)")
print(f"✅ Exported: {EXPORT_NAME}\n")

print("=== DISCORD CARD ===\n")
print("\n".join(final_card["discord_text"].tolist()))

# Keep globals for downstream (dogs, top10, etc.)
globals()["final_card"] = final_card
globals()["strict_probs_df"] = g

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Final postable plays: 17 (cap 2 per game)
✅ Exported: nba_stable_2026-03-03_FULL_STRICT_CARD_BOTTOM.csv

=== DISCORD CARD ===

Washington Wizards at Orlando Magic
Orlando Magic -15.5 (-110) — 1.00u
Sim Win%: 98.4% | Market: 52.4%
EV: 87.9%

Memphis Grizzlies at Minnesota Timberwolves
Minnesota Timberwolves -14.5 (-110) — 1.00u
Sim Win%: 97.8% | Market: 52.4%
EV: 86.7%

Dallas Mavericks at Charlotte Hornets
Charlotte Hornets -12.5 (-112) — 1.00u
Sim Win%: 95.8% | Market: 52.8%
EV: 81.3%

Brooklyn Nets at Miami Heat
Miami Heat -12.5 (-114) — 1.00u
Sim Win%: 95.7% | Market: 53.3%
EV: 79.7%

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 1.00u
Sim Win%: 92.9% | Market: 52.4%
EV: 77.3%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 1.00u
Sim Win%: 92.5% | Market: 52.4%
EV: 76.6%

New Orleans Pelicans at Los Angeles Lakers
Los Angeles Lakers -9.5 (-108) — 1.00u
Sim Win%: 90.5% | Market: 51.9%
EV: 7

In [33]:
# ==============================
# STRICT MEAN RESET + MARKET ANCHOR (MATCH 3/2 BEHAVIOR)
# ==============================

if "games_df" not in globals():
    raise RuntimeError("games_df not found.")

g = games_df.copy()

# Base projection from team strength if available
if all(c in g.columns for c in ["h_ppp_for","h_ppp_against","a_ppp_for","a_ppp_against","h_poss","a_poss"]):
    g["poss_proj"] = (g["h_poss"] + g["a_poss"]) / 2
    g["home_ppp_proj"] = (g["h_ppp_for"] + g["a_ppp_against"]) / 2
    g["away_ppp_proj"] = (g["a_ppp_for"] + g["h_ppp_against"]) / 2

    g["model_margin_raw"] = (g["home_ppp_proj"] - g["away_ppp_proj"]) * g["poss_proj"]
    g["model_total_raw"] = (g["home_ppp_proj"] + g["away_ppp_proj"]) * g["poss_proj"]
else:
    raise RuntimeError("PPP / possession layer missing.")

# MARKET ANCHOR BLEND (same philosophy as 3/2)
ANCHOR_WEIGHT = 0.65  # heavier anchor prevents crazy edges

g["mean_margin"] = (
    ANCHOR_WEIGHT * (-g["spread_home_line"]) +
    (1 - ANCHOR_WEIGHT) * g["model_margin_raw"]
)

g["mean_total"] = (
    ANCHOR_WEIGHT * g["total_line"] +
    (1 - ANCHOR_WEIGHT) * g["model_total_raw"]
)

print("✅ Stronger market anchor applied")
print("New margin range:", g["mean_margin"].min(), "to", g["mean_margin"].max())

games_df = g

✅ Stronger market anchor applied
New margin range: -11.211649222883683 to 13.199999999999996


In [40]:
# ==========================================
# STRONG MARKET ANCHOR (IDENTICAL STYLE FIX)
# ==========================================

MARKET_WEIGHT = 0.65  # heavier anchor
MODEL_WEIGHT  = 0.35

games_df["anchored_margin"] = (
    MODEL_WEIGHT  * games_df["mean_margin"] +
    MARKET_WEIGHT * (-games_df["spread_home_line"])
)

games_df["anchored_total"] = (
    MODEL_WEIGHT  * games_df["mean_total"] +
    MARKET_WEIGHT * games_df["total_line"]
)

# Replace original mean inputs
games_df["mean_margin"] = games_df["anchored_margin"]
games_df["mean_total"]  = games_df["anchored_total"]

print("✅ Strong market gravity applied.")
print("New margin range:",
      games_df["mean_margin"].min(),
      "to",
      games_df["mean_margin"].max())

✅ Strong market gravity applied.
New margin range: -10.67791230572092 to 13.764583333333336


In [41]:
# =========================
# FULL STRICT ENGINE (BOTTOM RUNNER)
# =========================
import numpy as np
import pandas as pd

SIMS = 50000
SD_MARGIN = 14.5
SD_TOTAL  = 18.0
MAX_PLAYS_PER_GAME = 2
MIN_UNIT = 0.25
MAX_UNIT = 1.00

def american_to_prob(odds):
    o = float(odds)
    if o == 0 or np.isnan(o): return np.nan
    if o > 0:  return 100.0 / (o + 100.0)
    return (-o) / ((-o) + 100.0)

def prob_to_ev(p, odds):
    o = float(odds)
    if np.isnan(p) or np.isnan(o): return np.nan
    if o > 0:
        b = o/100.0
    else:
        b = 100.0/(-o)
    return p*b - (1-p)

def half_kelly_units(p, odds):
    o = float(odds)
    if np.isnan(p) or np.isnan(o): return MIN_UNIT
    if o > 0:
        b = o/100.0
    else:
        b = 100.0/(-o)
    q = 1-p
    f = (b*p - q)/b
    f = max(0.0, f) * 0.5  # half Kelly
    # map to 0.25–1.0u
    u = MIN_UNIT + f*(MAX_UNIT-MIN_UNIT)
    return float(min(MAX_UNIT, max(MIN_UNIT, u)))

g = games_df.copy()

need_cols = ["away_team","home_team","mean_margin","mean_total","spread_home_line","total_line"]
missing = [c for c in need_cols if c not in g.columns]
if missing:
    raise RuntimeError(f"games_df missing required cols: {missing}")

# sims
rng = np.random.default_rng(7)
margins = rng.normal(loc=g["mean_margin"].values[:,None], scale=SD_MARGIN, size=(len(g), SIMS))
totals  = rng.normal(loc=g["mean_total"].values[:,None],  scale=SD_TOTAL,  size=(len(g), SIMS))

# probabilities
g["p_home_win"]   = (margins > 0).mean(axis=1)
g["p_away_win"]   = 1 - g["p_home_win"]
g["p_home_cover"] = (margins > (-g["spread_home_line"].values[:,None])).mean(axis=1)  # home cover
g["p_away_cover"] = 1 - g["p_home_cover"]
g["p_over"]       = (totals > (g["total_line"].values[:,None])).mean(axis=1)
g["p_under"]      = 1 - g["p_over"]

# build candidates (ML / Spread / Total)
rows = []

for i, r in g.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # ML (choose side with odds available)
    if "home_ml" in g.columns and pd.notna(r.get("home_ml")):
        p = r["p_home_win"]
        odds = r["home_ml"]
        rows.append((matchup,"ML",f"{r['home_team']} ML ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["home_team"]))
    if "away_ml" in g.columns and pd.notna(r.get("away_ml")):
        p = r["p_away_win"]
        odds = r["away_ml"]
        rows.append((matchup,"ML",f"{r['away_team']} ML ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["away_team"]))

    # Spread (home line, away is opposite)
    if "spread_home_odds" in g.columns and pd.notna(r.get("spread_home_odds")):
        odds = r["spread_home_odds"]
        p = r["p_home_cover"]
        rows.append((matchup,"SPREAD",f"{r['home_team']} {r['spread_home_line']} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["home_team"]))
    if "spread_away_odds" in g.columns and pd.notna(r.get("spread_away_odds")):
        odds = r["spread_away_odds"]
        p = r["p_away_cover"]
        away_line = -r["spread_home_line"]
        rows.append((matchup,"SPREAD",f"{r['away_team']} {away_line} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,r["away_team"]))

    # Totals
    if "over_odds" in g.columns and pd.notna(r.get("over_odds")):
        odds = r["over_odds"]
        p = r["p_over"]
        rows.append((matchup,"TOTAL",f"OVER {r['total_line']} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,"OVER"))
    if "under_odds" in g.columns and pd.notna(r.get("under_odds")):
        odds = r["under_odds"]
        p = r["p_under"]
        rows.append((matchup,"TOTAL",f"UNDER {r['total_line']} ({int(odds)})",p,american_to_prob(odds),prob_to_ev(p,odds),odds,"UNDER"))

card = pd.DataFrame(rows, columns=["matchup","market","bet","p_win","p_mkt","ev","odds","team"])
card = card.dropna(subset=["p_win","odds","ev"]).copy()

# +EV only
card = card[card["ev"] > 0].copy()

# units
card["units"] = card.apply(lambda x: half_kelly_units(x["p_win"], x["odds"]), axis=1)

# cap 2 per game
card = card.sort_values(["matchup","ev"], ascending=[True, False])
card = card.groupby("matchup").head(MAX_PLAYS_PER_GAME).copy()
card = card.sort_values("ev", ascending=False).reset_index(drop=True)

# discord text
def fmt_pct(x): return f"{100*x:.1f}%"
card["discord_text"] = card.apply(
    lambda x: f"{x['matchup']}\n{x['bet']} — {x['units']:.2f}u\nSim Win%: {fmt_pct(x['p_win'])} | Market: {fmt_pct(x['p_mkt'])}\nEV: {100*x['ev']:.1f}%\n",
    axis=1
)

print(f"✅ Sims: {SIMS} | SD_MARGIN: {SD_MARGIN} | SD_TOTAL: {SD_TOTAL}")
print(f"✅ Final postable plays: {len(card)} (cap {MAX_PLAYS_PER_GAME} per game)")

# export
out_path = f"nba_stable_2026-03-03_FULL_STRICT_CARD_BOTTOM.csv"
card.to_csv(out_path, index=False)
print("✅ Exported:", out_path)

print("\n=== DISCORD CARD ===\n")
print("\n".join(card["discord_text"].tolist()))

✅ Sims: 50000 | SD_MARGIN: 14.5 | SD_TOTAL: 18.0
✅ Final postable plays: 14 (cap 2 per game)
✅ Exported: nba_stable_2026-03-03_FULL_STRICT_CARD_BOTTOM.csv

=== DISCORD CARD ===

Washington Wizards at Orlando Magic
Washington Wizards ML (700) — 0.27u
Sim Win%: 17.1% | Market: 12.5%
EV: 37.1%

Memphis Grizzlies at Minnesota Timberwolves
Memphis Grizzlies ML (610) — 0.27u
Sim Win%: 18.7% | Market: 14.1%
EV: 32.7%

Brooklyn Nets at Miami Heat
Brooklyn Nets ML (500) — 0.27u
Sim Win%: 20.2% | Market: 16.7%
EV: 21.2%

Oklahoma City Thunder at Chicago Bulls
Chicago Bulls ML (350) — 0.27u
Sim Win%: 26.1% | Market: 22.2%
EV: 17.4%

San Antonio Spurs at Philadelphia 76ers
Philadelphia 76ers ML (235) — 0.27u
Sim Win%: 32.9% | Market: 29.9%
EV: 10.3%

Phoenix Suns at Sacramento Kings
Sacramento Kings ML (370) — 0.26u
Sim Win%: 23.1% | Market: 21.3%
EV: 8.6%

Dallas Mavericks at Charlotte Hornets
UNDER 229.5 (-110) — 0.28u
Sim Win%: 56.3% | Market: 52.4%
EV: 7.5%

Dallas Mavericks at Charlotte Horne

In [42]:
# ==============================
# STRICT STABILIZER PATCH
# ==============================

# 1) Increase anchor weight
ANCHOR_WEIGHT = 0.75

g = games_df.copy()

g["mean_margin"] = (
    ANCHOR_WEIGHT * (-g["spread_home_line"]) +
    (1 - ANCHOR_WEIGHT) * g["model_margin_raw"]
)

g["mean_total"] = (
    ANCHOR_WEIGHT * g["total_line"] +
    (1 - ANCHOR_WEIGHT) * g["model_total_raw"]
)

games_df = g

# 2) Raise variance slightly
SD_MARGIN = 15.5
SD_TOTAL = 18.5

print("✅ Stabilizer applied")
print("New margin range:", g["mean_margin"].min(), "to", g["mean_margin"].max())

✅ Stabilizer applied
New margin range: -11.008320873488344 to 12.999999999999996


In [43]:
# ==========================================
# BLENDED RANKING: +EV only, favor high p_win
# ==========================================

import numpy as np
import pandas as pd

# expects a dataframe named final_card with at least:
# matchup, bet, market, p_win, p_mkt, ev, units, discord_text
df = final_card.copy()

need = ["matchup","bet","market","p_win","p_mkt","ev","units","discord_text"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f"final_card missing columns: {missing}")

# 1) +EV filter (hard gate)
df = df[df["ev"] > 0].copy()

# 2) Remove fake longshot spikes if you want (recommended)
# - eliminates super low win% dogs that look +EV from noise
df = df[(df["p_win"] >= 0.08) & (df["p_win"] <= 0.80)].copy()

# 3) Normalize EV to 0..1 (clip so outliers don't dominate)
ev_clip = df["ev"].clip(lower=0, upper=df["ev"].quantile(0.90))
df["ev_norm"] = (ev_clip - ev_clip.min()) / (ev_clip.max() - ev_clip.min() + 1e-9)

# 4) Normalize win prob to 0..1
df["pwin_norm"] = (df["p_win"] - df["p_win"].min()) / (df["p_win"].max() - df["p_win"].min() + 1e-9)

# 5) Blend weights (tune these)
W_PWIN = 0.65   # “most probable” priority
W_EV   = 0.35   # still rewards +EV

df["blend_score"] = W_PWIN * df["pwin_norm"] + W_EV * df["ev_norm"]

# 6) Correlation cap: max 2 picks per game
df["game_key"] = df["matchup"].astype(str)

df = df.sort_values("blend_score", ascending=False)
kept = []
counts = {}

for _, r in df.iterrows():
    k = r["game_key"]
    if counts.get(k, 0) >= 2:
        continue
    kept.append(r)
    counts[k] = counts.get(k, 0) + 1

df_blend = pd.DataFrame(kept).reset_index(drop=True)

print(f"✅ +EV plays after blend rank + cap: {len(df_blend)}")
df_blend[["matchup","bet","p_win","ev","units","blend_score"]].head(20)

✅ +EV plays after blend rank + cap: 9


,matchup,bet,p_win,ev,units,blend_score
0,New York Knicks at Toronto Raptors,New York Knicks -1.5 (-114),0.58260,0.093653,1.000000,0.878719
1,Detroit Pistons at Cleveland Cavaliers,Detroit Pistons -1 (-106),0.55254,0.073804,0.782324,0.773029
2,Brooklyn Nets at Miami Heat,Brooklyn Nets ML (500),0.19400,0.164000,0.328000,0.429871
3,Memphis Grizzlies at Minnesota Timberwolves,Memphis Grizzlies ML (610),0.15844,0.124924,0.250000,0.353450
4,Phoenix Suns at Sacramento Kings,Sacramento Kings ML (370),0.23026,0.082222,0.250000,0.326320
5,Washington Wizards at Orlando Magic,Washington Wizards ML (700),0.13956,0.116480,0.250000,0.299550
6,Dallas Mavericks at Charlotte Hornets,Dallas Mavericks ML (460),0.19404,0.086624,0.250000,0.286839
7,Oklahoma City Thunder at Chicago Bulls,Chicago Bulls ML (350),0.23686,0.065870,0.250000,0.285265
8,San Antonio Spurs at Philadelphia 76ers,Philadelphia 76ers ML (235),0.30446,0.019941,0.250000,0.241931


In [44]:
# =========================
# DISCORD CARD (BLENDED TOP)
# =========================

TOP_N = 10  # set 8 / 10 / 12

out = df_blend.head(TOP_N).copy()

print("=== DISCORD CARD (BLENDED: +EV + PROBABLE) ===\n")
print("\n".join(out["discord_text"].astype(str).tolist()))

# optional export
fname = f"nba_stable_{SLATE_DATE}_BLENDED_TOP{TOP_N}.csv"
out.to_csv(fname, index=False)
print(f"\n✅ Exported: {fname}")

=== DISCORD CARD (BLENDED: +EV + PROBABLE) ===

New York Knicks at Toronto Raptors
New York Knicks -1.5 (-114) — 1.00u
Sim Win%: 58.3% | Market: 53.3%
EV: 9.4%

Detroit Pistons at Cleveland Cavaliers
Detroit Pistons -1 (-106) — 0.78u
Sim Win%: 55.3% | Market: 51.5%
EV: 7.4%

Brooklyn Nets at Miami Heat
Brooklyn Nets ML (500) — 0.33u
Sim Win%: 19.4% | Market: 16.7%
EV: 16.4%

Memphis Grizzlies at Minnesota Timberwolves
Memphis Grizzlies ML (610) — 0.25u
Sim Win%: 15.8% | Market: 14.1%
EV: 12.5%

Phoenix Suns at Sacramento Kings
Sacramento Kings ML (370) — 0.25u
Sim Win%: 23.0% | Market: 21.3%
EV: 8.2%

Washington Wizards at Orlando Magic
Washington Wizards ML (700) — 0.25u
Sim Win%: 14.0% | Market: 12.5%
EV: 11.6%

Dallas Mavericks at Charlotte Hornets
Dallas Mavericks ML (460) — 0.25u
Sim Win%: 19.4% | Market: 17.9%
EV: 8.7%

Oklahoma City Thunder at Chicago Bulls
Chicago Bulls ML (350) — 0.25u
Sim Win%: 23.7% | Market: 22.2%
EV: 6.6%

San Antonio Spurs at Philadelphia 76ers
Philadelph

In [45]:
# ==========================================
# BLENDED RANKING v2 (PROBABILITY-LEANING)
# ==========================================

df = final_card.copy()

# +EV only
df = df[df["ev"] > 0].copy()

# Remove ultra longshots (keeps card sharp)
df = df[df["p_win"] >= 0.18].copy()

# Normalize EV (cap outliers)
ev_clip = df["ev"].clip(upper=df["ev"].quantile(0.85))
df["ev_norm"] = (ev_clip - ev_clip.min()) / (ev_clip.max() - ev_clip.min() + 1e-9)

# Win probability normalized
df["pwin_norm"] = (df["p_win"] - df["p_win"].min()) / (df["p_win"].max() - df["p_win"].min() + 1e-9)

# Stronger probability lean
W_PWIN = 0.75
W_EV   = 0.25

df["blend_score"] = W_PWIN * df["pwin_norm"] + W_EV * df["ev_norm"]

# Correlation cap
df = df.sort_values("blend_score", ascending=False)
kept = []
counts = {}

for _, r in df.iterrows():
    k = r["matchup"]
    if counts.get(k, 0) >= 2:
        continue
    kept.append(r)
    counts[k] = counts.get(k, 0) + 1

df_blend = pd.DataFrame(kept).reset_index(drop=True)

print(f"✅ Blended plays (probability leaning): {len(df_blend)}")
df_blend[["matchup","bet","p_win","ev","units","blend_score"]].head(15)

✅ Blended plays (probability leaning): 15


,matchup,bet,p_win,ev,units,blend_score
0,Washington Wizards at Orlando Magic,Orlando Magic -15.5 (-110),0.98408,0.878698,1.000000,1.000000
1,Memphis Grizzlies at Minnesota Timberwolves,Minnesota Timberwolves -14.5 (-110),0.97810,0.867282,1.000000,0.994323
2,Dallas Mavericks at Charlotte Hornets,Charlotte Hornets -12.5 (-112),0.95774,0.812865,1.000000,0.974996
3,Brooklyn Nets at Miami Heat,Miami Heat -12.5 (-114),0.95728,0.796999,1.000000,0.970048
4,Phoenix Suns at Sacramento Kings,Phoenix Suns -10.5 (-110),0.92888,0.773316,1.000000,0.935607
5,Oklahoma City Thunder at Chicago Bulls,Oklahoma City Thunder -10.5 (-110),0.92506,0.766024,1.000000,0.929677
6,New Orleans Pelicans at Los Angeles Lakers,Los Angeles Lakers -9.5 (-108),0.90458,0.742154,1.000000,0.902695
7,San Antonio Spurs at Philadelphia 76ers,San Antonio Spurs -7.5 (-110),0.84816,0.619215,1.000000,0.810298
8,New York Knicks at Toronto Raptors,New York Knicks -1.5 (-114),0.58260,0.093653,1.000000,0.392174
9,Detroit Pistons at Cleveland Cavaliers,Detroit Pistons -1 (-106),0.55254,0.073804,0.782324,0.357368


In [39]:
TOP_N = 8

out = df_blend.head(TOP_N).copy()

print("=== DISCORD CARD (BLENDED v2: HIGHER HIT RATE) ===\n")
print("\n".join(out["discord_text"].tolist()))

=== DISCORD CARD (BLENDED v2: HIGHER HIT RATE) ===

Washington Wizards at Orlando Magic
Orlando Magic -15.5 (-110) — 1.00u
Sim Win%: 98.4% | Market: 52.4%
EV: 87.9%

Memphis Grizzlies at Minnesota Timberwolves
Minnesota Timberwolves -14.5 (-110) — 1.00u
Sim Win%: 97.8% | Market: 52.4%
EV: 86.7%

Dallas Mavericks at Charlotte Hornets
Charlotte Hornets -12.5 (-112) — 1.00u
Sim Win%: 95.8% | Market: 52.8%
EV: 81.3%

Brooklyn Nets at Miami Heat
Miami Heat -12.5 (-114) — 1.00u
Sim Win%: 95.7% | Market: 53.3%
EV: 79.7%

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 1.00u
Sim Win%: 92.9% | Market: 52.4%
EV: 77.3%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 1.00u
Sim Win%: 92.5% | Market: 52.4%
EV: 76.6%

New Orleans Pelicans at Los Angeles Lakers
Los Angeles Lakers -9.5 (-108) — 1.00u
Sim Win%: 90.5% | Market: 51.9%
EV: 74.2%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 1.00u
Sim Win%: 84.8% | Market: 52.4%
EV: 61.9%


In [46]:
# ======================================
# HIGH HIT-RATE BRAND CARD (BOTTOM CELL)
# +EV only + probability-first ranking
# suppress longshot ML dogs
# cap 2 picks per game
# ======================================

import numpy as np
import pandas as pd
from datetime import datetime

# ---- EXPECTED INPUT ----
# final_card must exist and include at least:
# matchup, market, bet, odds (or AMERICAN_ODDS), p_win, p_mkt, ev, units, discord_text
df = final_card.copy()

# ---- SAFETY: normalize columns ----
if "odds" not in df.columns and "AMERICAN_ODDS" in df.columns:
    df["odds"] = df["AMERICAN_ODDS"]

need = ["matchup","market","bet","p_win","p_mkt","ev","units"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"final_card missing required columns: {missing}")

# ---- 1) +EV ONLY ----
df = df[df["ev"] > 0].copy()

# ---- 2) HIGH HIT-RATE FILTERS ----
# (A) Global minimum win prob (keeps card from being all lottery tickets)
PWIN_MIN = 0.52
df = df[df["p_win"] >= PWIN_MIN].copy()

# (B) Extra suppression for ML dogs (even if they barely clear PWIN_MIN somehow)
# If ML and american odds >= +200 => require higher p_win
ML_DOG_ODDS_CUTOFF = 200
ML_DOG_PWIN_MIN = 0.58

is_ml = df["market"].astype(str).str.upper().eq("ML")
is_dog = df["odds"].astype(float) >= ML_DOG_ODDS_CUTOFF
df = df[~(is_ml & is_dog & (df["p_win"] < ML_DOG_PWIN_MIN))].copy()

# (C) Optional: remove ultra-thin edges that clutter brand cards
EV_MIN = 0.02  # 2%+
df = df[df["ev"] >= EV_MIN].copy()

# ---- 3) PROBABILITY-FIRST SCORING ----
# Primary = p_win, Secondary = ev (soft tiebreak)
# quality_score favors hit rate while still rewarding edge
df["quality_score"] = (0.80 * df["p_win"]) + (0.20 * np.clip(df["ev"], 0, 1))

# ---- 4) CAP 2 PLAYS PER GAME ----
df = df.sort_values(["quality_score","p_win","ev"], ascending=False).copy()
df["game_rank"] = df.groupby("matchup").cumcount() + 1
df = df[df["game_rank"] <= 2].copy()

# ---- 5) FINAL SORT + OPTIONAL TOP N ----
TOP_N = 10
df = df.sort_values(["quality_score","p_win","ev"], ascending=False).head(TOP_N).copy()

# ---- 6) BUILD DISCORD TEXT (if not already present / or refresh) ----
def fmt_pct(x): return f"{100*x:.1f}%"
def fmt_ev(x):  return f"{100*x:.1f}%"

out_lines = []
out_lines.append("=== DISCORD CARD (HIGH HIT RATE / +EV) ===\n")

for _, r in df.iterrows():
    out_lines.append(f"{r['matchup']}")
    out_lines.append(f"{r['bet']} — {float(r['units']):.2f}u")
    out_lines.append(f"Sim Win%: {fmt_pct(float(r['p_win']))} | Market: {fmt_pct(float(r['p_mkt']))}")
    out_lines.append(f"EV: {fmt_ev(float(r['ev']))}\n")

discord_card = "\n".join(out_lines).strip()
print(discord_card)

# ---- 7) EXPORT ----
stamp = datetime.now().strftime("%Y-%m-%d_%H%M")
export_path = f"nba_stable_HIGH_HIT_RATE_{stamp}.csv"
export_cols = ["matchup","market","bet","odds","p_win","p_mkt","ev","units","quality_score"]
df[export_cols].to_csv(export_path, index=False)
print(f"\n✅ Exported: {export_path}")

# Keep for next cells
high_hit_card = df

=== DISCORD CARD (HIGH HIT RATE / +EV) ===

Washington Wizards at Orlando Magic
Orlando Magic -15.5 (-110) — 1.00u
Sim Win%: 98.4% | Market: 52.4%
EV: 87.9%

Memphis Grizzlies at Minnesota Timberwolves
Minnesota Timberwolves -14.5 (-110) — 1.00u
Sim Win%: 97.8% | Market: 52.4%
EV: 86.7%

Dallas Mavericks at Charlotte Hornets
Charlotte Hornets -12.5 (-112) — 1.00u
Sim Win%: 95.8% | Market: 52.8%
EV: 81.3%

Brooklyn Nets at Miami Heat
Miami Heat -12.5 (-114) — 1.00u
Sim Win%: 95.7% | Market: 53.3%
EV: 79.7%

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 1.00u
Sim Win%: 92.9% | Market: 52.4%
EV: 77.3%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 1.00u
Sim Win%: 92.5% | Market: 52.4%
EV: 76.6%

New Orleans Pelicans at Los Angeles Lakers
Los Angeles Lakers -9.5 (-108) — 1.00u
Sim Win%: 90.5% | Market: 51.9%
EV: 74.2%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 1.00u
Sim Win%: 84.8% | Market: 52.4%
EV: 61.9%

New Yor

In [47]:
# ======================================
# STRICT PROBABILITY REBUILD (SPREAD/TOTAL SANITY FIX)
# Recomputes cover/over probs from mean_margin & mean_total
# and current market lines with correct sign conventions.
# ======================================

import numpy as np
import pandas as pd
from math import erf, sqrt
from datetime import datetime

# ---- EXPECTED INPUTS ----
# games_df must exist with: away_team, home_team, mean_margin, mean_total,
# spread_home_line, total_line, home_ml, away_ml (ml optional)
# mean_margin MUST be: (home_points - away_points) predicted mean margin
g = games_df.copy()

req = ["away_team","home_team","mean_margin","mean_total","spread_home_line","total_line"]
miss = [c for c in req if c not in g.columns]
if miss:
    raise KeyError(f"games_df missing: {miss}")

# ---- Normal CDF helper ----
def norm_cdf(x, mu=0.0, sigma=1.0):
    z = (x - mu) / (sigma * sqrt(2))
    return 0.5 * (1 + erf(z))

# ---- CONFIG (match your run) ----
SD_MARGIN = float(globals().get("SD_MARGIN", 14.5))
SD_TOTAL  = float(globals().get("SD_TOTAL", 18.0))

# ---- SPREAD: probability HOME covers spread_home_line ----
# Market spread_home_line is usually negative when home favored.
# Home covers if (home - away) > spread_home_line
g["p_home_cover"] = 1 - g.apply(lambda r: norm_cdf(r["spread_home_line"], mu=r["mean_margin"], sigma=SD_MARGIN), axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]  # away covers opposite side

# ---- TOTAL: probability OVER total_line ----
g["p_over"]  = 1 - g.apply(lambda r: norm_cdf(r["total_line"], mu=r["mean_total"], sigma=SD_TOTAL), axis=1)
g["p_under"] = 1 - g["p_over"]

# ---- MONEYLINE win probs (if mean_margin exists) ----
# home wins if margin > 0
g["p_home_win"] = 1 - g.apply(lambda r: norm_cdf(0.0, mu=r["mean_margin"], sigma=SD_MARGIN), axis=1)
g["p_away_win"] = 1 - g["p_home_win"]

# ---- SANITY CHECKS ----
# If you see 90%+ cover probs on spreads larger than ~10 often, something is off.
san = g[["away_team","home_team","mean_margin","spread_home_line","p_home_cover","mean_total","total_line","p_over","p_home_win"]].copy()
san["abs_spread"] = san["spread_home_line"].abs()

print("✅ Rebuilt probs:", "SD_MARGIN=", SD_MARGIN, "| SD_TOTAL=", SD_TOTAL)
print("Sanity preview (largest spreads):")
display(san.sort_values("abs_spread", ascending=False).head(10))

# Flag suspicious: huge spreads with crazy cover %
sus = san[(san["abs_spread"] >= 10) & (san["p_home_cover"] >= 0.85)]
print(f"⚠ Suspicious covers (abs_spread>=10 & p_home_cover>=85%): {len(sus)}")
if len(sus):
    display(sus[["away_team","home_team","mean_margin","spread_home_line","p_home_cover"]].head(20))

# Write back to games_df so your engine uses corrected columns
games_df = g

✅ Rebuilt probs: SD_MARGIN= 15.5 | SD_TOTAL= 18.5
Sanity preview (largest spreads):


,away_team,home_team,mean_margin,spread_home_line,p_home_cover,mean_total,total_line,p_over,p_home_win,abs_spread
2,Washington Wizards,Orlando Magic,10.541667,-15.5,0.953532,226.455357,227.5,0.477485,0.751782,15.5
6,Memphis Grizzlies,Minnesota Timberwolves,9.670844,-14.5,0.940550,235.125000,237.5,0.448925,0.733662,14.5
3,Brooklyn Nets,Miami Heat,11.105332,-12.5,0.936111,224.125000,227.5,0.427622,0.763150,12.5
0,Dallas Mavericks,Charlotte Hornets,13.000000,-12.5,0.950032,220.750000,229.5,0.318116,0.799184,12.5
9,Phoenix Suns,Sacramento Kings,-11.008321,10.5,0.082624,224.502466,225.5,0.478499,0.238786,10.5
5,Oklahoma City Thunder,Chicago Bulls,-6.779233,10.5,0.132470,221.055633,227.5,0.363791,0.330922,10.5
8,New Orleans Pelicans,Los Angeles Lakers,10.918362,-9.5,0.906133,236.593597,237.5,0.480462,0.759411,9.5
7,San Antonio Spurs,Philadelphia 76ers,-4.388274,7.5,0.221545,226.250000,232.5,0.367743,0.388545,7.5
4,New York Knicks,Toronto Raptors,-3.460093,1.5,0.374482,225.174884,223.5,0.536069,0.411678,1.5
1,Detroit Pistons,Cleveland Cavaliers,-2.030376,1.0,0.422498,221.367361,227.5,0.370136,0.447891,1.0


⚠ Suspicious covers (abs_spread>=10 & p_home_cover>=85%): 4


,away_team,home_team,mean_margin,spread_home_line,p_home_cover
0,Dallas Mavericks,Charlotte Hornets,13.000000,-12.5,0.950032
2,Washington Wizards,Orlando Magic,10.541667,-15.5,0.953532
3,Brooklyn Nets,Miami Heat,11.105332,-12.5,0.936111
6,Memphis Grizzlies,Minnesota Timberwolves,9.670844,-14.5,0.940550


In [48]:
# ============================================
# (BOTTOM FIX CELL 1) — DEFINITIONS + HELPERS
# Paste this at the very bottom and run once
# ============================================

import numpy as np
import pandas as pd
from math import erf, sqrt

def norm_cdf(x, mu=0.0, sigma=1.0):
    """Normal CDF with mean=mu, sd=sigma (no scipy needed)."""
    if sigma <= 0 or pd.isna(sigma):
        return np.nan
    z = (x - mu) / (sigma * sqrt(2))
    return 0.5 * (1.0 + erf(z))

def american_to_prob(odds):
    """Implied prob from American odds (no vig removed)."""
    if pd.isna(odds):
        return np.nan
    odds = float(odds)
    if odds > 0:
        return 100.0 / (odds + 100.0)
    else:
        return (-odds) / ((-odds) + 100.0)

print("✅ Helpers loaded (norm_cdf, american_to_prob)")

✅ Helpers loaded (norm_cdf, american_to_prob)


In [56]:
# ============================================
# FINAL CORRECT SPREAD PROBABILITY LOGIC
# Handles favorite and dog properly
# ============================================

g = games_df.copy()

def home_cover_prob(row):
    spread = row["spread_home_line"]
    mu = row["mean_margin"]
    sd = SD_MARGIN

    if spread < 0:
        # home favorite
        threshold = abs(spread)
        return 1 - norm_cdf(threshold, mu=mu, sigma=sd)
    else:
        # home underdog
        threshold = -abs(spread)
        return 1 - norm_cdf(threshold, mu=mu, sigma=sd)

g["p_home_cover"] = g.apply(home_cover_prob, axis=1)
g["p_away_cover"] = 1 - g["p_home_cover"]

games_df = g

print("✅ Spread probabilities rebuilt (favorite/dog aware).")
display(g[["away_team","home_team","mean_margin","spread_home_line","p_home_cover"]])

✅ Spread probabilities rebuilt (favorite/dog aware).


,away_team,home_team,mean_margin,spread_home_line,p_home_cover
0,Dallas Mavericks,Charlotte Hornets,13.000000,-12.5,0.512867
1,Detroit Pistons,Cleveland Cavaliers,-2.030376,1.0,0.473499
2,Washington Wizards,Orlando Magic,10.541667,-15.5,0.374525
3,Brooklyn Nets,Miami Heat,11.105332,-12.5,0.464152
4,New York Knicks,Toronto Raptors,-3.460093,1.5,0.449685
5,Oklahoma City Thunder,Chicago Bulls,-6.779233,10.5,0.594854
6,Memphis Grizzlies,Minnesota Timberwolves,9.670844,-14.5,0.377688
7,San Antonio Spurs,Philadelphia 76ers,-4.388274,7.5,0.579556
8,New Orleans Pelicans,Los Angeles Lakers,10.918362,-9.5,0.536455
9,Phoenix Suns,Sacramento Kings,-11.008321,10.5,0.486919


In [50]:
# =========================================================
# (BOTTOM FIX CELL 3) — REBUILD BET CARDS + DISCORD TEXT
# Recomputes p_mkt, EV, units, and discord_text using fixed p_home_cover
# Assumes you already have your card-builder dataframes OR games_df only.
# =========================================================

if "games_df" not in globals():
    raise RuntimeError("games_df not found.")

df = games_df.copy()

# --- MARKET PROBS ---
# Spread market implied prob from spread odds (fallback -110 if missing)
if "spread_home_odds" not in df.columns:
    df["spread_home_odds"] = -110
if "spread_away_odds" not in df.columns:
    df["spread_away_odds"] = -110

df["p_mkt_home_cover"] = df["spread_home_odds"].apply(american_to_prob)
df["p_mkt_away_cover"] = df["spread_away_odds"].apply(american_to_prob)

# ML market implied prob
if "home_ml" in df.columns and "away_ml" in df.columns:
    df["p_mkt_home_win"] = df["home_ml"].apply(american_to_prob)
    df["p_mkt_away_win"] = df["away_ml"].apply(american_to_prob)
else:
    df["p_mkt_home_win"] = np.nan
    df["p_mkt_away_win"] = np.nan

# Totals market implied prob
if "over_odds" not in df.columns:
    df["over_odds"] = -110
if "under_odds" not in df.columns:
    df["under_odds"] = -110
df["p_mkt_over"] = df["over_odds"].apply(american_to_prob)
df["p_mkt_under"] = df["under_odds"].apply(american_to_prob)

# --- BUILD CANDIDATE ROWS (ML / SPREAD / TOTALS) ---
rows = []

for _, r in df.iterrows():
    matchup = f"{r.get('away_team','')} at {r.get('home_team','')}"
    commence = r.get("commence_et", r.get("commence_utc", None))

    # SPREAD (home side shown as team +spread or -spread based on sign)
    if not pd.isna(r.get("spread_home_line", np.nan)):
        # home bet string uses signed spread_home_line (e.g., -1.5)
        h_line = float(r["spread_home_line"])
        h_odds = int(r.get("spread_home_odds", -110))
        a_line = -h_line
        a_odds = int(r.get("spread_away_odds", -110))

        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "SPREAD",
            "team": r.get("home_team",""),
            "bet": f"{r.get('home_team','')} {h_line:+g} ({h_odds:+d})".replace("+ -", "-").replace("+", "+"),
            "odds": h_odds,
            "p_win": float(r["p_home_cover"]),
            "p_mkt": float(r["p_mkt_home_cover"]) if not pd.isna(r["p_mkt_home_cover"]) else np.nan,
        })
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "SPREAD",
            "team": r.get("away_team",""),
            "bet": f"{r.get('away_team','')} {a_line:+g} ({a_odds:+d})".replace("+ -", "-").replace("+", "+"),
            "odds": a_odds,
            "p_win": float(r["p_away_cover"]),
            "p_mkt": float(r["p_mkt_away_cover"]) if not pd.isna(r["p_mkt_away_cover"]) else np.nan,
        })

    # ML
    if not pd.isna(r.get("home_ml", np.nan)) and not pd.isna(r.get("away_ml", np.nan)):
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "ML",
            "team": r.get("home_team",""),
            "bet": f"{r.get('home_team','')} ML ({int(r['home_ml']):+d})".replace("+ -", "-"),
            "odds": int(r["home_ml"]),
            "p_win": float(r.get("p_home_win", np.nan)) if "p_home_win" in r else np.nan,
            "p_mkt": float(r["p_mkt_home_win"]),
        })
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "ML",
            "team": r.get("away_team",""),
            "bet": f"{r.get('away_team','')} ML ({int(r['away_ml']):+d})".replace("+ -", "-"),
            "odds": int(r["away_ml"]),
            "p_win": float(r.get("p_away_win", np.nan)) if "p_away_win" in r else np.nan,
            "p_mkt": float(r["p_mkt_away_win"]),
        })

    # TOTALS (if p_over exists)
    if "total_line" in r and not pd.isna(r.get("total_line", np.nan)) and "p_over" in df.columns:
        t = float(r["total_line"])
        o_odds = int(r.get("over_odds", -110))
        u_odds = int(r.get("under_odds", -110))
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "TOTAL",
            "team": "",
            "bet": f"OVER {t:g} ({o_odds:+d})".replace("+ -", "-"),
            "odds": o_odds,
            "p_win": float(r["p_over"]),
            "p_mkt": float(r["p_mkt_over"]),
        })
        rows.append({
            "matchup": matchup, "commence_time": commence,
            "market": "TOTAL",
            "team": "",
            "bet": f"UNDER {t:g} ({u_odds:+d})".replace("+ -", "-"),
            "odds": u_odds,
            "p_win": float(r["p_under"]) if "p_under" in r else (1.0 - float(r["p_over"])),
            "p_mkt": float(r["p_mkt_under"]),
        })

card = pd.DataFrame(rows)

# EV per 1u (simple edge vs market prob baseline)
card["ev"] = (card["p_win"] - card["p_mkt"]) / card["p_mkt"]
card = card.replace([np.inf, -np.inf], np.nan)

# half-kelly-ish units (bounded) using edge proxy; keep your existing sizing if you have it
# If you already have your own kelly sizing, comment this out.
card["units"] = (0.5 * card["ev"]).clip(lower=0.25, upper=1.0).fillna(0.0)
card = card[card["units"] > 0].copy()

# cap: max 2 picks per game
card["game_key"] = card["matchup"]
card = card.sort_values(["ev","p_win"], ascending=False)
card = card.groupby("game_key").head(2).reset_index(drop=True)

# discord text
def fmt_row(r):
    return (
        f"{r['matchup']}\n"
        f"{r['bet']} — {r['units']:.2f}u\n"
        f"Sim Win%: {100*r['p_win']:.1f}% | Market: {100*r['p_mkt']:.1f}%\n"
        f"EV: {100*r['ev']:.1f}%\n"
    )

card["discord_text"] = card.apply(fmt_row, axis=1)

print(f"✅ Rebuilt card: {len(card)} rows (after +EV & cap 2/game)")
print("\n=== DISCORD CARD (FIXED SPREAD LOGIC) ===\n")
print("\n".join(card["discord_text"].tolist()))

out_path = f"nba_stable_2026-03-03_FIXED_SPREAD_CARD.csv"
card.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

# make it available as final_card for next steps
final_card = card.copy()

✅ Rebuilt card: 20 rows (after +EV & cap 2/game)

=== DISCORD CARD (FIXED SPREAD LOGIC) ===

Washington Wizards at Orlando Magic
Washington Wizards ML (+700) — 0.49u
Sim Win%: 24.8% | Market: 12.5%
EV: 98.6%

Memphis Grizzlies at Minnesota Timberwolves
Memphis Grizzlies ML (+610) — 0.45u
Sim Win%: 26.6% | Market: 14.1%
EV: 89.1%

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 0.38u
Sim Win%: 91.7% | Market: 52.4%
EV: 75.1%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 0.33u
Sim Win%: 86.8% | Market: 52.4%
EV: 65.6%

Oklahoma City Thunder at Chicago Bulls
Chicago Bulls ML (+350) — 0.25u
Sim Win%: 33.1% | Market: 22.2%
EV: 48.9%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 0.25u
Sim Win%: 77.8% | Market: 52.4%
EV: 48.6%

Brooklyn Nets at Miami Heat
Brooklyn Nets ML (+500) — 0.25u
Sim Win%: 23.7% | Market: 16.7%
EV: 42.1%

Dallas Mavericks at Charlotte Hornets
UNDER 229.5 (-110) — 0.25u
Sim Win%: 68.2% | Market: 52.4

In [51]:
# ============================================
# (BOTTOM CELL) — ANTI DOUBLE-DIP (SIDE CONFLICT)
# Keep max 2 picks/game, but NOT both sides.
# Allows: (spread or ML) + total
# Blocks: ML + opposite spread, or both spreads, or both MLs
# ============================================

import pandas as pd
import numpy as np

if "final_card" not in globals():
    raise RuntimeError("final_card not found. Run the rebuild cell first.")

fc = final_card.copy()

def market_bucket(m):
    m = str(m).upper()
    if m in ["ML", "SPREAD"]:
        return "SIDE"
    if m in ["TOTAL", "TOTALS", "OU", "O/U"]:
        return "TOTAL"
    return "OTHER"

fc["bucket"] = fc["market"].apply(market_bucket)

# Identify "side team" for SIDE bets (to detect opposing sides)
def side_team_from_bet(row):
    if row["bucket"] != "SIDE":
        return None
    # If team column exists use it; else parse from bet string
    if "team" in row and pd.notna(row["team"]) and str(row["team"]).strip() != "":
        return str(row["team"]).strip()
    # fallback parse: first token chunk before " ML" or " +" or " -"
    b = str(row["bet"])
    return b.split(" ML")[0].split(" +")[0].split(" -")[0].strip()

fc["side_team"] = fc.apply(side_team_from_bet, axis=1)

clean_rows = []
for game, g in fc.groupby("matchup", sort=False):
    g = g.sort_values(["ev","p_win"], ascending=False).copy()

    picks = []
    side_picks = g[g["bucket"] == "SIDE"].copy()
    total_picks = g[g["bucket"] == "TOTAL"].copy()

    # pick best SIDE (at most one)
    if len(side_picks) > 0:
        picks.append(side_picks.iloc[0])

    # pick best TOTAL (at most one)
    if len(total_picks) > 0:
        picks.append(total_picks.iloc[0])

    # if still <2 and there are OTHER bets, fill (rare)
    if len(picks) < 2:
        other = g[g["bucket"] == "OTHER"]
        for _, r in other.iterrows():
            if len(picks) >= 2:
                break
            picks.append(r)

    clean_rows.extend(picks)

fc2 = pd.DataFrame(clean_rows).reset_index(drop=True)

# rebuild discord text in same format
def fmt_row(r):
    return (
        f"{r['matchup']}\n"
        f"{r['bet']} — {r['units']:.2f}u\n"
        f"Sim Win%: {100*r['p_win']:.1f}% | Market: {100*r['p_mkt']:.1f}%\n"
        f"EV: {100*r['ev']:.1f}%\n"
    )

fc2["discord_text"] = fc2.apply(fmt_row, axis=1)

print(f"✅ Anti-double-dip applied. Rows: {len(final_card)} → {len(fc2)}")
print("\n=== DISCORD CARD (NO SIDE CONFLICTS) ===\n")
print("\n".join(fc2["discord_text"].tolist()))

out_path = "nba_stable_2026-03-03_FIXED_SPREAD_NO_DOUBLEDIP.csv"
fc2.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

final_card = fc2.copy()

✅ Anti-double-dip applied. Rows: 20 → 14

=== DISCORD CARD (NO SIDE CONFLICTS) ===

Washington Wizards at Orlando Magic
Washington Wizards ML (+700) — 0.49u
Sim Win%: 24.8% | Market: 12.5%
EV: 98.6%

Memphis Grizzlies at Minnesota Timberwolves
Memphis Grizzlies ML (+610) — 0.45u
Sim Win%: 26.6% | Market: 14.1%
EV: 89.1%

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 0.38u
Sim Win%: 91.7% | Market: 52.4%
EV: 75.1%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 0.33u
Sim Win%: 86.8% | Market: 52.4%
EV: 65.6%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 0.25u
Sim Win%: 77.8% | Market: 52.4%
EV: 48.6%

Brooklyn Nets at Miami Heat
Brooklyn Nets ML (+500) — 0.25u
Sim Win%: 23.7% | Market: 16.7%
EV: 42.1%

Brooklyn Nets at Miami Heat
UNDER 227.5 (-114) — 0.25u
Sim Win%: 57.2% | Market: 53.3%
EV: 7.4%

Dallas Mavericks at Charlotte Hornets
Dallas Mavericks ML (+460) — 0.25u
Sim Win%: 20.1% | Market: 17.9%
EV: 12.5%

Dalla

In [52]:
# ============================================
# (BOTTOM CELL) — HIGH HIT-RATE FILTER
# Keeps +EV only, then filters by p_win threshold
# ============================================

HITRATE_MIN = 0.58  # tweak: 0.58 / 0.60 / 0.62

if "final_card" not in globals():
    raise RuntimeError("final_card not found.")

fc = final_card.copy()

# keep +EV and hit-rate threshold
fc = fc[(fc["ev"] > 0) & (fc["p_win"] >= HITRATE_MIN)].copy()
fc = fc.sort_values(["p_win","ev"], ascending=False).reset_index(drop=True)

print(f"✅ High hit-rate filter applied (p_win >= {HITRATE_MIN:.2f}). Rows: {len(fc)}")
print("\n=== DISCORD CARD (HIGH HIT-RATE) ===\n")
print("\n".join(fc["discord_text"].tolist()))

out_path = f"nba_stable_2026-03-03_HIGH_HITRATE_{int(HITRATE_MIN*100)}.csv"
fc.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

final_card = fc.copy()

✅ High hit-rate filter applied (p_win >= 0.58). Rows: 6

=== DISCORD CARD (HIGH HIT-RATE) ===

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 0.38u
Sim Win%: 91.7% | Market: 52.4%
EV: 75.1%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 0.33u
Sim Win%: 86.8% | Market: 52.4%
EV: 65.6%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 0.25u
Sim Win%: 77.8% | Market: 52.4%
EV: 48.6%

Dallas Mavericks at Charlotte Hornets
UNDER 229.5 (-110) — 0.25u
Sim Win%: 68.2% | Market: 52.4%
EV: 30.2%

Detroit Pistons at Cleveland Cavaliers
UNDER 227.5 (-114) — 0.25u
Sim Win%: 63.0% | Market: 53.3%
EV: 18.2%

New York Knicks at Toronto Raptors
New York Knicks -1.5 (-114) — 0.25u
Sim Win%: 62.6% | Market: 53.3%
EV: 17.4%


✅ Exported: nba_stable_2026-03-03_HIGH_HITRATE_57.csv


In [53]:
# ============================================
# (BOTTOM CELL) — HIGH HIT-RATE FILTER (LOCKED + CLEAN FILENAME)
# ============================================

HITRATE_MIN = 0.58  # lock it here
TAG = "58"          # lock filename tag

fc = final_card.copy()

# If your final_card still includes non +EV, keep this:
fc = fc[fc["ev"] > 0].copy()

fc = fc[fc["p_win"] >= HITRATE_MIN].copy()
fc = fc.sort_values(["p_win","ev"], ascending=False).reset_index(drop=True)

print(f"✅ High hit-rate filter applied (p_win >= {HITRATE_MIN:.2f}). Rows: {len(fc)}")
print("\n=== DISCORD CARD (HIGH HIT-RATE) ===\n")
print("\n".join(fc["discord_text"].tolist()))

out_path = f"nba_stable_2026-03-03_HIGH_HITRATE_{TAG}.csv"
fc.to_csv(out_path, index=False)
print(f"\n✅ Exported: {out_path}")

final_card = fc.copy()

✅ High hit-rate filter applied (p_win >= 0.58). Rows: 6

=== DISCORD CARD (HIGH HIT-RATE) ===

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 0.38u
Sim Win%: 91.7% | Market: 52.4%
EV: 75.1%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 0.33u
Sim Win%: 86.8% | Market: 52.4%
EV: 65.6%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 0.25u
Sim Win%: 77.8% | Market: 52.4%
EV: 48.6%

Dallas Mavericks at Charlotte Hornets
UNDER 229.5 (-110) — 0.25u
Sim Win%: 68.2% | Market: 52.4%
EV: 30.2%

Detroit Pistons at Cleveland Cavaliers
UNDER 227.5 (-114) — 0.25u
Sim Win%: 63.0% | Market: 53.3%
EV: 18.2%

New York Knicks at Toronto Raptors
New York Knicks -1.5 (-114) — 0.25u
Sim Win%: 62.6% | Market: 53.3%
EV: 17.4%


✅ Exported: nba_stable_2026-03-03_HIGH_HITRATE_58.csv


In [54]:
print("=== CURRENT FINAL_CARD STATE ===\n")
print("\n".join(final_card["discord_text"].tolist()))

=== CURRENT FINAL_CARD STATE ===

Phoenix Suns at Sacramento Kings
Phoenix Suns -10.5 (-110) — 0.38u
Sim Win%: 91.7% | Market: 52.4%
EV: 75.1%

Oklahoma City Thunder at Chicago Bulls
Oklahoma City Thunder -10.5 (-110) — 0.33u
Sim Win%: 86.8% | Market: 52.4%
EV: 65.6%

San Antonio Spurs at Philadelphia 76ers
San Antonio Spurs -7.5 (-110) — 0.25u
Sim Win%: 77.8% | Market: 52.4%
EV: 48.6%

Dallas Mavericks at Charlotte Hornets
UNDER 229.5 (-110) — 0.25u
Sim Win%: 68.2% | Market: 52.4%
EV: 30.2%

Detroit Pistons at Cleveland Cavaliers
UNDER 227.5 (-114) — 0.25u
Sim Win%: 63.0% | Market: 53.3%
EV: 18.2%

New York Knicks at Toronto Raptors
New York Knicks -1.5 (-114) — 0.25u
Sim Win%: 62.6% | Market: 53.3%
EV: 17.4%



In [55]:
games_df[["away_team","home_team","mean_margin","spread_home_line","p_home_cover"]]

,away_team,home_team,mean_margin,spread_home_line,p_home_cover
0,Dallas Mavericks,Charlotte Hornets,13.000000,-12.5,0.512867
1,Detroit Pistons,Cleveland Cavaliers,-2.030376,1.0,0.422498
2,Washington Wizards,Orlando Magic,10.541667,-15.5,0.374525
3,Brooklyn Nets,Miami Heat,11.105332,-12.5,0.464152
4,New York Knicks,Toronto Raptors,-3.460093,1.5,0.374482
5,Oklahoma City Thunder,Chicago Bulls,-6.779233,10.5,0.132470
6,Memphis Grizzlies,Minnesota Timberwolves,9.670844,-14.5,0.377688
7,San Antonio Spurs,Philadelphia 76ers,-4.388274,7.5,0.221545
8,New Orleans Pelicans,Los Angeles Lakers,10.918362,-9.5,0.536455
9,Phoenix Suns,Sacramento Kings,-11.008321,10.5,0.082624


In [57]:
# ============================================
# REBUILD CARD FROM CORRECTED games_df
# ============================================

rows = []

for _, r in games_df.iterrows():
    matchup = f"{r['away_team']} at {r['home_team']}"

    # HOME SPREAD
    rows.append({
        "matchup": matchup,
        "market": "SPREAD",
        "bet": f"{r['home_team']} {r['spread_home_line']} (-110)",
        "p_win": r["p_home_cover"],
        "p_mkt": 0.524,
    })

    # AWAY SPREAD
    rows.append({
        "matchup": matchup,
        "market": "SPREAD",
        "bet": f"{r['away_team']} {-(r['spread_home_line'])} (-110)",
        "p_win": 1 - r["p_home_cover"],
        "p_mkt": 0.524,
    })

card = pd.DataFrame(rows)
card["ev"] = card["p_win"] - card["p_mkt"]
card = card[card["ev"] > 0].copy()

card = card.sort_values(["p_win","ev"], ascending=False).reset_index(drop=True)

print("=== REBUILT SPREAD CARD ===\n")
display(card.head(15))

=== REBUILT SPREAD CARD ===



,matchup,market,bet,p_win,p_mkt,ev
0,Washington Wizards at Orlando Magic,SPREAD,Washington Wizards 15.5 (-110),0.625475,0.524,0.101475
1,Memphis Grizzlies at Minnesota Timberwolves,SPREAD,Memphis Grizzlies 14.5 (-110),0.622312,0.524,0.098312
2,Oklahoma City Thunder at Chicago Bulls,SPREAD,Chicago Bulls 10.5 (-110),0.594854,0.524,0.070854
3,San Antonio Spurs at Philadelphia 76ers,SPREAD,Philadelphia 76ers 7.5 (-110),0.579556,0.524,0.055556
4,New York Knicks at Toronto Raptors,SPREAD,New York Knicks -1.5 (-110),0.550315,0.524,0.026315
5,New Orleans Pelicans at Los Angeles Lakers,SPREAD,Los Angeles Lakers -9.5 (-110),0.536455,0.524,0.012455
6,Brooklyn Nets at Miami Heat,SPREAD,Brooklyn Nets 12.5 (-110),0.535848,0.524,0.011848
7,Detroit Pistons at Cleveland Cavaliers,SPREAD,Detroit Pistons -1.0 (-110),0.526501,0.524,0.002501
